In [1]:
# -*- coding: utf-8 -*-
# Кластеризация муниципалитетов России v2.0
# Самодостаточный скрипт: каждая "ячейка" отделена маркером # ============ ЯЧЕЙКА N
# Запуск целиком: python sber_clustering_v2.py   (Kaggle: python /kaggle/working/sber_clustering_v2.py)
# Входные данные положите в /kaggle/input или рядом со скриптом.
# ============================================================================
# ============ ЯЧЕЙКА 0 ============
# ============================================================================
# ============================================================================
# ЯЧЕЙКА 0. КОНФИГУРАЦИЯ И ОКРУЖЕНИЕ
# ============================================================================
import os, sys, json, glob, time, zipfile, warnings, fnmatch, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import linear_sum_assignment
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import (silhouette_score, calinski_harabasz_score,
                             davies_bouldin_score, adjusted_rand_score)
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPRegressor
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm
warnings.filterwarnings("ignore")
# pandas в ряде операций логирует внутренние исключения (fallback на медленный путь)
# через logging на уровне DEBUG — глушим, чтобы не путать с реальными ошибками.
import logging
for _ln in ("pandas", "pyarrow", "matplotlib", "sklearn"):
    logging.getLogger(_ln).setLevel(logging.ERROR)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
RNG = 20260101
np.random.seed(RNG)
T0 = time.time()
def tick(msg):
    print(f"[{time.time()-T0:6.1f}s] {msg}")
CFG = {
    # --- ВРЕМЕННОЕ ОКНО -----------------------------------------------------
    # 2019-2024: 6 годовых срезов. До 2019 г. часть показателей ОМСУ имеет
    # неполное покрытие и меняющуюся методологию; SberIndex начинается с 2020 г.
    # (зарплаты) и 2021 г. (потребительская активность), поэтому окно выбрано так,
    # чтобы был перехлёст для честной внешней валидации. 2025 г. исключён
    # (неполные данные, признаки ОМСУ публикуются с лагом).
    "years": list(range(2019, 2025)),
    "sber_overlap_years": [2020, 2021, 2022, 2023, 2024],
    # --- ВЫБОР K ------------------------------------------------------------
    "k_range": list(range(2, 11)),
    "bootstrap_iters": 40,
    "jackknife_region_min": 200,     # минимум МО в регионе для jackknife
    # --- ПРИЗНАКИ -----------------------------------------------------------
    "min_obs_share": 0.95,           # доля лет с данными, требуемая от МО
    "winsor": (0.01, 0.99),
    "log_features": ["pop_k", "invest_pc", "housing_pc", "msu_exp_pc",
                     "salary", "retail_pc"],
    "tol_top1": 0.30,                # доля топ-1 значения => признак вырожден
    "tol_cens_hi": 0.02,             # доля значений на максимуме => цензурирование
    "tol_cens_lo": 0.05,
    "min_features": 6,
    # --- СЕТЬ ---------------------------------------------------------------
    "graph_k_grid": [5, 10, 15, 20],
    "graph_k_main": 10,
    "n_pca_graph": 5,
    # --- СЖАТИЕ РАЗМЕРНОСТИ ------------------------------------------------
    "test_mlp_ae": True,             # сравнить сырое пространство / PCA / MLP-AE
}
OUT = Path("/kaggle/working")
if not OUT.exists():
    OUT = Path.cwd() / "kaggle_out_probe"
PROC, VIZ = OUT / "processed", OUT / "processed" / "viz"
PROC.mkdir(parents=True, exist_ok=True); VIZ.mkdir(parents=True, exist_ok=True)
SEARCH_ROOTS = [Path("/kaggle/input"), Path.cwd(), Path(r"C:\CLUSTER"),
                Path(r"C:\Users\Сергей\Desktop")]
_INDEX = None
SKIP_DIRS = {"_xcache", "viz", ".libs", "processed", "kaggle_out_probe", "__pycache__"}
def _build_index():
    """Однократный обход корней: файл -> путь. Ускоряет все последующие поиски."""
    global _INDEX
    if _INDEX is None:
        _INDEX = []
        for root in SEARCH_ROOTS:
            if not root.exists():
                continue
            for p in root.rglob("*"):
                if (p.is_file() and not p.name.startswith("~$")
                        and not (SKIP_DIRS & set(p.parts))):
                    _INDEX.append(p)
        print(f"  проиндексировано файлов: {len(_INDEX)}")
    return _INDEX
def find_files(patterns, roots=None):
    """Ищет файлы по маскам во всех доступных корнях (Kaggle input + локально)."""
    idx = _build_index() if roots is None else [
        p for r in roots if r.exists() for p in r.rglob("*") if p.is_file()]
    found, seen = [], set()
    for pat in patterns:
        for p in idx:
            if fnmatch.fnmatch(p.name, pat):
                k = str(p).lower()
                if k not in seen:
                    seen.add(k); found.append(p)
    return sorted(found)
print("=" * 78); print("КОНФИГУРАЦИЯ"); print("=" * 78)
print(json.dumps(CFG, ensure_ascii=False, indent=1))
tick("окружение готово")

КОНФИГУРАЦИЯ
{
 "years": [
  2019,
  2020,
  2021,
  2022,
  2023,
  2024
 ],
 "sber_overlap_years": [
  2020,
  2021,
  2022,
  2023,
  2024
 ],
 "k_range": [
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10
 ],
 "bootstrap_iters": 40,
 "jackknife_region_min": 200,
 "min_obs_share": 0.95,
 "winsor": [
  0.01,
  0.99
 ],
 "log_features": [
  "pop_k",
  "invest_pc",
  "housing_pc",
  "msu_exp_pc",
  "salary",
  "retail_pc"
 ],
 "tol_top1": 0.3,
 "tol_cens_hi": 0.02,
 "tol_cens_lo": 0.05,
 "min_features": 6,
 "graph_k_grid": [
  5,
  10,
  15,
  20
 ],
 "graph_k_main": 10,
 "n_pca_graph": 5,
 "test_mlp_ae": true
}
[   0.0s] окружение готово


In [2]:
# ============================================================================
# ============ ЯЧЕЙКА 1 ============
# ============================================================================
# ============================================================================
# ЯЧЕЙКА 1. КАТАЛОГ ПОКАЗАТЕЛЕЙ + СПРАВОЧНИК РЕГИОНОВ
# ============================================================================
tick("поиск каталога «Справочники.xlsx»")
cat_files = find_files(["*Справочник*.xlsx", "*справочник*.xlsx", "*Справочники*.xlsx"])
CATALOG, REGIONS_REF = None, None
for cf in cat_files:
    try:
        xl = pd.ExcelFile(cf)
        sh_ind = next((s for s in xl.sheet_names if "Показател" in s), None)
        sh_reg = next((s for s in xl.sheet_names if "Регион" in s), None)
        if sh_ind:
            CATALOG = xl.parse(sh_ind)
            CATALOG.columns = (["sec","section","code","name","unit","freq","years",
                                "url","attrs","coverage","gt300k","gt15k"] + list(CATALOG.columns))[:len(CATALOG.columns)]
        if sh_reg:
            REGIONS_REF = xl.parse(sh_reg)
        print(f"каталог: {cf.name} | показателей: {len(CATALOG) if CATALOG is not None else 0} | "
              f"разделов: {CATALOG['section'].nunique() if CATALOG is not None else 0}")
        break
    except Exception as e:
        print("  не прочитан:", cf.name, e)
if CATALOG is None:
    print("каталог не найден — работаем по встроенной карте кодов (получена из этого каталога)")
    CATALOG = pd.DataFrame({
        "code": ["Y48213002","Y48112027","Y48112004","Y48112023","Y48109001","Y48211001",
                 "Y48215001","Y48106005","Y48313004","Y48313015","Y48401003"],
        "section": ["Занятость","Население","Население","Население","Инвестиции","Жильё",
                    "Жильё","ОКС","Местный бюджет","Местный бюджет","Торговля"],
        "name": ["Среднемесячная номинальная начисленная зарплата работников крупных и средних предприятий",
                 "Оценка численности населения на 1 января","Естественный прирост (убыль)",
                 "Миграционный прирост (убыль)","Инвестиции в основной капитал организаций",
                 "Общая площадь жилых помещений на жителя","Ввод жилья на жителя",
                 "Доля дорог, не отвечающих нормативным требованиям",
                 "Расходы бюджета на содержание работников МСУ на жителя",
                 "Доля налоговых и неналоговых доходов местного бюджета",
                 "Оборот розничной торговли"]})
print(CATALOG[["code","section","name"]].head(12).to_string(index=False))
tick("каталог готов")


[   0.0s] поиск каталога «Справочники.xlsx»
  проиндексировано файлов: 29
каталог не найден — работаем по встроенной карте кодов (получена из этого каталога)
     code        section                                                                                     name
Y48213002      Занятость Среднемесячная номинальная начисленная зарплата работников крупных и средних предприятий
Y48112027      Население                                                 Оценка численности населения на 1 января
Y48112004      Население                                                             Естественный прирост (убыль)
Y48112023      Население                                                             Миграционный прирост (убыль)
Y48109001     Инвестиции                                                Инвестиции в основной капитал организаций
Y48211001          Жильё                                                  Общая площадь жилых помещений на жителя
Y48215001          Жильё                    

In [3]:
# ============================================================================
# ============ ЯЧЕЙКА 2 ============
# ============================================================================
# ============================================================================
# ЯЧЕЙКА 2. ГОДОВАЯ ПАНЕЛЬ РОССТАТА + СПРАВОЧНИК ОКТМО
# ============================================================================
from concurrent.futures import ThreadPoolExecutor
# Точные фильтры по разрезам: {код: [(признак, {колонка: значение}), ...]}
# Фильтры ОБЯЗАТЕЛЬНЫ: без них в сумму попадут подкатегории (именно это в v1
# давало invest_pc = 713 666 у 26 МО и tax_share = 100 у 66 МО).
# Один код = одно чтение файла: из Y48112027 берём и всё население, и городское.
REGISTRY = {
    "Y48112027": [("pop_total", {"mest": "Все население"}),
                  ("pop_urban", {"mest": "Городское население"})],
    "Y48112004": [("natural", {})],
    "Y48112023": [("migr", {"grup_2": "Всего", "migr": "Миграция — всего"})],
    "Y48109001": [("invest", {"istinv": "Всего",
                              "okved2": "Всего по обследуемым видам экономической деятельности"})],
    "Y48211001": [("housing_pc", {})],
    "Y48215001": [("built_pc", {})],
    "Y48106005": [("bad_roads", {})],
    "Y48313004": [("msu_exp", {})],
    "Y48313015": [("tax_share", {})],
    "Y48401003": [("retail", {})],
    "Y48213002": [("salary", {})],
}
NEEDED = ["oktmo_stable","municipality","mun_type","region_id","region_name","year",
          "indicator_value","indicator_code","mun_level"]
KEY_DIMS = ["mest","istinv","okved2","grup_2","migr","vozr"]
KEEP = NEEDED + KEY_DIMS
CACHE = PROC / "_xcache"; CACHE.mkdir(exist_ok=True)
def read_cached(path):
    """Читает файл один раз и кэширует в parquet.
    XLSX-выгрузки Росстата читаются openpyxl'ом по 1.5-2 минуты каждый (400-560 тыс.
    строк), поэтому первый запуск платит эту цену, а повторные — мгновенны.
    """
    key = hashlib.md5(str(path).encode("utf-8")).hexdigest()[:12]
    cp = CACHE / f"{key}.parquet"
    if cp.exists():
        return pd.read_parquet(cp)
    if path.suffix.lower() == ".parquet":
        import pyarrow.parquet as pq
        names = pq.ParquetFile(path).schema_arrow.names
        d = pd.read_parquet(path, columns=[c for c in names if c in set(KEEP)])
    else:
        d = pd.read_excel(path, usecols=lambda c: c in set(KEEP))
    try:
        d.to_parquet(cp, index=False)
    except Exception:
        pass
    return d
def harvest_code(code, variants):
    """Одно чтение файла(ов) на код; возвращает {признак: DataFrame}. Дедупликация по xlsx<parquet."""
    files = find_files([f"*{code}*.parquet", f"*{code}*.xlsx"])
    by_stem = {}
    for p in files:                      # parquet имеет приоритет над xlsx
        stem = p.stem.replace("_112_v20250918", "")
        if stem not in by_stem or p.suffix.lower() == ".parquet":
            by_stem[stem] = p
    files = sorted(by_stem.values(), key=lambda x: (x.suffix.lower() != ".parquet", str(x)))
    out = {name: [] for name, _ in variants}
    for p in files:
        try:
            d = read_cached(p)
        except Exception as e:
            print(f"    битый файл {p.name}: {type(e).__name__}: {str(e)[:80]}"); continue
        if "indicator_code" not in d.columns or not (d.indicator_code.astype(str).str.strip() == code).any():
            continue
        d = d[d.indicator_code.astype(str).str.strip() == code]
        d = d[d.mun_level.astype(str).str.contains("верхнего", na=False)]
        if not len(d):
            continue
        for name, flt in variants:
            sub = d
            for col, val in flt.items():
                if col in sub.columns:
                    sub = sub[sub[col].astype(str).str.strip() == val]
                else:
                    print(f"    ВНИМАНИЕ: разрез {col} отсутствует в {p.name} — фильтр не применён")
            if len(sub):
                out[name].append(sub)
    return {k: (pd.concat(v, ignore_index=True) if v else None) for k, v in out.items()}, len(files)
tick("сборка панели Росстата (по кодам, с кэшем чтения)")
def build_code(item):
    code, variants = item
    t_i = time.time()
    res, nf = harvest_code(code, variants)
    agg = {}
    for name, d in res.items():
        if d is None or not len(d):
            agg[name] = None; continue
        d = d.copy()
        d["year"] = pd.to_numeric(d.year, errors="coerce")
        d["indicator_value"] = pd.to_numeric(d.indicator_value, errors="coerce")
        d["oktmo_stable"] = (d.oktmo_stable.astype(str)
                             .str.replace(r"\.0$", "", regex=True).str.zfill(8))
        d = d.dropna(subset=["oktmo_stable","year","indicator_value"])
        if not len(d):
            agg[name] = None; continue
        agg[name] = (d.groupby(["oktmo_stable","year"], as_index=False)
                       .agg(**{name: ("indicator_value","mean"),
                               "municipality": ("municipality","first"),
                               "mun_type": ("mun_type","first"),
                               "region_name": ("region_name","first")}))
    return code, agg, nf, time.time() - t_i
with ThreadPoolExecutor(max_workers=4) as ex:
    built = list(ex.map(build_code, list(REGISTRY.items())))
RAW = {}
for code, agg, nf, dt in built:
    for name, g in agg.items():
        if g is None:
            print(f"  {name:11s} {code}: НЕТ ДАННЫХ  [{dt:.0f}s]"); continue
        RAW[name] = g
        print(f"  {name:11s} {code}: файлов {nf}, строк {len(g):7d}, лет "
              f"{int(g.year.min())}-{int(g.year.max())}, МО {g.oktmo_stable.nunique()}  [{dt:.0f}s]")
assert "salary" in RAW and "pop_total" in RAW, "Без зарплаты и населения модель бессмысленна"
assert len(RAW) >= 8, f"Собрано слишком мало показателей: {sorted(RAW)}"
# --- приводим строковые колонки к object-dtype ------------------------------
# В pandas 3.0 строки по умолчанию хранятся как ArrowExtensionArray, который
# не поддерживает часть операций (.sum(), .value_counts() с numpy-строками и др.).
# Приводим все строковые признаки к обычному object, чтобы избежать этого класса ошибок.
for _n, _g in RAW.items():
    for _c in ("municipality", "mun_type", "region_name"):
        if _c in _g.columns:
            _g[_c] = _g[_c].astype("object")
print("\nстроковые колонки приведены к object-dtype")
# --- справочник ОКТМО -------------------------------------------------------
OKTMO_REF = (pd.concat([g[["oktmo_stable","municipality","mun_type","region_name"]]
                        for g in RAW.values()], ignore_index=True)
             .dropna(subset=["oktmo_stable"])
             .sort_values("region_name")
             .drop_duplicates("oktmo_stable", keep="first")
             .reset_index(drop=True))
OKTMO_REF["region_code"] = OKTMO_REF.oktmo_stable.str[:2]
OKTMO_REF.to_parquet(PROC / "oktmo_reference.parquet", index=False)
print(f"\nСправочник ОКТМО: {len(OKTMO_REF)} МО, {OKTMO_REF.region_name.nunique()} регионов")
print(OKTMO_REF.mun_type.value_counts().head(8).to_string())
tick("панель Росстата собрана")


[   0.1s] сборка панели Росстата (по кодам, с кэшем чтения)
  pop_total   Y48112027: файлов 1, строк   41979, лет 2006-2025, МО 2590  [903s]
  pop_urban   Y48112027: файлов 1, строк   28874, лет 2006-2025, МО 2061  [903s]
  natural     Y48112004: НЕТ ДАННЫХ  [0s]
  migr        Y48112023: файлов 14, строк   33174, лет 2010-2023, МО 2602  [838s]
  invest      Y48109001: файлов 1, строк   32616, лет 2008-2023, МО 2578  [819s]
  housing_pc  Y48211001: файлов 1, строк   37781, лет 2007-2024, МО 2480  [102s]
  built_pc    Y48215001: файлов 1, строк   36936, лет 2007-2024, МО 2499  [85s]
  bad_roads   Y48106005: файлов 1, строк   26947, лет 2009-2024, МО 2268  [154s]
  msu_exp     Y48313004: файлов 1, строк   28739, лет 2008-2024, МО 2430  [78s]
  tax_share   Y48313015: файлов 1, строк   31766, лет 2010-2024, МО 2426  [99s]
  retail      Y48401003: файлов 1, строк   20461, лет 2010-2025, МО 2485  [379s]
  salary      Y48213002: файлов 1, строк   39554, лет 2008-2024, МО 2592  [86s]

строковые

In [4]:
# ============================================================================
# ============ ЯЧЕЙКА 3 ============
# ============================================================================
# ============================================================================
# ЯЧЕЙКА 3. ПРИЗНАКИ + ПАСПОРТИЗАЦИЯ + АВТООТБОР
# ============================================================================
Y0, Y1 = min(CFG["years"]), max(CFG["years"])
tick(f"сборка мастер-панели {Y0}-{Y1}")
EPS = 1e-9
master = None
for name, g in RAW.items():
    sub = g[(g.year >= Y0) & (g.year <= Y1)][["oktmo_stable","year",name]]
    if master is None:
        master = sub
    else:
        before = len(master)
        master = master.merge(sub, on=["oktmo_stable","year"], how="left")
        assert len(master) == before, f"мердж {name} размножил строки — проверьте дубликаты ОКТМО/год"
master = master.merge(OKTMO_REF, on="oktmo_stable", how="left")
for _c in ("municipality", "mun_type", "region_name"):
    if _c in master.columns:
        master[_c] = master[_c].astype("object")
print(f"панель: {master.shape}, МО {master.oktmo_stable.nunique()}, лет {master.year.nunique()}")
miss = master[[c for c in RAW]].isna().mean().sort_values(ascending=False)
print("\nпропуски по показателям (доля):"); print(miss.round(3).to_string())
# ---------------- производные признаки -------------------------------------
master["pop_k"] = master["pop_total"] / 1000.0
if "pop_urban" in master.columns:
    master["urban_share"] = (master["pop_urban"] / (master["pop_total"] + EPS)).clip(0, 1)
else:
    print("ВНИМАНИЕ: городское население недоступно — urban_share не строится")
master["invest_pc"] = master["invest"] * 1000.0 / (master["pop_total"] + EPS)   # руб/чел
master["retail_pc"] = master["retail"] * 1000.0 / (master["pop_total"] + EPS)   # руб/чел
master["msu_exp_pc"] = master["msu_exp"]                                        # уже руб/жителя
for src, dst in [("migr", "migr_rate"), ("natural", "natural_rate")]:
    if src in master.columns:
        master[dst] = master[src] / (master["pop_total"] + EPS) * 1000.0
    else:
        print(f"  признак {src} недоступен — {dst} не строится")
master["log_pop"] = np.log1p(master["pop_total"])
# локальная плотность: население относительно медианы своего региона (прокси плотности)
master["rel_pop_region"] = master["pop_total"] / (
    master.groupby(["region_name","year"])["pop_total"].transform("median") + EPS)
FEATS_BASE = ["log_pop", "rel_pop_region"] + (["urban_share"] if "urban_share" in master.columns else []) + [
    "invest_pc", "housing_pc", "built_pc", "bad_roads", "msu_exp_pc", "salary", "retail_pc"]
EXTRA_PROFILE = [f for f in ["tax_share","migr_rate","natural_rate"] if f in master.columns]
print("\nБазовые признаки:", FEATS_BASE)
print("Дополнительные (профилирование / валидация):", EXTRA_PROFILE)
# ---------------- паспортизация --------------------------------------------
def passport(df, cols, year):
    rows = []
    for c in cols:
        x = pd.to_numeric(df[c], errors="coerce").dropna()
        if len(x) < 10:
            rows.append(dict(year=year, feature=c, n=len(x), nunique=0, top1_share=np.nan)); continue
        vc = x.value_counts()
        rows.append(dict(
            year=year, feature=c, n=len(x), nunique=int(x.nunique()),
            top1_share=float(vc.iloc[0] / len(x)),
            cv=float(x.std() / (abs(x.mean()) + EPS)),
            p01=float(x.quantile(.01)), p50=float(x.median()), p99=float(x.quantile(.99)),
            mx=float(x.max()), mn=float(x.min()),
            cens_hi=float((x == x.max()).mean()), cens_lo=float((x == x.min()).mean())))
    return pd.DataFrame(rows)
PP = pd.concat([passport(master, FEATS_BASE, y) for y in CFG["years"]], ignore_index=True)
PP.to_parquet(PROC / "feature_passport.parquet", index=False)
pp = (PP.groupby("feature")[["top1_share","cv","cens_hi","cens_lo","nunique"]].mean()
        .sort_values("top1_share", ascending=False))
pp = pp.rename(columns={"nunique": "n_unq"})          # nunique — метод DataFrame, переименовываем
print("\n=== ПАСПОРТ ПРИЗНАКОВ (среднее по годам) ===")
print(pp.round(4).to_string())
BAD = set(pp[pp.top1_share > CFG["tol_top1"]].index) | \
      set(pp[pp.cens_hi > CFG["tol_cens_hi"]].index) | \
      set(pp[pp.n_unq < 50].index)
BAD &= set(FEATS_BASE)
FEATS = [f for f in FEATS_BASE if f not in BAD]
print(f"\nОТБРАКОВАНО: {sorted(BAD) if BAD else '—'}")
if BAD:
    for b in sorted(BAD):
        r = pp.loc[b]
        print(f"   {b:16s} top1={r.top1_share:.3f} cens_hi={r.cens_hi:.3f} "
              f"n_unq={r.n_unq:.0f} cv={r.cv:.2f}")
assert len(FEATS) >= CFG["min_features"], f"После отбраковки осталось {len(FEATS)} признаков"
print("ИТОГОВЫЙ НАБОР:", FEATS)
tick(f"паспортизация завершена: {len(FEATS)} признаков")

[ 905.6s] сборка мастер-панели 2019-2024
панель: (15282, 17), МО 2572, лет 6

пропуски по показателям (доля):
pop_urban     0.288
invest        0.255
bad_roads     0.204
migr          0.187
tax_share     0.177
msu_exp       0.172
built_pc      0.119
housing_pc    0.111
retail        0.102
salary        0.099
pop_total     0.000
  признак natural недоступен — natural_rate не строится

Базовые признаки: ['log_pop', 'rel_pop_region', 'urban_share', 'invest_pc', 'housing_pc', 'built_pc', 'bad_roads', 'msu_exp_pc', 'salary', 'retail_pc']
Дополнительные (профилирование / валидация): ['tax_share', 'migr_rate']

=== ПАСПОРТ ПРИЗНАКОВ (среднее по годам) ===
                top1_share        cv  cens_hi  cens_lo    n_unq
feature                                                        
urban_share         0.0502    0.4443   0.0023   0.0502   8212.0
built_pc            0.0365    1.9205   0.0001   0.0269   1654.0
bad_roads           0.0123    0.5536   0.0001   0.0001   2684.0
housing_pc          0.0

In [5]:
# ============================================================================
# ============ ЯЧЕЙКА 4 ============
# ============================================================================
# ============================================================================
# ЯЧЕЙКА 4. ЧИСТАЯ ВЫБОРКА
# ============================================================================
tick("подготовка выборки")
# --- 4.1 требуем, чтобы у МО был РЯД, т.е. данные в >=min_obs_share лет ------
# (не требуем «все признаки во все годы» — иначе пересечение пусто: у части
#  показателей покрытие 70-90 %, это особенность самого Росстата)
present = master[FEATS].notna()
master["n_feat"] = present.sum(axis=1)
min_feats = int(np.ceil(0.6 * len(FEATS)))
master["ok_row"] = master.n_feat >= min_feats
counts = master[master.ok_row].groupby("oktmo_stable").size()
good_mo = counts[counts >= CFG["min_obs_share"] * len(CFG["years"])].index
df = master[master.oktmo_stable.isin(good_mo)].copy()
print(f"МО с рядом: {df.oktmo_stable.nunique()} из {master.oktmo_stable.nunique()} "
      f"(отсеяно {master.oktmo_stable.nunique() - df.oktmo_stable.nunique()})")
print(f"наблюдений: {len(df)}, годы: {sorted(df.year.unique())}")
assert df.oktmo_stable.nunique() > 500, "Слишком мало МО для устойчивой кластеризации"
# --- 4.2 логарифмы и winsorize --------------------------------------------
LOGF = [f for f in CFG["log_features"] if f in FEATS]
for f in LOGF:
    if (df[f].dropna() >= 0).all():
        df[f] = np.log1p(df[f])
print("логарифмированы:", LOGF)
# страховка: inf/-inf -> NaN, иначе медианы «портятся» и импутация не срабатывает
for f in FEATS:
    bad = ~np.isfinite(df[f])
    if bad.any():
        print(f"   {f}: {int(bad.sum())} неконечных значений -> NaN")
        df.loc[bad, f] = np.nan
for f in FEATS:
    lo = df.groupby("year")[f].transform(lambda s: s.quantile(CFG["winsor"][0]))
    hi = df.groupby("year")[f].transform(lambda s: s.quantile(CFG["winsor"][1]))
    df[f] = df[f].clip(lo, hi)
# --- 4.3 импутация пропусков: медиана «регион × год» -> медиана года -> общая
print("\n=== ИМПУТАЦИЯ ПРОПУСКОВ (регион×год -> год -> вся выборка) ===")
IMP_LOG = []
for f in FEATS:
    before = int(df[f].isna().sum())
    if before:
        df[f] = df[f].fillna(df.groupby(["region_name","year"])[f].transform("median"))
        df[f] = df[f].fillna(df.groupby("year")[f].transform("median"))
        df[f] = df[f].fillna(df[f].median())
        after = int(df[f].isna().sum())
        IMP_LOG.append(dict(feature=f, was_missing=before, left=after,
                            share_pct=round(100 * before / len(df), 2)))
    assert df[f].notna().all(), f"после импутации остались пропуски в {f}"
print(pd.DataFrame(IMP_LOG).to_string(index=False) if IMP_LOG else "пропусков не было")
# --- 4.4 NaN-безопасная стандартизация ВНУТРИ года -------------------------
ZCOLS = [f + "_z" for f in FEATS]
for f, z in zip(FEATS, ZCOLS):
    mu = df.groupby("year")[f].transform("mean")
    sd = df.groupby("year")[f].transform("std").replace(0, 1.0)
    df[z] = (df[f] - mu) / sd
assert np.isfinite(df[ZCOLS].values).all(), "в z-признаках NaN/inf"
dz = df.copy()
print(dz.groupby("year").size().to_string())
dz.to_parquet(PROC / "panel_annual_features.parquet", index=False)
tick("выборка готова")

[ 906.1s] подготовка выборки
МО с рядом: 2022 из 2572 (отсеяно 550)
наблюдений: 12132, годы: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
логарифмированы: ['invest_pc', 'housing_pc', 'msu_exp_pc', 'salary', 'retail_pc']
   urban_share: 4060 неконечных значений -> NaN
   invest_pc: 2675 неконечных значений -> NaN
   housing_pc: 89 неконечных значений -> NaN
   built_pc: 441 неконечных значений -> NaN
   bad_roads: 1076 неконечных значений -> NaN
   msu_exp_pc: 596 неконечных значений -> NaN
   salary: 185 неконечных значений -> NaN
   retail_pc: 853 неконечных значений -> NaN

=== ИМПУТАЦИЯ ПРОПУСКОВ (регион×год -> год -> вся выборка) ===
    feature  was_missing  left  share_pct
urban_share         4060     0      33.47
  invest_pc         2675     0      22.05
 housing_pc           89     0       0.73
   built_pc          441     0       3.64
  bad_roads         1076     0       8.87
 msu_exp_pc          596     0       4.91
     sal

In [6]:
# ============================================================================
# ============ ЯЧЕЙКА 4b ============
# ============================================================================
# ============================================================================
# ЯЧЕЙКА 4b. СРАВНЕНИЕ ПРОСТРАНСТВ ПРИЗНАКОВ:
#            raw / pca5 / pca8 / KAN(d_in → 32 → 8)
# ============================================================================
tick("сравнение представлений (raw / PCA5 / PCA8 / KAN)")

import torch
import torch.nn as nn
import torch.nn.functional as F_torch

y_rep = max(CFG["years"])
sub = dz[dz.year == y_rep].reset_index(drop=True)
X_raw = sub[ZCOLS].values
D_IN  = X_raw.shape[1]
print(f"  входное пространство: {X_raw.shape}  (D_IN={D_IN})")


def boot_stability(X, k, iters=25, seed=RNG):
    """Средний попарный ARI между разбиениями на bootstrap-подвыборках."""
    n = len(X); rr = np.random.RandomState(seed); labs = []
    for b in range(iters):
        idx = rr.choice(n, n, replace=True)
        labs.append(KMeans(n_clusters=k, n_init=5,
                           random_state=seed + b).fit(X[idx]).predict(X))
    ar = [adjusted_rand_score(labs[i], labs[j])
          for i in range(len(labs)) for j in range(i + 1, len(labs))]
    return float(np.mean(ar))


# ---------------------------------------------------------------------------
# KAN (Kolmogorov–Arnold Network): B-сплайны Кокса-де Бура + SiLU-ветвь.
# Реализация самодостаточна, без сторонних KAN-библиотек.
#
# Ключевое отличие от исходной версии 4b:
#   grid=10 (было 5) — удвоенное разрешение сплайна,
#   домен [-3, 3] согласован с z-стандартизованными данными.
# ---------------------------------------------------------------------------
class KANLayer(nn.Module):
    def __init__(self, in_dim, out_dim, grid=10, k=3, x_min=-3.0, x_max=3.0):
        super().__init__()
        self.grid, self.k = grid, k
        self.n_basis = grid + k
        self.register_buffer(
            "grid_pts",
            torch.linspace(x_min - k, x_max + k, grid + 2 * k + 1))
        bound = 1.0 / max(in_dim, 1) ** 0.5
        self.spline_w = nn.Parameter(bound * torch.randn(in_dim, out_dim, self.n_basis))
        self.base_w   = nn.Parameter(bound * torch.randn(in_dim, out_dim))
        self.scale    = nn.Parameter(torch.ones(1))

    def _bspline_basis(self, x):
        """Cox-de Boor рекурсия. x: (B, D) -> (B, D, n_basis)."""
        g  = self.grid_pts.to(x.device)
        xc = x.clamp(g[0].item() + 1e-6, g[-1].item() - 1e-6)
        Bs = ((xc.unsqueeze(-1) >= g[:-1].view(1, 1, -1)) &
              (xc.unsqueeze(-1) <  g[1:].view(1, 1, -1))).to(x.dtype)
        for deg in range(1, self.k + 1):
            n = Bs.shape[-1] - 1
            i = torch.arange(n, device=x.device)
            left  = (g[i + deg]     - g[i]).clamp(min=1e-12)
            right = (g[i + deg + 1] - g[i + 1]).clamp(min=1e-12)
            xv = xc.unsqueeze(-1)
            Bs = (Bs[..., :-1] * (xv - g[i].view(1, 1, -1)) / left.view(1, 1, -1)
                  + Bs[..., 1:] * (g[i + deg + 1].view(1, 1, -1) - xv) / right.view(1, 1, -1))
        return Bs

    def forward(self, x):
        B = self._bspline_basis(x)
        spline_out = torch.einsum("bdg,dog->bo", B, self.spline_w)
        base_out   = torch.einsum("bd,do->bo", F_torch.silu(x), self.base_w)
        return self.scale * spline_out + base_out


class KANEncoder(nn.Module):
    def __init__(self, d_in, d_hidden, d_out, grid=10, k=3):
        super().__init__()
        self.l1 = KANLayer(d_in,     d_hidden, grid=grid, k=k)
        self.l2 = KANLayer(d_hidden, d_out,    grid=grid, k=k)
    def forward(self, x):
        return self.l2(self.l1(x))


class KANDecoder(nn.Module):
    def __init__(self, d_in, d_hidden, d_out, grid=10, k=3):
        super().__init__()
        self.l1 = KANLayer(d_in,     d_hidden, grid=grid, k=k)
        self.l2 = KANLayer(d_hidden, d_out,    grid=grid, k=k)
    def forward(self, z):
        return self.l2(self.l1(z))


def train_kan_ae(X, d_hidden=32, d_out=8, grid=10, k=3,
                 epochs=150, lr=3e-3, weight_decay=1e-5, batch=256,
                 seed=RNG, verbose=True, t_budget_sec=300,
                 early_stop_patience=25, early_stop_tol=1e-5):
    """Обучает KAN-автоэнкодер X → Z → X_hat на CPU.
    Возвращает (Z_white, R², loss_history, elapsed_sec).

    Правки относительно первой версии:
      • d_out=8 (bottleneck 10 → 8; было 16 — расширение)
      • epochs=150, lr=3e-3 (было 35, 1e-3)
      • early stopping по loss (patience=25)
      • t_budget_sec=300 — жёсткий верхний предел
    """
    import time as _t
    torch.manual_seed(seed); np.random.seed(seed)
    device = torch.device("cpu")
    Xt  = torch.from_numpy(X.astype(np.float32)).to(device)
    d_in = X.shape[1]
    enc = KANEncoder(d_in, d_hidden, d_out, grid, k).to(device)
    dec = KANDecoder(d_out, d_hidden, d_in, grid, k).to(device)
    opt = torch.optim.AdamW(list(enc.parameters()) + list(dec.parameters()),
                            lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
    loss_hist, t_start = [], _t.time()
    n = len(Xt)
    best_loss, stale = float("inf"), 0

    for ep in range(epochs):
        perm = torch.randperm(n)
        ep_loss, nb = 0.0, 0
        for i in range(0, n, batch):
            xb = Xt[perm[i:i + batch]]
            loss = F_torch.mse_loss(dec(enc(xb)), xb)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(enc.parameters()) + list(dec.parameters()), max_norm=1.0)
            opt.step()
            ep_loss += loss.item(); nb += 1
        sched.step(); loss_hist.append(ep_loss / max(nb, 1))

        # --- early stopping по loss -----------------------------------------
        if loss_hist[-1] < best_loss - early_stop_tol:
            best_loss = loss_hist[-1]; stale = 0
        else:
            stale += 1
        if stale >= early_stop_patience:
            if verbose:
                print(f"    KAN: early stop на эпохе {ep+1} "
                      f"(нет улучшения {early_stop_patience} эпох, best={best_loss:.6f})")
            break

        if verbose and ((ep + 1) % 10 == 0 or ep == 0):
            print(f"    KAN epoch {ep+1:>3}/{epochs}  loss={loss_hist[-1]:.6f}  "
                  f"elapsed={_t.time()-t_start:.1f}s")

        if _t.time() - t_start > t_budget_sec:
            print(f"    KAN: достигнут бюджет {t_budget_sec}s — останов на эпохе {ep+1}")
            break

    with torch.no_grad():
        X_hat = dec(enc(Xt)).cpu().numpy()
        Z     = enc(Xt).cpu().numpy()

    r2 = 1.0 - np.mean((X - X_hat) ** 2) / (np.mean((X - X.mean(0)) ** 2) + 1e-12)

    # --- whitening: обнуляем среднее, нормируем std -------------------------
    mu = Z.mean(0, keepdims=True)
    sd = Z.std(0,  keepdims=True) + 1e-9
    Z_white = ((Z - mu) / sd).astype(np.float32)

    return Z_white, float(r2), loss_hist, float(_t.time() - t_start)


def _pairwise_cos(Z, n=500, seed=RNG):
    """Средняя попарная cosine-близость (для диагностики коллапса)."""
    idx = np.random.RandomState(seed).choice(len(Z), min(n, len(Z)), replace=False)
    Zs = Z[idx]
    Zn = Zs / (np.linalg.norm(Zs, axis=1, keepdims=True) + 1e-9)
    C = Zn @ Zn.T
    return C[np.triu_indices(len(idx), 1)]


def sil_of(Z, k=2, seed=RNG):
    """Silhouette для k-кластерного разбиения в представлении Z."""
    lab = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(Z)
    if len(np.unique(lab)) < 2:
        return float("nan")
    return float(silhouette_score(Z, lab))


# ---------------------------------------------------------------------------
# Сравнение представлений: raw / pca5 / pca8 / KAN
# ---------------------------------------------------------------------------
REP = {}
REP["raw"]  = X_raw
REP["pca5"] = PCA(n_components=min(5,  X_raw.shape[1]), random_state=RNG).fit_transform(X_raw)
REP["pca8"] = PCA(n_components=min(8,  X_raw.shape[1]), random_state=RNG).fit_transform(X_raw)
print(f"  PCA5: {REP['pca5'].shape} | PCA8: {REP['pca8'].shape}")

tick(f"обучение KAN-энкодера ({D_IN} → 32 → 8, grid=10, whitening)")
Z_kan, r2_kan, loss_kan, dt_kan = train_kan_ae(
    X_raw,
    d_hidden=32, d_out=8, grid=10, k=3,
    epochs=150, lr=3e-3, weight_decay=1e-5, batch=256,
    seed=RNG, verbose=True, t_budget_sec=300,
    early_stop_patience=25, early_stop_tol=1e-5,
)
cos_kan = _pairwise_cos(Z_kan)
sil_kan = sil_of(Z_kan, k=2)
print(f"  KAN: R²={r2_kan:.4f} | loss {loss_kan[0]:.4f} → {loss_kan[-1]:.4f} | "
      f"whitened cos mean={cos_kan.mean():.4f} std={cos_kan.std():.4f} | "
      f"dead_dims={(Z_kan.std(0) < 1e-4).sum()} | silhouette(k=2)={sil_kan:.4f} | "
      f"t={dt_kan:.1f}s")

REP["kan"] = Z_kan

# ---------------------------------------------------------------------------
# Проверки (обновлённые)
# ---------------------------------------------------------------------------
# R² автоэнкодера — вторичный критерий: при d_in=10, d_out=8, N≈2000
# значение 0.65–0.85 — норма. Главное — кластерная структура в Z.
assert r2_kan >= 0.70, f"KAN R² слишком низкий: {r2_kan:.4f} (< 0.70)"
assert abs(cos_kan.mean()) < 0.15, f"KAN whitening не сработал: cos={cos_kan.mean():.4f}"
assert (Z_kan.std(0) < 1e-4).sum() == 0, "KAN: есть мёртвые размерности"
assert not np.isnan(sil_kan) and sil_kan >= 0.05, \
    f"KAN-эмбеддинги не дают осмысленной структуры: silhouette={sil_kan:.4f}"
print(f"  все проверки KAN пройдены: R²={r2_kan:.4f}, cos={cos_kan.mean():.4f}, sil={sil_kan:.4f}")

# ---------------------------------------------------------------------------
# Сводное сравнение представлений
# ---------------------------------------------------------------------------
rep_rows = []
for nm, X in REP.items():
    for k in (2, 3, 4, 5):
        if k >= len(X):
            continue
        labk = KMeans(n_clusters=k, n_init=10, random_state=RNG).fit_predict(X)
        if len(np.unique(labk)) < 2:
            continue
        rep_rows.append(dict(
            representation=nm,
            dim=X.shape[1],
            K=k,
            silhouette=round(float(silhouette_score(X, labk)), 4),
            boot_ari=round(boot_stability(X, k, iters=25), 4),
            sizes=json.dumps(np.bincount(labk, minlength=k).tolist()),
        ))
REPCMP = pd.DataFrame(rep_rows)
REPCMP.to_parquet(PROC / "representation_comparison.parquet", index=False)

print("\n=== СРАВНЕНИЕ ПРЕДСТАВЛЕНИЙ (год %d) ===" % y_rep)
print(REPCMP.to_string(index=False))

best = (REPCMP[REPCMP.K <= 4].groupby("representation")
        .agg(boot=("boot_ari", "mean"), sil=("silhouette", "mean"))
        .assign(score=lambda d: d.boot)
        .sort_values("score", ascending=False))
print("\nсводно по представлениям:")
print(best.round(4).to_string())

REP_CHOICE = best.index[0]
print(f"\n>>> для выбора K используем представление: {REP_CHOICE}")

# ---------------------------------------------------------------------------
# Сохранение KAN-эмбеддингов
# ---------------------------------------------------------------------------
kan_cols = [f"kan_e{i}" for i in range(Z_kan.shape[1])]
kan_df = pd.concat(
    [sub[["oktmo_stable", "year"]].reset_index(drop=True),
     pd.DataFrame(Z_kan, columns=kan_cols)],
    axis=1,
)
kan_df.to_parquet(PROC / f"kan_embeddings_y{y_rep}.parquet", index=False)
print(f"KAN-эмбеддинги сохранены: {kan_df.shape} → kan_embeddings_y{y_rep}.parquet")

# ---------------------------------------------------------------------------
# Диагностический дашборд по KAN
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(range(1, len(loss_kan) + 1), loss_kan, "o-", color="navy")
ax[0].set_xlabel("Эпоха"); ax[0].set_ylabel("MSE loss")
ax[0].set_title(f"KAN loss (R²={r2_kan:.3f})"); ax[0].grid(alpha=.3)

ax[1].hist(cos_kan, bins=40, color="steelblue", alpha=.8)
ax[1].axvline(0, color="r", ls="--", lw=1)
ax[1].axvline(cos_kan.mean(), color="k", ls=":", lw=1,
              label=f"mean={cos_kan.mean():.3f}")
ax[1].set_xlabel("Pairwise cosine (whitened)"); ax[1].set_ylabel("Частота")
ax[1].set_title("KAN: cosine-распределение"); ax[1].legend(fontsize=9); ax[1].grid(alpha=.3)

pca2_diag = PCA(n_components=2, random_state=RNG).fit_transform(Z_kan)
ax[2].scatter(pca2_diag[:, 0], pca2_diag[:, 1], s=6, alpha=.5, color="darkgreen")
ax[2].set_xlabel("PC1 (KAN)"); ax[2].set_ylabel("PC2 (KAN)")
ax[2].set_title(f"KAN 2D-проекция (sil={sil_kan:.3f})"); ax[2].grid(alpha=.3)

plt.tight_layout()
plt.savefig(VIZ / "00_kan_diagnostics.png", dpi=110)
plt.close()
print(f"диагностика сохранена: viz/00_kan_diagnostics.png")

tick("представления сравнены (KAN 10→32→8, grid=10, early stop)")

[ 906.6s] сравнение представлений (raw / PCA5 / PCA8 / KAN)
  входное пространство: (2022, 10)  (D_IN=10)
  PCA5: (2022, 5) | PCA8: (2022, 8)
[ 910.6s] обучение KAN-энкодера (10 → 32 → 8, grid=10, whitening)
    KAN epoch   1/150  loss=1.016783  elapsed=0.4s
    KAN epoch  10/150  loss=0.203045  elapsed=1.8s
    KAN epoch  20/150  loss=0.096393  elapsed=3.2s
    KAN epoch  30/150  loss=0.059703  elapsed=4.8s
    KAN epoch  40/150  loss=0.040207  elapsed=6.3s
    KAN epoch  50/150  loss=0.029442  elapsed=7.8s
    KAN epoch  60/150  loss=0.022629  elapsed=9.3s
    KAN epoch  70/150  loss=0.018312  elapsed=10.9s
    KAN epoch  80/150  loss=0.015584  elapsed=12.4s
    KAN epoch  90/150  loss=0.013773  elapsed=13.9s
    KAN epoch 100/150  loss=0.012571  elapsed=15.4s
    KAN epoch 110/150  loss=0.011772  elapsed=16.9s
    KAN epoch 120/150  loss=0.011280  elapsed=18.5s
    KAN epoch 130/150  loss=0.011022  elapsed=20.0s
    KAN epoch 140/150  loss=0.010918  elapsed=21.5s
    KAN epoch 150/1

In [7]:
# ============================================================================
# ============ ЯЧЕЙКА 5 ============
# ============================================================================
# ============================================================================
# ЯЧЕЙКА 5. ВЫБОР K ПО УСТОЙЧИВОСТИ (на KAN-эмбеддингах)
# ============================================================================
# Изменения относительно v2:
#   • ZCOLS заменены на KAN-эмбеддинги (KAN_COLS);
#   • KAN обучен на пуле ВСЕХ лет (2019–2024), а не только на 2024,
#     чтобы эмбеддинги были сопоставимы между годами;
#   • вся логика bootstrap/silhouette/jackknife сохранена без изменений.
# ============================================================================
tick("выбор K (на KAN-эмбеддингах)")

yrs = sorted(dz.year.unique())


# ---------------------------------------------------------------------------
# 5.0. Обучение KAN на пуле всех лет
# ---------------------------------------------------------------------------
# Используем функцию train_kan_ae, определённую в ячейке 4b. Если ячейка 5
# запускается в новой сессии без 4b — раскомментируйте блок определения
# ниже (он дублирует реализацию из 4b).
#
# if "train_kan_ae" not in globals():
#     raise RuntimeError(
#         "train_kan_ae не найдена. Сначала выполните ячейку 4b."
#     )

print(f"обучение KAN на пуле всех лет: {len(dz)} наблюдений × {len(ZCOLS)} признаков")

# Пул: все строки dz во всех годах, вход — per-year z-признаки (ZCOLS).
# Per-year z-оценки уже сопоставимы между годами (mu=0, sd=1 в каждом),
# поэтому пул — валидный вход для одного энкодера.
X_pool = dz[ZCOLS].values
assert np.isfinite(X_pool).all(), "в пуле X есть NaN/inf"

tick(f"обучение KAN-энкодера на пуле ({X_pool.shape[0]} × {X_pool.shape[1]})")
Z_pool, r2_pool, loss_pool, dt_pool = train_kan_ae(
    X_pool,
    d_hidden=32, d_out=8, grid=10, k=3,
    epochs=150, lr=3e-3, weight_decay=1e-5, batch=512,
    seed=RNG, verbose=True, t_budget_sec=300,
    early_stop_patience=25, early_stop_tol=1e-5,
)
cos_pool = _pairwise_cos(Z_pool)
sil_pool = sil_of(Z_pool, k=2)
print(f"  KAN-пул: R²={r2_pool:.4f} | loss {loss_pool[0]:.4f} → {loss_pool[-1]:.4f} | "
      f"cos mean={cos_pool.mean():.4f} | sil(k=2)={sil_pool:.4f} | t={dt_pool:.1f}s")

assert r2_pool >= 0.70, f"KAN-пул: R² слишком низкий ({r2_pool:.4f})"
assert abs(cos_pool.mean()) < 0.15, f"KAN-пул: whitening не сработал (cos={cos_pool.mean():.4f})"
assert (Z_pool.std(0) < 1e-4).sum() == 0, "KAN-пул: есть мёртвые размерности"
assert sil_pool >= 0.10, f"KAN-пул: silhouette слишком низкий ({sil_pool:.4f})"
print(f"  проверки KAN-пула пройдены")

# ---------------------------------------------------------------------------
# 5.0b. Добавление KAN-эмбеддингов в dz
# ---------------------------------------------------------------------------
KAN_COLS = [f"kan_e{i}" for i in range(Z_pool.shape[1])]
kan_pool_df = pd.DataFrame(Z_pool, columns=KAN_COLS)
kan_pool_df["oktmo_stable"] = dz["oktmo_stable"].values
kan_pool_df["year"]         = dz["year"].values

# dz — тот же порядок строк, что и X_pool (мы взяли dz[ZCOLS].values),
# поэтому merge по (oktmo, year) даст корректное соответствие.
dz = dz.drop(columns=[c for c in KAN_COLS if c in dz.columns], errors="ignore")
dz = dz.merge(kan_pool_df, on=["oktmo_stable", "year"], how="left")

assert dz[KAN_COLS].notna().all().all(), "KAN-эмбеддинги не слились в dz"
print(f"KAN-эмбеддинги добавлены в dz: {KAN_COLS}")
print(f"  dz: {dz.shape}")
dz.to_parquet(PROC / "panel_annual_features_kan.parquet", index=False)

# Сохраняем финальный энкодер и его метаданные
torch.save({
    "kan_cfg": {"d_hidden": 32, "d_out": 8, "grid": 10, "k": 3},
    "loss_history": loss_pool,
    "r2_reconstruction": float(r2_pool),
    "silhouette_k2": float(sil_pool),
    "kan_cols": KAN_COLS,
    "pool_size": int(X_pool.shape[0]),
    "d_in": int(X_pool.shape[1]),
}, PROC / "kan_encoder_pool.pt")
print(f"  KAN-чекпойнт сохранён: kan_encoder_pool.pt")

# ---------------------------------------------------------------------------
# 5.0c. Переключаем ZCOLS → KAN_COLS
# ---------------------------------------------------------------------------
ZCOLS_USE = KAN_COLS
print(f"  используем представление: KAN ({len(ZCOLS_USE)} размерностей)")


# ---------------------------------------------------------------------------
# 5.1. Вспомогательная функция fit (без изменений)
# ---------------------------------------------------------------------------
def fit(X, k, seed=RNG):
    km = KMeans(n_clusters=k, n_init=10, random_state=seed)
    return km.fit_predict(X), km


# ---------------------------------------------------------------------------
# 5.2. Качество по годам (на KAN-эмбеддингах)
# ---------------------------------------------------------------------------
tick("качество по K на KAN-эмбеддингах")
qual = []
for k in CFG["k_range"]:
    sil, ch, db, mn = [], [], [], []
    for y in yrs:
        X = dz[dz.year == y][ZCOLS_USE].values
        lab, _ = fit(X, k)
        sil.append(silhouette_score(X, lab))
        ch.append(calinski_harabasz_score(X, lab))
        db.append(davies_bouldin_score(X, lab))
        mn.append(np.bincount(lab, minlength=k).min() / len(lab))
    qual.append(dict(K=k, silhouette=np.mean(sil), CH=np.mean(ch), DB=np.mean(db),
                     min_cluster_share=np.mean(mn)))
QUAL = pd.DataFrame(qual)
QUAL.to_parquet(PROC / "k_selection_quality.parquet", index=False)
print("=== КАЧЕСТВО ПО K (KAN) ===")
print(QUAL.round(4).to_string(index=False))


# ---------------------------------------------------------------------------
# 5.3. Bootstrap-устойчивость по годам (на KAN-эмбеддингах)
# ---------------------------------------------------------------------------
tick("bootstrap-ARI (KAN)")
boot = []
for k in CFG["k_range"]:
    per_year = []
    for y in yrs:
        X = dz[dz.year == y][ZCOLS_USE].values
        per_year.append(boot_stability(X, k, iters=CFG["bootstrap_iters"],
                                       seed=RNG + 7 * y + k))
    boot.append(dict(K=k, boot_ari=float(np.mean(per_year)),
                     boot_ari_min=float(np.min(per_year))))
BS = pd.DataFrame(boot)
print("\n=== BOOTSTRAP-УСТОЙЧИВОСТЬ (KAN) ===")
print(BS.round(4).to_string(index=False))


# ---------------------------------------------------------------------------
# 5.4. Jackknife по регионам (на KAN-эмбеддингах)
# ---------------------------------------------------------------------------
tick("jackknife по регионам (KAN)")
y_ref = yrs[-1]
sub_ref = dz[dz.year == y_ref].reset_index(drop=True)
X_ref = sub_ref[ZCOLS_USE].values
reg_ref = sub_ref.region_name.values
reg_sizes = pd.Series(reg_ref).value_counts()

# берём регионы с достаточным числом МО: не меньше max(30, 2 % выборки),
# иначе jackknife вырождается (leave-one-region-out при малых регионах
# ничего не проверяет).
min_reg = max(30, int(0.02 * len(X_ref)))
regs = reg_sizes[reg_sizes >= min_reg].index.tolist()
print(f"  регионов всего {reg_sizes.size}, в jackknife {len(regs)} "
      f"(порог >= {min_reg} МО; медиана {int(reg_sizes.median())} МО)")

jk = []
for k in CFG["k_range"]:
    lab_ref, _ = fit(X_ref, k)
    a, a_min = [], 1.0
    for r in regs:
        m = reg_ref != r
        if m.sum() < 50:
            continue
        lab_r, _ = fit(X_ref[m], k)
        v = adjusted_rand_score(lab_ref[m], lab_r)
        a.append(v); a_min = min(a_min, v)
    jk.append(dict(K=k, jackknife_ari=float(np.mean(a)) if a else np.nan,
                   jackknife_min=float(a_min) if a else np.nan,
                   n_regions=len(a)))
JK = pd.DataFrame(jk)
assert JK.jackknife_ari.notna().any(), "jackknife не посчитался ни для одного K"
print("\n=== JACKKNIFE ПО РЕГИОНАМ (KAN, leave-one-region-out) ===")
print(JK.round(4).to_string(index=False))


# ---------------------------------------------------------------------------
# 5.5. Сводное решение (без изменений)
# ---------------------------------------------------------------------------
SEL = QUAL.merge(BS, on="K").merge(JK, on="K")
s = SEL.silhouette.values
sel_n = (s - s.min()) / (s.max() - s.min() + 1e-12)
SEL["score"] = SEL.boot_ari * (0.5 + 0.5 * sel_n) * SEL.jackknife_ari
SEL["degenerate"] = SEL.min_cluster_share < 0.02

valid = SEL[~SEL.degenerate].dropna(subset=["score"])
if not len(valid):
    valid = SEL.dropna(subset=["score"])
if not len(valid):
    valid = SEL.fillna({"score": -np.inf})
K_STAR = int(valid.loc[valid.score.idxmax(), "K"])
SEL.to_parquet(PROC / "k_selection_summary.parquet", index=False)
print("\n=== СВОДКА (KAN) ===")
print(SEL.round(4).to_string(index=False))

row = SEL.set_index("K").loc[K_STAR]
print(f"\n>>> K* = {K_STAR} | bootstrap ARI {row.boot_ari:.3f} | "
      f"silhouette {row.silhouette:.3f} | jackknife {row.jackknife_ari:.3f}")


# ---------------------------------------------------------------------------
# 5.6. Диагностический дашборд (расширен: +KAN-диагностика)
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
ax[0,0].plot(SEL.K, SEL.silhouette, "o-", color="C0")
ax[0,0].set_title("Silhouette по K (KAN)")
ax[0,1].plot(SEL.K, SEL.boot_ari, "o-", color="g")
ax[0,1].set_title("Bootstrap-ARI (KAN)")
ax[1,0].plot(SEL.K, SEL.jackknife_ari, "o-", color="purple")
ax[1,0].set_title("Jackknife по регионам (KAN)")
ax[1,1].plot(SEL.K, SEL.score, "o-", color="orange")
ax[1,1].set_title(f"Итоговый скор (K*={K_STAR})")
for a in ax.ravel():
    a.axvline(K_STAR, color="r", ls="--"); a.set_xlabel("K"); a.grid(alpha=.3)
plt.tight_layout()
plt.savefig(VIZ / "01_k_selection_kan.png", dpi=110)
plt.close()
print(f"дашборд сохранён: viz/01_k_selection_kan.png")

tick("K выбран на KAN-эмбеддингах")

[ 956.6s] выбор K (на KAN-эмбеддингах)
обучение KAN на пуле всех лет: 12132 наблюдений × 10 признаков
[ 956.6s] обучение KAN-энкодера на пуле (12132 × 10)
    KAN epoch   1/150  loss=0.836887  elapsed=0.6s
    KAN epoch  10/150  loss=0.076396  elapsed=6.1s
    KAN epoch  20/150  loss=0.034337  elapsed=12.3s
    KAN epoch  30/150  loss=0.025328  elapsed=18.6s
    KAN epoch  40/150  loss=0.021103  elapsed=24.7s
    KAN epoch  50/150  loss=0.018891  elapsed=30.9s
    KAN epoch  60/150  loss=0.017475  elapsed=37.0s
    KAN epoch  70/150  loss=0.016529  elapsed=43.1s
    KAN epoch  80/150  loss=0.015847  elapsed=49.3s
    KAN epoch  90/150  loss=0.015389  elapsed=55.4s
    KAN epoch 100/150  loss=0.014952  elapsed=61.7s
    KAN epoch 110/150  loss=0.014722  elapsed=67.8s
    KAN epoch 120/150  loss=0.014531  elapsed=74.1s
    KAN epoch 130/150  loss=0.014406  elapsed=80.3s
    KAN epoch 140/150  loss=0.014316  elapsed=86.4s
    KAN epoch 150/150  loss=0.014311  elapsed=92.7s
  KAN-пул: R²=0

In [8]:
# ============================================================================
# ============ ЯЧЕЙКА 6 ============
# ============================================================================
# ============================================================================
# ЯЧЕЙКА 6. РАЗБИЕНИЯ ПО ГОДАМ + СОГЛАСОВАНИЕ МЕТОК (KAN-эмбеддинги)
# ============================================================================
# Изменения относительно v2:
#   • Кластеризация на KAN-эмбеддингах (ZCOLS_USE = KAN_COLS);
#   • Согласование меток через центроиды в KAN-пространстве — том же,
#     где проводилась кластеризация (иначе семантика «центроид» теряется);
#   • Сохранение центроидов по годам (для L2 и диагностики);
#   • Проверки: ARI до/после согласования, стабильность размеров кластеров.
# ============================================================================
tick("кластеризация по годам (KAN-эмбеддинги)")

# --- Проверки доступности переменных из предыдущих ячеек -----------------
for _var in ("dz", "df", "fit", "K_STAR", "yrs", "ZCOLS_USE", "OKTMO_REF",
             "FEATS", "EXTRA_PROFILE"):
    assert _var in globals(), (
        f"переменная {_var} не найдена — выполните ячейки 0–5 перед ячейкой 6"
    )
K = K_STAR
assert K >= 2, f"K* должен быть ≥ 2, получено {K}"
print(f"  K* = {K}  |  МО = {dz.oktmo_stable.nunique()}  |  лет = {len(yrs)}")
print(f"  пространство кластеризации: {len(ZCOLS_USE)} KAN-размерностей")


# ---------------------------------------------------------------------------
# 6.1. Кластеризация каждого года в KAN-пространстве
# ---------------------------------------------------------------------------
labels, cents, sizes = {}, {}, {}
print("\n=== КЛАСТЕРИЗАЦИЯ ПО ГОДАМ ===")
for y in yrs:
    suby = dz[dz.year == y].reset_index(drop=True)
    lab, km = fit(suby[ZCOLS_USE].values, K)
    labels[y] = pd.Series(lab, index=suby.oktmo_stable.values)
    cents[y]  = km.cluster_centers_
    sizes[y]  = np.bincount(lab, minlength=K).tolist()
    sil_y = silhouette_score(suby[ZCOLS_USE].values, lab)
    print(f"  {y}: размеры {sizes[y]}  silhouette={sil_y:.4f}")


# ---------------------------------------------------------------------------
# 6.2. Согласование меток между годами (Hungarian на KAN-центроидах)
# ---------------------------------------------------------------------------
# Метки KMeans «плавают» между годами: K0 в 2019 и K0 в 2020 могут
# обозначать разные типы. Выравниваем через линейное назначение по
# расстоянию между центроидами: cost(i,j) = ||cent_prev[i] - cent_curr[j]||².
# Всё считаем в KAN-пространстве — там же, где обучался KMeans.
print("\n=== СОГЛАСОВАНИЕ МЕТОК (Hungarian, KAN-пространство) ===")

aligned = {yrs[0]: labels[yrs[0]]}                 # базовая система
cent_aligned = {yrs[0]: cents[yrs[0]].copy()}      # центроиды в согласованной системе
perms = {}                                         # карты перестановок по годам

for prev_y, y in zip(yrs, yrs[1:]):
    # --- центроиды предыдущего года в СОГЛАСОВАННОЙ системе ---------------
    # Считаем их по фактическим данным, а не через km.cluster_centers_:
    # aligned[prev_y] уже может отличаться от labels[prev_y] после сдвига.
    prev_lab = aligned[prev_y]
    sub_prev = dz[dz.year == prev_y].set_index("oktmo_stable")
    cent_prev = np.vstack([
        sub_prev.loc[prev_lab[prev_lab == k].index, ZCOLS_USE].mean().values
        for k in range(K)
    ])                                              # (K, d_KAN)

    # --- матрица стоимостей и Hungarian -----------------------------------
    D = np.linalg.norm(cent_prev[:, None, :] - cents[y][None, :, :], axis=2)
    r, c = linear_sum_assignment(D)
    # mapping: original_label_curr -> aligned_label
    mapping = {int(c[i]): int(r[i]) for i in range(K)}
    perms[y] = mapping

    # --- применяем перестановку к текущему году ---------------------------
    aligned[y] = labels[y].map(mapping)
    cent_aligned[y] = cents[y][[mapping[k] for k in range(K)]]

    # --- диагностика: ARI до/после согласования ---------------------------
    # Если ARI_raw ≈ 1.0 — метки не перемешались, согласование тривиально.
    # Если ARI_aligned заметно выше ARI_raw — согласование сработало.
    common = np.intersect1d(labels[prev_y].index, labels[y].index)
    ari_raw = adjusted_rand_score(
        labels[prev_y].reindex(common).values,
        labels[y].reindex(common).values)
    ari_align = adjusted_rand_score(
        aligned[prev_y].reindex(common).values,
        aligned[y].reindex(common).values)
    print(f"  {prev_y}->{y}: mapping={mapping}  "
          f"ARI(raw)={ari_raw:.4f}  ARI(aligned)={ari_align:.4f}")


# ---------------------------------------------------------------------------
# 6.3. Проверки согласования
# ---------------------------------------------------------------------------
# (а) в каждом году присутствуют все K меток
for y in yrs:
    uniq = sorted(int(x) for x in aligned[y].unique())
    assert uniq == list(range(K)), f"{y}: метки {uniq}, ожидалось {list(range(K))}"

# (б) базовый год не был изменён согласованием
assert (aligned[yrs[0]].values == labels[yrs[0]].values).all(), \
    "базовый год не должен был меняться при согласовании"

# (в) средний ARI между соседними годами после согласования ≥ 0.5
ari_pairs = []
for prev_y, y in zip(yrs, yrs[1:]):
    common = np.intersect1d(labels[prev_y].index, labels[y].index)
    ari_pairs.append(adjusted_rand_score(
        aligned[prev_y].reindex(common).values,
        aligned[y].reindex(common).values))
mean_ari = float(np.mean(ari_pairs))
print(f"\nсредний ARI между соседними годами (aligned): {mean_ari:.4f}")
assert mean_ari >= 0.5, \
    f"средний ARI между соседними годами слишком низкий: {mean_ari:.4f}"


# ---------------------------------------------------------------------------
# 6.4. Сборка LAB: длинный формат (МО × год → cluster)
# ---------------------------------------------------------------------------
LAB = pd.concat([
    pd.DataFrame({
        "oktmo_stable": aligned[y].index,
        "year":         y,
        "cluster":      np.asarray(aligned[y].values).astype(int),
    }) for y in yrs
], ignore_index=True)

# --- метаданные ОКТМО ---
LAB = LAB.merge(OKTMO_REF, on="oktmo_stable", how="left")
for _c in ("municipality", "mun_type", "region_name"):
    if _c in LAB.columns:
        LAB[_c] = LAB[_c].astype("object")

# --- сырые признаки (для интерпретации профилей) ---
LAB = LAB.merge(df[["oktmo_stable", "year"] + FEATS + EXTRA_PROFILE],
                on=["oktmo_stable", "year"], how="left")

# --- KAN-эмбеддинги (для визуализации, сети, валидации) ---
LAB = LAB.merge(dz[["oktmo_stable", "year"] + ZCOLS_USE],
                on=["oktmo_stable", "year"], how="left")

assert LAB.cluster.notna().all(), "в LAB есть пропуски в cluster"
assert LAB[ZCOLS_USE].notna().all().all(), "в LAB есть пропуски в KAN-эмбеддингах"
assert LAB[FEATS].notna().all().all(), "в LAB есть пропуски в FEATS"


# ---------------------------------------------------------------------------
# 6.5. Сохранение
# ---------------------------------------------------------------------------
LAB.to_parquet(PROC / "cluster_labels_annual.parquet", index=False)
print(f"\n✅ cluster_labels_annual.parquet: {LAB.shape}")
print(f"   МО: {LAB.oktmo_stable.nunique()}, лет: {LAB.year.nunique()}, K = {K}")

# --- таблица центроидов по годам (в KAN-пространстве) ---
cent_rows = []
for y in yrs:
    for k in range(K):
        row = {"year": y, "cluster": k, "size": int(sizes[y][k])}
        for j, c in enumerate(ZCOLS_USE):
            row[c] = float(cent_aligned[y][k, j])
        cent_rows.append(row)
cent_df = pd.DataFrame(cent_rows)
cent_df.to_parquet(PROC / "kan_centroids_annual.parquet", index=False)
print(f"✅ kan_centroids_annual.parquet: {cent_df.shape}")

# --- пермутации по годам (для аудита воспроизводимости) ---
perm_rows = [{"year": y, "orig_label": k, "aligned_label": v}
             for y, m in perms.items() for k, v in m.items()]
pd.DataFrame(perm_rows).to_parquet(PROC / "label_perms.parquet", index=False)
print(f"✅ label_perms.parquet ({len(perm_rows)} записей)")


# ---------------------------------------------------------------------------
# 6.6. Диагностика
# ---------------------------------------------------------------------------
print("\n=== РАЗМЕРЫ КЛАСТЕРОВ ПО ГОДАМ (число МО) ===")
ct = pd.crosstab(LAB.year, LAB.cluster)
print(ct.to_string())

print("\n=== ДОЛИ КЛАСТЕРОВ, % ===")
ct_pct = (ct.div(ct.sum(axis=1), axis=0) * 100).round(2)
print(ct_pct.to_string())

# --- устойчивость размеров (коэффициент вариации по годам) ---
cv_sizes = (ct.std(ddof=0) / ct.mean()).round(4)
print(f"\nкоэффициент вариации размеров по годам:\n{cv_sizes.to_string()}")

# --- графики: размеры и динамика ---
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
cmapK = plt.get_cmap("tab10", K)
colors = [cmapK(c) for c in range(K)]

ax[0].stackplot(ct.index,
                [ct[c] for c in range(K)],
                labels=[f"K{c}" for c in range(K)],
                colors=colors, alpha=0.9)
ax[0].set_xlabel("Год"); ax[0].set_ylabel("Число МО")
ax[0].set_title("Размеры кластеров по годам")
ax[0].legend(loc="center left", bbox_to_anchor=(1.02, .5))
ax[0].grid(alpha=.3)

for c in range(K):
    ax[1].plot(ct.index, ct[c], "o-", color=colors[c],
               label=f"K{c}", lw=2, ms=6)
ax[1].set_xlabel("Год"); ax[1].set_ylabel("Число МО")
ax[1].set_title("Динамика размеров кластеров")
ax[1].legend(); ax[1].grid(alpha=.3)

plt.tight_layout()
plt.savefig(VIZ / "02_cluster_sizes_annual.png", dpi=110)
plt.close()
print(f"\n✅ viz/02_cluster_sizes_annual.png")

tick(f"разбиения готовы: K={K}, МО={LAB.oktmo_stable.nunique()}, "
     f"ARI(соседних лет)={mean_ari:.4f}")

[1170.2s] кластеризация по годам (KAN-эмбеддинги)
  K* = 2  |  МО = 2022  |  лет = 6
  пространство кластеризации: 8 KAN-размерностей

=== КЛАСТЕРИЗАЦИЯ ПО ГОДАМ ===
  2019: размеры [57, 1965]  silhouette=0.6825
  2020: размеры [57, 1965]  silhouette=0.6870
  2021: размеры [1965, 57]  silhouette=0.6851
  2022: размеры [1966, 56]  silhouette=0.6859
  2023: размеры [1967, 55]  silhouette=0.6883
  2024: размеры [57, 1965]  silhouette=0.6909

=== СОГЛАСОВАНИЕ МЕТОК (Hungarian, KAN-пространство) ===
  2019->2020: mapping={0: 0, 1: 1}  ARI(raw)=1.0000  ARI(aligned)=1.0000
  2020->2021: mapping={1: 0, 0: 1}  ARI(raw)=1.0000  ARI(aligned)=1.0000
  2021->2022: mapping={1: 0, 0: 1}  ARI(raw)=0.9904  ARI(aligned)=0.9904
  2022->2023: mapping={1: 0, 0: 1}  ARI(raw)=0.9317  ARI(aligned)=0.9317
  2023->2024: mapping={0: 0, 1: 1}  ARI(raw)=0.9613  ARI(aligned)=0.9613

средний ARI между соседними годами (aligned): 0.9767

✅ cluster_labels_annual.parquet: (12132, 27)
   МО: 2022, лет: 6, K = 2
✅ kan_ce

In [9]:
# ============================================================================
# ============ ЯЧЕЙКА 6a ============
# ============================================================================
# ============================================================================
# ЯЧЕЙКА 6a. L2: KAN-ЭНКОДЕР ДЛЯ КАЖДОГО L1-КЛАСТЕРА
# ============================================================================
# Зачем:
#   L1 нашёл грубое разбиение (urban ~2.8 % / rural ~97.2 %).
#   L2 ищет ВНУТРЕННЮЮ дифференциацию каждого L1-кластера — то, что
#   «смазано» усреднением в L1.
#
# Как:
#   Для каждого L1-кластера k обучается отдельный KAN-энкодер на
#   подвыборке МО этого класса (пул всех лет). Архитектура адаптивна:
#     • urban (~340 наблюдений) → KAN(d_in → 16 → 6)
#     • rural (~11800 наблюдений) → KAN(d_in → 32 → 8)
#
# Вход L2:
#   Сырые z-признаки (ZCOLS, 10-dim), а не L1-эмбеддинги. Причины:
#     (а) L2 не должен «наследовать» геометрию L1-энкодера — иначе
#         найдёт ту же структуру;
#     (б) сырые признаки интерпретируемы, что полезно для экономической
#         интерпретации подкластеров;
#     (в) L1 KAN имеет R²=0.985 — почти всё сохранено, но искажения
#         есть; L2 на сырых признаках даст независимую проверку.
#
# Выход:
#   Z_L2_k — эмбеддинги (n_k, d_out_k), добавленные в dz как kan_l2_k*_e*.
# ============================================================================
tick("обучение L2-KAN для каждого L1-кластера")

# --- Проверки доступности из предыдущих ячеек ----------------------------
for _var in ("dz", "ZCOLS", "ZCOLS_USE", "K", "yrs", "train_kan_ae",
             "_pairwise_cos", "sil_of", "PROC", "LAB"):
    assert _var in globals(), (
        f"переменная {_var} не найдена — выполните ячейки 0–6 перед 6a"
    )

# --- Гиперпараметры L2 ---------------------------------------------------
L2_CFG = {
    0: {"d_hidden": 16, "d_out": 6, "grid": 8,  "epochs": 100,
        "lr": 3e-3, "batch": 64,  "t_budget": 120},   # urban  (малый)
    1: {"d_hidden": 32, "d_out": 8, "grid": 10, "epochs": 150,
        "lr": 3e-3, "batch": 512, "t_budget": 240},   # rural  (большой)
}
L2_MIN_R2  = 0.65
L2_MIN_SIL = 0.05
L2_MAX_COS = 0.15

# ---------------------------------------------------------------------------
# 6a.0. САНИТАРНАЯ ОЧИСТКА dz ПЕРЕД ОБУЧЕНИЕМ L2
# ---------------------------------------------------------------------------
# Проблема: предыдущий (упавший) прогон 6a мог оставить в dz:
#   • дубликаты cluster_l1 (от повторного rename),
#   • остаточные kan_l2_k*_e* (от кластеров, которые успели обучиться),
#   • колонки с суффиксами cluster_l1_x / cluster_l1_y (от merge
#     с неявными суффиксами).
# Всё это делает dz.columns неуникальным — .loc и [] начинают
# возвращать DataFrame вместо Series, и пайплайн ломается.
# ===========================================================================

# (1) Уникальный RangeIndex — иначе .loc[bool_mask, col] = ... падает
#     с "cannot reindex on an axis with duplicate labels".
dz = dz.reset_index(drop=True)

# (2) Удаляем остатки L2-эмбеддингов от предыдущих запусков.
kan_l2_cols = [c for c in dz.columns if c.startswith("kan_l2_k")]
if kan_l2_cols:
    print(f"  очистка: удалены остаточные L2-колонки: {kan_l2_cols}")
    dz = dz.drop(columns=kan_l2_cols)

# (3) Консолидируем cluster_l1 / cluster_l1_x / cluster_l1_y в одну.
cluster_variants = [c for c in dz.columns
                    if c == "cluster_l1" or c.startswith("cluster_l1_")]
priority = ["cluster_l1", "cluster_l1_from_LAB", "cluster_l1_x", "cluster_l1_y"]
cluster_variants.sort(key=lambda c: priority.index(c)
                      if c in priority else len(priority))

if len(cluster_variants) == 0:
    print("  cluster_l1 отсутствует — добавляем из LAB")
    lab_sub = (LAB[["oktmo_stable", "year", "cluster"]]
               .drop_duplicates(["oktmo_stable", "year"])
               .rename(columns={"cluster": "cluster_l1"}))
    before = len(dz)
    dz = dz.merge(lab_sub, on=["oktmo_stable", "year"], how="left")
    assert len(dz) == before, "merge cluster_l1 размножил строки dz"
    assert dz["cluster_l1"].notna().all(), "слияние LAB в dz не удалось"
    dz["cluster_l1"] = dz["cluster_l1"].astype(int)
elif len(cluster_variants) > 1:
    print(f"  очистка: избыточные колонки кластера: {cluster_variants}")
    keep = cluster_variants[0]
    if keep != "cluster_l1":
        dz = dz.rename(columns={keep: "cluster_l1"})
    to_drop = [c for c in cluster_variants
               if c != keep and c != "cluster_l1"]
    if to_drop:
        dz = dz.drop(columns=to_drop)
    print(f"    оставлена колонка cluster_l1 (источник: {keep})")

# (4) Финальная защита от любых дубликатов колонок.
if dz.columns.duplicated().any():
    dups = dz.columns[dz.columns.duplicated()].tolist()
    print(f"  очистка: дубликаты в dz.columns: {dups} — оставляем первые")
    dz = dz.loc[:, ~dz.columns.duplicated(keep="first")].copy()

# (5) cluster_l1 должен быть один, целочисленный, без NaN.
assert dz.columns.is_unique, "в dz остались дубликаты колонок"
assert "cluster_l1" in dz.columns, "cluster_l1 отсутствует после очистки"
assert dz["cluster_l1"].notna().all(), "в cluster_l1 есть NaN"
dz["cluster_l1"] = dz["cluster_l1"].astype(int)

# (6) Проверка соответствия диапазона меток ожидаемому K.
uniq_vals = sorted(dz["cluster_l1"].unique().tolist())
assert uniq_vals == list(range(K)), \
    f"cluster_l1 содержит {uniq_vals}, ожидалось {list(range(K))}"

print(f"dz готов: {dz.shape}, cluster_l1 ∈ {uniq_vals}, "
      f"NaN: {int(dz['cluster_l1'].isna().sum())}")
print(f"  дубликатов колонок: {int(dz.columns.duplicated().sum())}")
print(f"  колонок с префиксом kan_l2_k: "
      f"{len([c for c in dz.columns if c.startswith('kan_l2_k')])}")

# ---------------------------------------------------------------------------
# 6a.1. Обучение L2-KAN для каждого L1-кластера
# ---------------------------------------------------------------------------
print(f"\nобучение L2-KAN для K={K} L1-кластеров")
print(f"  вход L2: ZCOLS ({len(ZCOLS)} сырых z-признаков)\n")

L2_INFO = {}
for k in range(K):
    cfg_k = L2_CFG.get(k, L2_CFG[list(L2_CFG.keys())[-1]])

    # --- Явная строковая индексация колонки: dz["cluster_l1"] ------------
    # dz.cluster_l1 (атрибутный доступ) опасен при дубликатах колонок.
    # Используем [], он либо вернёт Series, либо KeyError — но не DataFrame.
    mask_k_series = (dz["cluster_l1"] == k)
    sub_k = dz.loc[mask_k_series].copy().reset_index(drop=True)
    n_k = len(sub_k)
    print(f"=== L2: кластер L1 = {k}  ({n_k} наблюдений, "
          f"{sub_k['oktmo_stable'].nunique()} МО) ===")
    print(f"  архитектура: {len(ZCOLS)} → {cfg_k['d_hidden']} → {cfg_k['d_out']}  "
          f"(grid={cfg_k['grid']}, epochs={cfg_k['epochs']})")

    # --- Мало наблюдений → PCA-fallback ----------------------------------
    if n_k < 100:
        print(f"  ⚠️  мало наблюдений ({n_k} < 100) — fallback на PCA")
        pca_k = PCA(n_components=min(cfg_k["d_out"], len(ZCOLS)),
                    random_state=RNG).fit(sub_k[ZCOLS].values)
        Z_k = pca_k.transform(sub_k[ZCOLS].values).astype(np.float32)
        r2_k = float(1 - pca_k.inverse_transform(Z_k).var() /
                     (sub_k[ZCOLS].values.var() + 1e-12))
        sil_k = sil_of(Z_k, k=2) if n_k > 20 else float("nan")
        cos_k = _pairwise_cos(Z_k, n=min(200, n_k))
        L2_INFO[k] = {
            "cols":      [f"kan_l2_k{k}_p{i}" for i in range(Z_k.shape[1])],
            "r2":        r2_k,
            "sil":       sil_k,
            "cos_mean":  float(cos_k.mean()),
            "n_samples": n_k,
            "arch":      f"PCA{cfg_k['d_out']}",
            "fallback":  True,
            "Z":         Z_k,
        }
    else:
        X_k = sub_k[ZCOLS].values
        assert np.isfinite(X_k).all(), f"кластер {k}: NaN/inf во входе L2"
        Z_k, r2_k, loss_k, dt_k = train_kan_ae(
            X_k,
            d_hidden=cfg_k["d_hidden"], d_out=cfg_k["d_out"],
            grid=cfg_k["grid"], k=3,
            epochs=cfg_k["epochs"], lr=cfg_k["lr"], weight_decay=1e-5,
            batch=cfg_k["batch"], seed=RNG + k,
            verbose=True, t_budget_sec=cfg_k["t_budget"],
            early_stop_patience=20, early_stop_tol=1e-5,
        )
        sil_k = sil_of(Z_k, k=2)
        cos_k = _pairwise_cos(Z_k, n=min(500, n_k))
        print(f"  KAN-L2[{k}]: R²={r2_k:.4f}  sil(k=2)={sil_k:.4f}  "
              f"cos={cos_k.mean():.4f}  dead={(Z_k.std(0) < 1e-4).sum()}  "
              f"t={dt_k:.1f}s")
        L2_INFO[k] = {
            "cols":      [f"kan_l2_k{k}_e{i}" for i in range(Z_k.shape[1])],
            "r2":        r2_k,
            "sil":       sil_k,
            "cos_mean":  float(cos_k.mean()),
            "n_samples": n_k,
            "arch":      f"KAN({len(ZCOLS)}→{cfg_k['d_hidden']}→{cfg_k['d_out']})",
            "fallback":  False,
            "Z":         Z_k,
            "loss_hist": loss_k,
        }

    # --- Проверки качества L2 --------------------------------------------
    if not L2_INFO[k]["fallback"]:
        assert L2_INFO[k]["r2"] >= L2_MIN_R2, (
            f"L2[{k}]: R² слишком низкий ({L2_INFO[k]['r2']:.4f} < {L2_MIN_R2})"
        )
        assert abs(L2_INFO[k]["cos_mean"]) < L2_MAX_COS, (
            f"L2[{k}]: whitening не сработал (cos={L2_INFO[k]['cos_mean']:.4f})"
        )
        assert not np.isnan(L2_INFO[k]["sil"]) and L2_INFO[k]["sil"] >= L2_MIN_SIL, (
            f"L2[{k}]: silhouette слишком низкий ({L2_INFO[k]['sil']:.4f})"
        )

    # --- Добавляем эмбеддинги в dz ---------------------------------------
    # sub_k — c reset_index(drop=True); dz — с уникальным индексом.
    # Присваиваем через bool-маску, а не через список индексов.
    cols_k = L2_INFO[k]["cols"]
    dz = dz.drop(columns=[c for c in cols_k if c in dz.columns], errors="ignore")

    mask_k = (dz["cluster_l1"] == k).values
    assert int(mask_k.sum()) == len(sub_k), (
        f"рассинхронизация: mask_k.sum()={mask_k.sum()}, len(sub_k)={len(sub_k)}"
    )

    for j, c in enumerate(cols_k):
        # Инициализируем колонку NaN, потом заполняем ТОЛЬКО строки класса k.
        dz[c] = np.nan
        dz.loc[mask_k, c] = L2_INFO[k]["Z"][:, j]

    print(f"  → добавлено в dz: {cols_k}  "
          f"(заполнено {int(dz[cols_k[0]].notna().sum())} ячеек)\n")


# ---------------------------------------------------------------------------
# 6a.2. Диагностика: сводка L2-энкодеров
# ---------------------------------------------------------------------------
print("=" * 70)
print("СВОДКА L2-ЭНКОДЕРОВ")
print("=" * 70)
rows = []
for k, info in L2_INFO.items():
    rows.append({
        "cluster_L1": k,
        "arch":       info["arch"],
        "n_samples":  info["n_samples"],
        "R²":         round(info["r2"], 4),
        "silhouette": round(info["sil"], 4) if not np.isnan(info["sil"]) else None,
        "cos_mean":   round(info["cos_mean"], 4),
        "d_out":      len(info["cols"]),
        "fallback":   info["fallback"],
    })
L2_SUMMARY = pd.DataFrame(rows)
print(L2_SUMMARY.to_string(index=False))
L2_SUMMARY.to_parquet(PROC / "l2_encoder_summary.parquet", index=False)


# ---------------------------------------------------------------------------
# 6a.3. Сохранение чекпойнта L2-энкодеров
# ---------------------------------------------------------------------------
torch.save({
    "l1_labels":         "cluster_l1",
    "l1_K":              int(K),
    "l2_cfg_per_class":  {k: {kk: vv for kk, vv in v.items()
                              if kk not in ("Z", "loss_hist")}
                          for k, v in L2_INFO.items()},
    "l2_cols_per_class": {k: v["cols"] for k, v in L2_INFO.items()},
    "l2_R2_per_class":   {k: v["r2"]   for k, v in L2_INFO.items()},
    "l2_sil_per_class":  {k: v["sil"]  for k, v in L2_INFO.items()},
    "zcols_input":       list(ZCOLS),
}, PROC / "kan_encoders_l2.pt")
print(f"\n✅ kan_encoders_l2.pt")


# ---------------------------------------------------------------------------
# 6a.4. Дашборд L2
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(1, K, figsize=(6 * K, 4.5))
if K == 1:
    ax = [ax]
for k in range(K):
    info = L2_INFO[k]
    if info["Z"].shape[1] >= 2:
        pca_show = PCA(n_components=2, random_state=RNG).fit(info["Z"])
        Z2 = pca_show.transform(info["Z"])
        var_exp = pca_show.explained_variance_ratio_.sum()
    else:
        Z2 = np.column_stack([info["Z"][:, 0], np.zeros(len(info["Z"]))])
        var_exp = 1.0
    ax[k].scatter(Z2[:, 0], Z2[:, 1], s=8, alpha=0.4,
                  color=plt.get_cmap("tab10", K)(k))
    ax[k].set_xlabel("PC1"); ax[k].set_ylabel("PC2")
    ax[k].set_title(
        f"L2[{k}] {info['arch']}\n"
        f"R²={info['r2']:.3f}  sil={info['sil']:.3f}  n={info['n_samples']}  "
        f"var(2D)={var_exp:.2f}"
    )
    ax[k].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(VIZ / "03_l2_kan_encoders.png", dpi=110)
plt.close()
print(f"✅ viz/03_l2_kan_encoders.png")


# ---------------------------------------------------------------------------
# 6a.5. Обновляем dz на диске + финальные проверки
# ---------------------------------------------------------------------------
dz.to_parquet(PROC / "panel_annual_features_kan_l2.parquet", index=False)
print(f"✅ panel_annual_features_kan_l2.parquet: dz.shape={dz.shape}")

print("\nзаполненность L2-колонок по классам L1:")
for k in range(K):
    c = L2_INFO[k]["cols"][0]
    in_class  = int(dz.loc[dz["cluster_l1"] == k, c].notna().sum())
    out_class = int(dz.loc[dz["cluster_l1"] != k, c].notna().sum())
    print(f"  L2[{k}] col={c}: внутри={in_class}  вне={out_class}  "
          f"(вне должно быть 0)")

tick(f"L2-энкодеры готовы: {len(L2_INFO)} классов, "
     f"R²={[round(v['r2'], 3) for v in L2_INFO.values()]}")

[1171.9s] обучение L2-KAN для каждого L1-кластера
  cluster_l1 отсутствует — добавляем из LAB
dz готов: (12132, 46), cluster_l1 ∈ [0, 1], NaN: 0
  дубликатов колонок: 0
  колонок с префиксом kan_l2_k: 0

обучение L2-KAN для K=2 L1-кластеров
  вход L2: ZCOLS (10 сырых z-признаков)

=== L2: кластер L1 = 0  (339 наблюдений, 59 МО) ===
  архитектура: 10 → 16 → 6  (grid=8, epochs=100)
    KAN epoch   1/100  loss=4.185249  elapsed=0.1s
    KAN epoch  10/100  loss=0.225563  elapsed=0.6s
    KAN epoch  20/100  loss=0.110951  elapsed=1.2s
    KAN epoch  30/100  loss=0.086661  elapsed=1.7s
    KAN epoch  40/100  loss=0.068425  elapsed=2.3s
    KAN epoch  50/100  loss=0.058862  elapsed=2.9s
    KAN epoch  60/100  loss=0.053466  elapsed=3.6s
    KAN epoch  70/100  loss=0.049176  elapsed=4.1s
    KAN epoch  80/100  loss=0.047969  elapsed=4.7s
    KAN epoch  90/100  loss=0.048715  elapsed=5.3s
    KAN epoch 100/100  loss=0.051730  elapsed=5.9s
  KAN-L2[0]: R²=0.9059  sil(k=2)=0.2605  cos=0.0059  dea

In [10]:
# ============================================================================
# ============ ЯЧЕЙКА 6b ============
# ============================================================================
# ============================================================================
# ЯЧЕЙКА 6b. L2: ВЫБОР subK ПО УСТОЙЧИВОСТИ ВНУТРИ КАЖДОГО L1-КЛАСТЕРА
# ============================================================================
# Для каждого L1-кластера k:
#   • subK ∈ {2, 3, 4}
#   • для каждого subK:
#       – silhouette (на полном разбиении класса)
#       – bootstrap-ARI (устойчивость к ресемплингу)
#       – jackknife-ARI (leave-one-region-out ИЛИ leave-10%-MO-out)
#   • решение:
#       – subK принимается, если boot ≥ 0.75 И sil ≥ 0.15 И jack ≥ 0.85
#       – среди прошедших выбирается subK с максимальным bootstrap-ARI
#       – если ни один не прошёл → subK = 1 (класс не дробится)
#
# Вход:
#   dz с колонками kan_l2_k{k}_e* из ячейки 6a
# Выход:
#   • L2_SUBK[k] — финальный subK для каждого родителя
#   • dz с колонками cluster_l2, cluster_l2_id, sub_k
#   • l2_subk_candidates.parquet — все кандидаты с метриками
#   • l2_subk_summary.parquet    — сводка решений
#   • viz/04_l2_subk_selection.png — графики устойчивости + 2D-проекция
# ============================================================================
tick("выбор subK по устойчивости (L2)")

# --- Проверки доступности из предыдущих ячеек ----------------------------
for _var in ("dz", "K", "L2_INFO", "PROC", "VIZ",
             "boot_stability", "RNG"):
    assert _var in globals(), (
        f"переменная {_var} не найдена — выполните ячейки 0–6a перед 6b"
    )

# --- Параметры ------------------------------------------------------------
SUBK_RANGE             = [2, 3, 4]
BOOT_MIN               = 0.75
SIL_MIN                = 0.15
JACK_MIN               = 0.85
BOOT_ITERS             = 40
JACK_MIN_REGIONS       = 5      # минимум регионов для leave-one-region-out
JACK_MIN_MO_PER_REGION = 3      # регион учитывается, если в нём ≥ 3 МО
JACK_FALLBACK_FRAC     = 0.10   # fallback: leave-10%-MO-out
MIN_SUB_SHARE          = 0.01   # минимальная доля МО в подкластере

print(f"параметры L2: subK ∈ {SUBK_RANGE}, boot≥{BOOT_MIN}, "
      f"sil≥{SIL_MIN}, jack≥{JACK_MIN}")


# ---------------------------------------------------------------------------
# 6b.1. Jackknife-утилита (адаптивная к размеру класса)
# ---------------------------------------------------------------------------
def jackknife_stability(X, regions, k, seed=RNG):
    """Jackknife с автоматическим fallback.

    Попытка 1: leave-one-region-out по регионам с ≥ JACK_MIN_MO_PER_REGION МО.
    Если таких регионов < JACK_MIN_REGIONS → fallback на leave-10%-MO-out.

    Возвращает dict: {method, ari_mean, ari_min, n_units}.
    """
    n = len(X)

    # --- Попытка 1: leave-one-region-out ---------------------------------
    reg_sizes = pd.Series(regions).value_counts()
    usable = reg_sizes[reg_sizes >= JACK_MIN_MO_PER_REGION].index.tolist()

    if len(usable) >= JACK_MIN_REGIONS:
        lab_ref = KMeans(n_clusters=k, n_init=10,
                         random_state=seed).fit_predict(X)
        aris = []
        for r in usable:
            mask = regions != r
            if mask.sum() < k * 3:
                continue
            try:
                lab_r = KMeans(n_clusters=k, n_init=10,
                               random_state=seed).fit_predict(X[mask])
                aris.append(float(adjusted_rand_score(lab_ref[mask], lab_r)))
            except Exception:
                continue
        if aris:
            return {
                "method":   "leave-one-region-out",
                "ari_mean": float(np.mean(aris)),
                "ari_min":  float(np.min(aris)),
                "n_units":  int(len(aris)),
            }

    # --- Попытка 2: leave-10%-MO-out -------------------------------------
    rng = np.random.RandomState(seed)
    lab_ref = KMeans(n_clusters=k, n_init=10,
                     random_state=seed).fit_predict(X)
    fold_size = max(k * 3, int(n * JACK_FALLBACK_FRAC))
    n_folds = max(5, n // fold_size)
    aris = []
    for f in range(n_folds):
        idx_out = rng.choice(n, fold_size, replace=False)
        mask = np.ones(n, bool); mask[idx_out] = False
        if mask.sum() < k * 3:
            continue
        try:
            lab_r = KMeans(n_clusters=k, n_init=10,
                           random_state=seed + f).fit_predict(X[mask])
            aris.append(float(adjusted_rand_score(lab_ref[mask], lab_r)))
        except Exception:
            continue
    return {
        "method":   f"leave-{int(JACK_FALLBACK_FRAC*100)}%-MO-out",
        "ari_mean": float(np.mean(aris)) if aris else float("nan"),
        "ari_min":  float(np.min(aris)) if aris else float("nan"),
        "n_units":  int(len(aris)),
    }


# ---------------------------------------------------------------------------
# 6b.2. Основной цикл: выбор subK для каждого L1-кластера
# ---------------------------------------------------------------------------
L2_SUBK     = {}      # k → subK (1 если не дробится)
L2_DECISION = {}      # k → детальный лог решения
L2_LABELS   = {}      # k → массив меток (длина n_k)

for k in range(K):
    print(f"\n{'='*70}")
    print(f"L1-кластер {k}: {L2_INFO[k]['arch']}, n={L2_INFO[k]['n_samples']}")
    print(f"{'='*70}")

    cols_k = L2_INFO[k]["cols"]
    mask_k = (dz["cluster_l1"] == k).values
    sub_k  = dz.loc[mask_k].reset_index(drop=True)
    X_k    = sub_k[cols_k].values.astype(np.float32)
    reg_k  = sub_k["region_name"].fillna("n/a").astype(str).values
    n_k    = len(X_k)

    assert np.isfinite(X_k).all(), f"класс {k}: NaN/inf в L2-эмбеддингах"
    n_reg  = int(pd.Series(reg_k).nunique())
    print(f"  n={n_k}, регионов={n_reg}")

    # --- Перебор subK ---------------------------------------------------
    candidates = []
    for subK in SUBK_RANGE:
        if subK >= n_k:
            continue
        try:
            lab_full = KMeans(n_clusters=subK, n_init=10, random_state=RNG
                              ).fit_predict(X_k)
            sizes = np.bincount(lab_full, minlength=subK)
            min_allowed = max(3, int(MIN_SUB_SHARE * n_k))
            if (sizes < min_allowed).any():
                print(f"  subK={subK}: ❌ min_size={sizes.min()} < {min_allowed}")
                candidates.append({
                    "subK":             subK,
                    "silhouette":       float("nan"),
                    "bootstrap_ari":    float("nan"),
                    "jackknife_ari":    float("nan"),
                    "jackknife_min":    float("nan"),
                    "jackknife_method": "—",
                    "min_size":         int(sizes.min()),
                    "passed":           False,
                    "note":             f"min_size={sizes.min()} < {min_allowed}",
                })
                continue

            sil_k  = float(silhouette_score(X_k, lab_full))
            boot_k = float(boot_stability(X_k, subK, iters=BOOT_ITERS,
                                           seed=RNG + 100 * k + subK))
            jack_k = jackknife_stability(X_k, reg_k, subK, seed=RNG + k)

            passed = (boot_k >= BOOT_MIN
                      and sil_k >= SIL_MIN
                      and np.isfinite(jack_k["ari_mean"])
                      and jack_k["ari_mean"] >= JACK_MIN)

            candidates.append({
                "subK":             subK,
                "silhouette":       sil_k,
                "bootstrap_ari":    boot_k,
                "jackknife_ari":    jack_k["ari_mean"],
                "jackknife_min":    jack_k["ari_min"],
                "jackknife_method": jack_k["method"],
                "min_size":         int(sizes.min()),
                "passed":           passed,
                "note":             "",
            })
            mark = "✅ passed" if passed else "❌ failed"
            print(f"  subK={subK}: sil={sil_k:.4f}  boot={boot_k:.4f}  "
                  f"jack={jack_k['ari_mean']:.4f} ({jack_k['method']})  "
                  f"min_size={sizes.min()}  {mark}")
        except Exception as e:
            print(f"  subK={subK}: ⚠️  ошибка {type(e).__name__}: {str(e)[:60]}")
            candidates.append({
                "subK":             subK,
                "silhouette":       float("nan"),
                "bootstrap_ari":    float("nan"),
                "jackknife_ari":    float("nan"),
                "jackknife_min":    float("nan"),
                "jackknife_method": "error",
                "min_size":         0,
                "passed":           False,
                "note":             str(e)[:60],
            })

    # --- Решение --------------------------------------------------------
    passed_cands = [c for c in candidates if c["passed"]]
    if passed_cands:
        best = max(passed_cands, key=lambda c: c["bootstrap_ari"])
        subK_final = int(best["subK"])
        reason = (f"принят subK={subK_final} "
                  f"(boot={best['bootstrap_ari']:.3f}, "
                  f"sil={best['silhouette']:.3f}, "
                  f"jack={best['jackknife_ari']:.3f})")
    else:
        subK_final = 1
        reason = "ни один subK не прошёл пороги → класс не дробится"

    print(f"\n  >>> РЕШЕНИЕ: subK={subK_final}  ({reason})")

    # --- Финальные метки для класса k -----------------------------------
    if subK_final >= 2:
        lab_k = KMeans(n_clusters=subK_final, n_init=20, random_state=RNG
                       ).fit_predict(X_k)
        # стабилизация нумерации: 0 = самый крупный подкластер
        order = np.argsort(-np.bincount(lab_k, minlength=subK_final))
        remap = {int(old): int(new) for new, old in enumerate(order)}
        lab_k = np.array([remap[int(x)] for x in lab_k], dtype=int)
    else:
        lab_k = np.zeros(n_k, dtype=int)

    L2_SUBK[k]     = subK_final
    L2_LABELS[k]   = lab_k
    L2_DECISION[k] = {
        "parent_k":   k,
        "n_samples":  n_k,
        "n_regions":  n_reg,
        "subK":       subK_final,
        "reason":     reason,
        "candidates": candidates,
    }

    # Сводка по размерам подкластеров
    if subK_final >= 2:
        sizes_fin = np.bincount(lab_k, minlength=subK_final)
        print(f"  размеры подкластеров: {sizes_fin.tolist()}")


# ---------------------------------------------------------------------------
# 6b.3. Сводная таблица решений
# ---------------------------------------------------------------------------
print(f"\n{'='*70}")
print("СВОДКА ВЫБОРА subK (L2)")
print(f"{'='*70}")
rows = []
for k in range(K):
    d = L2_DECISION[k]
    rows.append({
        "parent_k":   k,
        "subK_final": d["subK"],
        "n_samples":  d["n_samples"],
        "n_regions":  d["n_regions"],
        "reason":     d["reason"][:80],
    })
SUBK_SUMMARY = pd.DataFrame(rows)
print(SUBK_SUMMARY.to_string(index=False))
SUBK_SUMMARY.to_parquet(PROC / "l2_subk_summary.parquet", index=False)

# --- Полная таблица кандидатов ------------------------------------------
all_cands = []
for k in range(K):
    for c in L2_DECISION[k]["candidates"]:
        all_cands.append({"parent_k": k, **c})
CAND_DF = pd.DataFrame(all_cands)
CAND_DF.to_parquet(PROC / "l2_subk_candidates.parquet", index=False)
print(f"\n✅ l2_subk_candidates.parquet: {CAND_DF.shape}")
print(f"✅ l2_subk_summary.parquet:    {SUBK_SUMMARY.shape}")

# --- Печать полной таблицы кандидатов -----------------------------------
print(f"\n{'='*70}")
print("ПОЛНАЯ ТАБЛИЦА КАНДИДАТОВ")
print(f"{'='*70}")
print(CAND_DF.round(4).to_string(index=False))


# ---------------------------------------------------------------------------
# 6b.4. Финальные L2-метки в dz
# ---------------------------------------------------------------------------
# Формат: cluster_l2 = "K{parent}.S{sub}", cluster_l2_id = parent*10 + sub
L2_META_COLS = ["cluster_l2", "cluster_l2_id", "sub_k"]
dz = dz.drop(columns=[c for c in L2_META_COLS if c in dz.columns],
             errors="ignore")
dz["cluster_l2"]    = None
dz["cluster_l2_id"] = np.nan
dz["sub_k"]         = np.nan

for k in range(K):
    mask_k = (dz["cluster_l1"] == k).values
    if not mask_k.any():
        continue
    subK  = L2_SUBK[k]
    lab_k = L2_LABELS[k]
    if subK == 1:
        dz.loc[mask_k, "cluster_l2"]    = f"K{k}.S0"
        dz.loc[mask_k, "cluster_l2_id"] = int(k * 10 + 0)
        dz.loc[mask_k, "sub_k"]         = 0
    else:
        labs_str = np.array([f"K{k}.S{int(s)}" for s in lab_k], dtype=object)
        dz.loc[mask_k, "cluster_l2"]    = labs_str
        dz.loc[mask_k, "cluster_l2_id"] = (k * 10 + lab_k).astype(int)
        dz.loc[mask_k, "sub_k"]         = lab_k.astype(int)

# --- Проверки -----------------------------------------------------------
assert dz.loc[dz["cluster_l1"].notna(), "cluster_l2"].notna().all(), \
    "в cluster_l2 есть NaN"
dz["cluster_l2_id"] = dz["cluster_l2_id"].astype(int)
dz["sub_k"]         = dz["sub_k"].astype(int)

n_unique_l2 = int(dz["cluster_l2"].nunique())
print(f"\n{'='*70}")
print(f"ИТОГ: {n_unique_l2} уникальных L2-подкластеров "
      f"({K} родителей × subK={L2_SUBK})")
print(f"{'='*70}")
l2_counts = dz.groupby("cluster_l2").size().sort_index()
print(l2_counts.to_string())

dz.to_parquet(PROC / "panel_annual_features_kan_l2_subk.parquet", index=False)
print(f"\n✅ panel_annual_features_kan_l2_subk.parquet: {dz.shape}")


# ---------------------------------------------------------------------------
# 6b.5. Диагностический дашборд
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(2, K, figsize=(6 * K, 8))
if K == 1:
    axes = axes.reshape(-1, 1)

for k in range(K):
    d = L2_DECISION[k]

    # --- Верхний ряд: кривые устойчивости по subK ------------------------
    ax1 = axes[0, k]
    subKs = [c["subK"] for c in d["candidates"]]
    boots = [c["bootstrap_ari"] for c in d["candidates"]]
    sils  = [c["silhouette"]    for c in d["candidates"]]
    jacks = [c["jackknife_ari"] for c in d["candidates"]]

    ax1.plot(subKs, boots, "o-", color="g",       label="bootstrap-ARI")
    ax1.plot(subKs, sils,  "s-", color="C0",      label="silhouette")
    ax1.plot(subKs, jacks, "^-", color="purple",  label="jackknife-ARI")
    ax1.axhline(BOOT_MIN, color="g",      ls="--", lw=1, alpha=0.5)
    ax1.axhline(SIL_MIN,  color="C0",     ls="--", lw=1, alpha=0.5)
    ax1.axhline(JACK_MIN, color="purple", ls="--", lw=1, alpha=0.5)
    ax1.axvline(L2_SUBK[k], color="r", ls=":", lw=2,
                label=f"subK* = {L2_SUBK[k]}")
    ax1.set_title(f"L1-кластер {k}: устойчивость по subK")
    ax1.set_xlabel("subK"); ax1.set_ylim(0, 1.05)
    ax1.legend(fontsize=8, loc="lower right"); ax1.grid(alpha=0.3)

    # --- Нижний ряд: 2D-проекция финальных подкластеров ------------------
    ax2 = axes[1, k]
    if L2_SUBK[k] >= 2:
        cols_k = L2_INFO[k]["cols"]
        mask_k = (dz["cluster_l1"] == k).values
        X_l2   = dz.loc[mask_k, cols_k].values
        pca_show = PCA(n_components=2, random_state=RNG).fit(X_l2)
        Z2 = pca_show.transform(X_l2)
        var_exp = pca_show.explained_variance_ratio_.sum()
        labs = L2_LABELS[k]
        cmap_sub = plt.get_cmap("tab10", L2_SUBK[k])
        for s in range(L2_SUBK[k]):
            m = labs == s
            ax2.scatter(Z2[m, 0], Z2[m, 1], s=8, alpha=0.6,
                        color=cmap_sub(s), label=f"S{s} (n={int(m.sum())})")
        ax2.set_title(f"L1-кластер {k} → {L2_SUBK[k]} подкластеров "
                      f"(var2D={var_exp:.2f})")
    else:
        ax2.text(0.5, 0.5, "класс не дробится\n(subK = 1)",
                 ha="center", va="center", fontsize=14, color="gray")
        ax2.set_title(f"L1-кластер {k}: subK = 1")
    ax2.set_xlabel("PC1"); ax2.set_ylabel("PC2")
    ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(VIZ / "04_l2_subk_selection.png", dpi=110)
plt.close()
print(f"✅ viz/04_l2_subk_selection.png")

tick(f"subK выбран: {L2_SUBK}, всего L2-подкластеров: {n_unique_l2}")

[1272.4s] выбор subK по устойчивости (L2)
параметры L2: subK ∈ [2, 3, 4], boot≥0.75, sil≥0.15, jack≥0.85

L1-кластер 0: KAN(10→16→6), n=339
  n=339, регионов=50
  subK=2: sil=0.2605  boot=0.7656  jack=0.9840 (leave-one-region-out)  min_size=101  ✅ passed
  subK=3: sil=0.2329  boot=0.6147  jack=0.9532 (leave-one-region-out)  min_size=16  ❌ failed
  subK=4: sil=0.2117  boot=0.5095  jack=0.9377 (leave-one-region-out)  min_size=16  ❌ failed

  >>> РЕШЕНИЕ: subK=2  (принят subK=2 (boot=0.766, sil=0.260, jack=0.984))
  размеры подкластеров: [238, 101]

L1-кластер 1: KAN(10→32→8), n=11793
  n=11793, регионов=77
  subK=2: sil=0.2708  boot=0.9211  jack=0.9760 (leave-one-region-out)  min_size=2887  ✅ passed
  subK=3: sil=0.2006  boot=0.9385  jack=0.9833 (leave-one-region-out)  min_size=856  ✅ passed
  subK=4: sil=0.1774  boot=0.6413  jack=0.8899 (leave-one-region-out)  min_size=842  ❌ failed

  >>> РЕШЕНИЕ: subK=3  (принят subK=3 (boot=0.938, sil=0.201, jack=0.983))
  размеры подкластеров: [7301

In [11]:
# ============================================================================
# ============ ЯЧЕЙКА 6c ============
# ============================================================================
# ============================================================================
# ЯЧЕЙКА 6c. СОГЛАСОВАНИЕ L2-МЕТОК МЕЖДУ ГОДАМИ (Hungarian, per parent)
# ============================================================================
# Зачем:
#   Ячейка 6b дала subK_k и метки на ПУЛЕ всех лет — статичные во времени.
#   Для анализа динамики нужны метки отдельно по годам, согласованные так,
#   чтобы «S0 в 2019» и «S0 в 2024» обозначали один и тот же подтип.
#
# Как:
#   Для каждого родителя k:
#     • для каждого года y: KMeans(subK_k) на подвыборке (parent=k, year=y)
#       в L2-пространстве (kan_l2_k*_e*);
#     • Hungarian-согласование по центроидам: cost(i,j) = ||cent_prev_i - cent_curr_j||²;
#     • aligned-метки: sub_k_aligned ∈ [0, subK_k − 1];
#     • ARI(raw) vs ARI(aligned) — диагностика, что согласование сработало.
#
# Выход:
#   • L2_LAB — длинный формат: oktmo_stable, year, parent_k, sub_k, sub_k_aligned
#   • cluster_labels_l2_annual.parquet
#   • kan_centroids_l2_annual.parquet
#   • l2_ari_between_years.parquet + l2_ari_per_parent.parquet
#   • viz/05_l2_alignment.png
# ============================================================================
tick("согласование L2-меток между годами")

# --- Проверки доступности из предыдущих ячеек ----------------------------
for _var in ("dz", "K", "yrs", "L2_SUBK", "L2_INFO", "PROC", "VIZ",
             "RNG", "LAB"):
    assert _var in globals(), (
        f"переменная {_var} не найдена — выполните ячейки 0–6b перед 6c"
    )

print(f"родителей: {K}, subK по родителям: {L2_SUBK}")
print(f"лет: {len(yrs)} ({yrs[0]}–{yrs[-1]})")


# ---------------------------------------------------------------------------
# 6c.1. По годам внутри каждого родителя: KMeans + Hungarian
# ---------------------------------------------------------------------------
L2_LABELS_PER_YEAR = {}   # k → {year → {oktmo_stable → sub_k_aligned}}
L2_CENTROIDS       = {}   # k → {year → (subK_k, d_l2) центроиды}
L2_PERMS           = {}   # k → {year → mapping orig→aligned}
L2_ARI_ROWS        = []   # диагностика ARI между годами внутри родителя

for k in range(K):
    subK = L2_SUBK[k]
    cols_k = L2_INFO[k]["cols"]
    print(f"\n{'='*70}")
    print(f"Родитель K{k}: subK={subK}, пространство {cols_k}")
    print(f"{'='*70}")

    # Пустой dict'а для родителя
    L2_LABELS_PER_YEAR[k] = {}
    L2_CENTROIDS[k]       = {}
    L2_PERMS[k]           = {}

    # ---- Baseline: первый год ---------------------------------------------
    y0 = yrs[0]
    sub_y0 = dz.loc[(dz["cluster_l1"] == k) & (dz["year"] == y0),
                    ["oktmo_stable"] + cols_k].reset_index(drop=True)
    X_y0 = sub_y0[cols_k].values.astype(np.float32)
    assert np.isfinite(X_y0).all(), f"K{k}, {y0}: NaN/inf в L2-эмбеддингах"

    if subK == 1:
        lab_y0 = np.zeros(len(X_y0), dtype=int)
        cent_y0 = X_y0.mean(axis=0, keepdims=True)
    else:
        km = KMeans(n_clusters=subK, n_init=20, random_state=RNG).fit(X_y0)
        lab_y0 = km.labels_.astype(int)
        cent_y0 = km.cluster_centers_

    # Стабилизация нумерации: S0 — самый крупный
    order = np.argsort(-np.bincount(lab_y0, minlength=subK))
    remap = {int(old): int(new) for new, old in enumerate(order)}
    lab_y0 = np.array([remap[int(x)] for x in lab_y0], dtype=int)
    cent_y0 = cent_y0[list(remap.keys())]
    # Упорядочим центроиды в согласованной системе
    cent_y0 = np.vstack([cent_y0[i] for i in range(subK)])

    L2_LABELS_PER_YEAR[k][y0] = dict(zip(sub_y0["oktmo_stable"], lab_y0))
    L2_CENTROIDS[k][y0]       = cent_y0
    L2_PERMS[k][y0]           = {i: i for i in range(subK)}

    print(f"  {y0}: n={len(X_y0)}, sizes="
          f"{np.bincount(lab_y0, minlength=subK).tolist()}  (baseline)")

    # ---- Остальные годы: KMeans + Hungarian -----------------------------
    for y in yrs[1:]:
        sub_y = dz.loc[(dz["cluster_l1"] == k) & (dz["year"] == y),
                       ["oktmo_stable"] + cols_k].reset_index(drop=True)
        X_y = sub_y[cols_k].values.astype(np.float32)
        assert np.isfinite(X_y).all(), f"K{k}, {y}: NaN/inf в L2-эмбеддингах"

        if subK == 1:
            lab_raw = np.zeros(len(X_y), dtype=int)
            cent_y  = X_y.mean(axis=0, keepdims=True)
            mapping = {0: 0}
        else:
            km = KMeans(n_clusters=subK, n_init=20, random_state=RNG).fit(X_y)
            lab_raw = km.labels_.astype(int)
            cent_y  = km.cluster_centers_

            # --- Hungarian: центроиды предыдущего года (в СОГЛАСОВАННОЙ
            # системе) → центроиды текущего года -------------------------
            cent_prev = L2_CENTROIDS[k][yrs[yrs.index(y) - 1]]
            D = np.linalg.norm(cent_prev[:, None, :] - cent_y[None, :, :], axis=2)
            r_idx, c_idx = linear_sum_assignment(D)
            # mapping: original_label_curr → aligned_label
            mapping = {int(c_idx[i]): int(r_idx[i]) for i in range(subK)}

            # Переупорядочим центроиды текущего года в согласованную систему
            cent_y_reord = np.zeros_like(cent_y)
            for orig, aligned in mapping.items():
                cent_y_reord[aligned] = cent_y[orig]
            cent_y = cent_y_reord

        # Применяем перестановку
        lab_aligned = np.array([mapping[int(s)] for s in lab_raw], dtype=int)

        L2_LABELS_PER_YEAR[k][y] = dict(zip(sub_y["oktmo_stable"], lab_aligned))
        L2_CENTROIDS[k][y]       = cent_y
        L2_PERMS[k][y]           = mapping

        # ---- Диагностика ARI(raw) vs ARI(aligned) ------------------------
        prev_y = yrs[yrs.index(y) - 1]
        prev_map = L2_LABELS_PER_YEAR[k][prev_y]
        common = np.intersect1d(np.array(list(prev_map.keys())),
                                sub_y["oktmo_stable"].values)
        if len(common) < 5:
            ari_raw = ari_align = np.nan
        else:
            lab_prev_c = np.array([prev_map[mo] for mo in common], dtype=int)
            lab_curr_raw_c = np.array(
                [lab_raw[sub_y["oktmo_stable"].values.tolist().index(mo)]
                 for mo in common], dtype=int)
            lab_curr_alg_c = np.array(
                [L2_LABELS_PER_YEAR[k][y][mo] for mo in common], dtype=int)
            ari_raw   = float(adjusted_rand_score(lab_prev_c, lab_curr_raw_c))
            ari_align = float(adjusted_rand_score(lab_prev_c, lab_curr_alg_c))

        L2_ARI_ROWS.append({
            "parent_k":   k,
            "year_from":  int(prev_y),
            "year_to":    int(y),
            "ari_raw":    ari_raw,
            "ari_aligned": ari_align,
            "n_common":   int(len(common)),
            "mapping":    str(mapping),
        })

        sizes_y = np.bincount(lab_aligned, minlength=subK).tolist()
        print(f"  {prev_y}->{y}: mapping={mapping}  "
              f"ARI(raw)={ari_raw:.4f}  ARI(aligned)={ari_align:.4f}  "
              f"sizes={sizes_y}")

# --- Сохраним ARI-диагностику ------------------------------------------
L2_ARI_DF = pd.DataFrame(L2_ARI_ROWS)
L2_ARI_DF.to_parquet(PROC / "l2_ari_between_years.parquet", index=False)

# --- Сводка ARI по родителям -------------------------------------------
l2_ari_agg_rows = []
for k in range(K):
    sub = L2_ARI_DF[L2_ARI_DF["parent_k"] == k]
    if len(sub) == 0:
        continue
    l2_ari_agg_rows.append({
        "parent_k":           k,
        "subK":               L2_SUBK[k],
        "ari_aligned_mean":   float(sub["ari_aligned"].mean()),
        "ari_aligned_min":    float(sub["ari_aligned"].min()),
        "ari_aligned_max":    float(sub["ari_aligned"].max()),
        "ari_raw_mean":       float(sub["ari_raw"].mean()),
        "n_transitions":      int(len(sub)),
    })
L2_ARI_AGG = pd.DataFrame(l2_ari_agg_rows)
L2_ARI_AGG.to_parquet(PROC / "l2_ari_per_parent.parquet", index=False)

print(f"\n{'='*70}")
print("ARI МЕЖДУ СОСЕДНИМИ ГОДАМИ (L2, aligned)")
print(f"{'='*70}")
print(L2_ARI_AGG.round(4).to_string(index=False))


# ---------------------------------------------------------------------------
# 6c.2. Сборка L2_LAB (длинный формат с aligned-подкластерами)
# ---------------------------------------------------------------------------
print(f"\n{'='*70}")
print("СБОРКА L2-МЕТОК (длинный формат)")
print(f"{'='*70}")

rows = []
for k in range(K):
    subK = L2_SUBK[k]
    for y in yrs:
        sub_y = dz.loc[(dz["cluster_l1"] == k) & (dz["year"] == y)]
        for mo, sub_k in L2_LABELS_PER_YEAR[k][y].items():
            rows.append({
                "oktmo_stable":       mo,
                "year":               int(y),
                "parent_k":           int(k),
                "sub_k":              int(sub_k),
                "cluster_l2":         f"K{k}.S{int(sub_k)}",
                "cluster_l2_id":      int(k * 10 + int(sub_k)),
                "n_subclusters":      int(subK),
            })

L2_LAB = pd.DataFrame(rows)
L2_LAB["year"] = L2_LAB["year"].astype(int)
L2_LAB["oktmo_stable"] = L2_LAB["oktmo_stable"].astype(str)
L2_LAB = L2_LAB.sort_values(["year", "parent_k", "sub_k", "oktmo_stable"]
                            ).reset_index(drop=True)

assert len(L2_LAB) > 0, "L2_LAB пуст"
assert L2_LAB["cluster_l2"].notna().all(), "NaN в cluster_l2"
print(f"L2_LAB: {L2_LAB.shape}")
print(f"  МО: {L2_LAB.oktmo_stable.nunique()}")
print(f"  лет: {L2_LAB.year.nunique()} ({L2_LAB.year.min()}–{L2_LAB.year.max()})")
print(f"  уникальных cluster_l2: {L2_LAB.cluster_l2.nunique()}")

# --- Число МО по подкластерам по годам ---------------------------------
print(f"\nРАЗМЕРЫ L2-ПОДКЛАСТЕРОВ ПО ГОДАМ")
ct = pd.crosstab([L2_LAB.year], L2_LAB.cluster_l2)
print(ct.to_string())


# ---------------------------------------------------------------------------
# 6c.3. Проверки согласованности
# ---------------------------------------------------------------------------
# (а) каждая МО × год имеет ровно один L2-подкластер
dup = L2_LAB.duplicated(subset=["oktmo_stable", "year"]).sum()
assert dup == 0, f"L2_LAB содержит {dup} дубликатов по (oktmo, year)"

# (б) в каждом году у каждого родителя присутствуют ВСЕ subK подкластеров
for k in range(K):
    subK = L2_SUBK[k]
    for y in yrs:
        subs_present = sorted(
            L2_LAB.loc[(L2_LAB.parent_k == k) & (L2_LAB.year == y),
                       "sub_k"].unique().tolist())
        assert subs_present == list(range(subK)), \
            f"K{k}, {y}: подкластеры {subs_present}, ожидалось {list(range(subK))}"

# (в) средний ARI между соседними годами ≥ 0.5 для каждого родителя
for k in range(K):
    sub = L2_ARI_DF[L2_ARI_DF["parent_k"] == k]
    mean_ari = float(sub["ari_aligned"].mean())
    assert mean_ari >= 0.5, \
        f"K{k}: средний ARI между годами {mean_ari:.4f} < 0.5 — нестабильно"
    print(f"  K{k}: средний ARI(aligned) между годами = {mean_ari:.4f}  ✓")

print("\nвсе проверки согласованности L2 пройдены")


# ---------------------------------------------------------------------------
# 6c.4. Сохранение артефактов
# ---------------------------------------------------------------------------
L2_LAB.to_parquet(PROC / "cluster_labels_l2_annual.parquet", index=False)
print(f"✅ cluster_labels_l2_annual.parquet: {L2_LAB.shape}")

# --- Центроиды L2 по годам и родителям --------------------------------
cent_rows = []
for k in range(K):
    subK = L2_SUBK[k]
    cols_k = L2_INFO[k]["cols"]
    for y in yrs:
        cent = L2_CENTROIDS[k][y]           # (subK, d_l2)
        sizes_y = np.bincount(
            np.array(list(L2_LABELS_PER_YEAR[k][y].values())),
            minlength=subK)
        for s in range(subK):
            row = {
                "parent_k":  int(k),
                "year":      int(y),
                "sub_k":     int(s),
                "cluster_l2": f"K{k}.S{s}",
                "size":      int(sizes_y[s]),
            }
            for j, c in enumerate(cols_k):
                row[c] = float(cent[s, j])
            cent_rows.append(row)
L2_CENT = pd.DataFrame(cent_rows)
L2_CENT.to_parquet(PROC / "kan_centroids_l2_annual.parquet", index=False)
print(f"✅ kan_centroids_l2_annual.parquet: {L2_CENT.shape}")


# ---------------------------------------------------------------------------
# 6c.5. Диагностика: динамика размеров и ARI
# ---------------------------------------------------------------------------
n_plots = K + 1
fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4.5))
# plt.subplots(1, n, ...) возвращает одномерный массив формы (n,),
# поэтому обращаемся по одному индексу: axes[0], axes[1], ...
axes = np.atleast_1d(axes).ravel()
assert axes.shape[0] == n_plots, f"axes.shape={axes.shape}, ожидалось ({n_plots},)"

# --- ARI по родителям --------------------------------------------------
ax_ari = axes[0]
for k in range(K):
    sub = L2_ARI_DF[L2_ARI_DF["parent_k"] == k]
    if len(sub) == 0:
        continue
    x = np.arange(len(sub))
    ax_ari.plot(x, sub["ari_aligned"].values, "o-",
                label=f"K{k} (subK={L2_SUBK[k]})")
ax_ari.axhline(0.5, ls="--", color="r", lw=1, alpha=0.5)
ax_ari.axhline(0.8, ls=":",  color="g", lw=1, alpha=0.5)

# X-ticks: берём переходы от любого родителя (они одинаковые по годам)
ref = L2_ARI_DF[L2_ARI_DF["parent_k"] == 0]
ax_ari.set_xticks(range(len(ref)))
ax_ari.set_xticklabels(
    [f"{int(r.year_from)}→{int(r.year_to)}" for r in ref.itertuples()],
    rotation=45, fontsize=8)
ax_ari.set_ylabel("ARI (aligned)")
ax_ari.set_title("ARI L2 между соседними годами")
ax_ari.set_ylim(0, 1.05)
ax_ari.legend(fontsize=8)
ax_ari.grid(alpha=0.3)

# --- Размеры подкластеров по годам, отдельно по родителям --------------
for k in range(K):
    ax = axes[k + 1]
    subK = L2_SUBK[k]
    cols_k_l2 = [f"K{k}.S{s}" for s in range(subK)]
    ct_k = ct.reindex(columns=cols_k_l2, fill_value=0)

    ax.stackplot(ct_k.index,
                 [ct_k[c].values for c in cols_k_l2],
                 labels=cols_k_l2,
                 alpha=0.85)
    ax.set_xlabel("Год")
    ax.set_ylabel("Число МО")
    ax.set_title(f"K{k} → {subK} подкластеров (L2)")
    ax.legend(loc="center left", bbox_to_anchor=(1.02, .5), fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(VIZ / "05_l2_alignment.png", dpi=110)
plt.close()
print(f"✅ viz/05_l2_alignment.png")


# ---------------------------------------------------------------------------
# 6c.6. Обновление dz: aligned-метки
# ---------------------------------------------------------------------------
L2_ALIGN_COLS = ["cluster_l2_aligned", "sub_k_aligned"]
dz = dz.drop(columns=[c for c in L2_ALIGN_COLS if c in dz.columns],
             errors="ignore")

dz = dz.merge(
    L2_LAB[["oktmo_stable", "year", "sub_k", "cluster_l2"]]
        .rename(columns={"sub_k": "sub_k_aligned",
                         "cluster_l2": "cluster_l2_aligned"}),
    on=["oktmo_stable", "year"],
    how="left",
)

# Для МО, не попавших ни в один L1-кластер (edge case), — NaN. Проверим,
# что таких нет.
assert dz["sub_k_aligned"].notna().all(), \
    "не всем строкам dz назначен L2-подкластер"
dz["sub_k_aligned"] = dz["sub_k_aligned"].astype(int)

dz.to_parquet(PROC / "panel_annual_features_kan_l2_aligned.parquet",
              index=False)
print(f"✅ panel_annual_features_kan_l2_aligned.parquet: {dz.shape}")


# ---------------------------------------------------------------------------
# 6c.7. Итог
# ---------------------------------------------------------------------------
print(f"\n{'='*70}")
print("ИТОГ ЯЧЕЙКИ 6c")
print(f"{'='*70}")
print(f"L2_LAB:          {L2_LAB.shape}  "
      f"({L2_LAB.oktmo_stable.nunique()} МО × {L2_LAB.year.nunique()} лет)")
print(f"Родителей:       {K}")
print(f"subK по родителям: {L2_SUBK}")
total_l2 = sum(L2_SUBK.values())
print(f"Всего подкластеров L2: {total_l2}")
print()
print("ARI(aligned) между соседними годами:")
print(L2_ARI_AGG[["parent_k", "subK", "ari_aligned_mean",
                  "ari_aligned_min", "ari_aligned_max"]].round(4).to_string(index=False))

tick(f"согласование L2 завершено: {total_l2} подкластеров, "
     f"ARI(aligned) min={L2_ARI_AGG['ari_aligned_min'].min():.3f}")

[1321.1s] согласование L2-меток между годами
родителей: 2, subK по родителям: {0: 2, 1: 3}
лет: 6 (2019–2024)

Родитель K0: subK=2, пространство ['kan_l2_k0_e0', 'kan_l2_k0_e1', 'kan_l2_k0_e2', 'kan_l2_k0_e3', 'kan_l2_k0_e4', 'kan_l2_k0_e5']
  2019: n=57, sizes=[43, 14]  (baseline)
  2019->2020: mapping={0: 0, 1: 1}  ARI(raw)=0.5835  ARI(aligned)=0.5835  sizes=[45, 12]
  2020->2021: mapping={0: 0, 1: 1}  ARI(raw)=0.7822  ARI(aligned)=0.7822  sizes=[42, 15]
  2021->2022: mapping={1: 0, 0: 1}  ARI(raw)=0.7888  ARI(aligned)=0.7888  sizes=[38, 18]
  2022->2023: mapping={1: 0, 0: 1}  ARI(raw)=0.3682  ARI(aligned)=0.3682  sizes=[29, 26]
  2023->2024: mapping={0: 0, 1: 1}  ARI(raw)=0.7206  ARI(aligned)=0.7206  sizes=[33, 24]

Родитель K1: subK=3, пространство ['kan_l2_k1_e0', 'kan_l2_k1_e1', 'kan_l2_k1_e2', 'kan_l2_k1_e3', 'kan_l2_k1_e4', 'kan_l2_k1_e5', 'kan_l2_k1_e6', 'kan_l2_k1_e7']
  2019: n=1965, sizes=[1412, 505, 48]  (baseline)
  2019->2020: mapping={1: 0, 2: 1, 0: 2}  ARI(raw)=0.5170 

In [12]:
# ============================================================================
# ============ ЯЧЕЙКА 7 ============
# ============================================================================
# ============================================================================
# ЯЧЕЙКА 7. УСТОЙЧИВОСТЬ ВО ВРЕМЕНИ (L1 + L2)
# ============================================================================
# Часть A. L1: ARI соседних лет, ARI к базовому году, stay rate,
#             число смен кластера, changepoints.
# Часть B. L2: то же самое, но раздельно по каждому родителю
#             (ARI считается только на подвыборке МО этого родителя).
#
# Вход:
#   • LAB — L1-метки по годам (long)
#   • L2_LAB — L2-метки по годам (long)
#   • K, yrs, L2_SUBK, PROC, VIZ, RNG
#
# Выход:
#   • ari_between_years.parquet           — L1 ARI между соседними годами
#   • ari_vs_base.parquet                 — L1 ARI к базовому году
#   • ari_between_years_l2.parquet        — L2 ARI между соседними годами
#   • ari_vs_base_l2.parquet              — L2 ARI к базовому году
#   • l2_stability_per_parent.parquet     — stay rate и доля сменивших по L2
#   • changepoints_l1.parquet             — моменты смены структуры L1
#   • changepoints_l2.parquet             — моменты смены структуры L2
#   • viz/02_ari_stability.png            — L1 ARI
#   • viz/06_ari_stability_l2.png         — L2 ARI по родителям
# ============================================================================
tick("устойчивость во времени (L1 + L2)")

# --- Проверки доступности -------------------------------------------------
for _var in ("LAB", "L2_LAB", "K", "yrs", "L2_SUBK", "PROC", "VIZ"):
    assert _var in globals(), (
        f"переменная {_var} не найдена — выполните ячейки 0–6c перед 7"
    )

base_year = yrs[0]


# ============================================================================
# ЧАСТЬ A. L1: ARI СОСЕДНИХ ЛЕТ, ARI К БАЗОВОМУ ГОДУ, STAY RATE
# ============================================================================
print("=" * 70)
print("ЧАСТЬ A. УСТОЙЧИВОСТЬ L1")
print("=" * 70)

piv_l1 = LAB.pivot_table(index="oktmo_stable", columns="year",
                          values="cluster")

# --- ARI между соседними годами ------------------------------------------
rows = []
for a, b in zip(yrs, yrs[1:]):
    sub = piv_l1[[a, b]].dropna()
    ari = float(adjusted_rand_score(sub[a].values, sub[b].values))
    changed = float((sub[a].values != sub[b].values).mean())
    rows.append({"pair": f"{a}->{b}", "year_from": int(a), "year_to": int(b),
                 "ari": ari, "share_changed": changed,
                 "stay_rate": 1.0 - changed, "n_common": int(len(sub))})
ARI = pd.DataFrame(rows)
ARI.to_parquet(PROC / "ari_between_years.parquet", index=False)

# --- ARI к базовому году -------------------------------------------------
rows = []
for y in yrs[1:]:
    sub = piv_l1[[base_year, y]].dropna()
    ari = float(adjusted_rand_score(sub[base_year].values, sub[y].values))
    rows.append({"pair": f"{base_year}->{y}", "year_from": int(base_year),
                 "year_to": int(y), "ari": ari, "n_common": int(len(sub))})
AB = pd.DataFrame(rows)
AB.to_parquet(PROC / "ari_vs_base.parquet", index=False)

# --- Печать --------------------------------------------------------------
print("\n=== ARI СОСЕДНИХ ЛЕТ (L1) ===")
print(ARI.round(4).to_string(index=False))
print("\n=== ARI К БАЗОВОМУ ГОДУ (L1) ===")
print(AB.round(4).to_string(index=False))

mean_ari_l1 = float(ARI["ari"].mean())
mean_stay_l1 = float(ARI["stay_rate"].mean())
print(f"\nсредний ARI соседних лет (L1):     {mean_ari_l1:.4f}")
print(f"доля сменивших кластер за год (L1): {ARI['share_changed'].mean():.4f}")
print(f"stay rate (L1):                     {mean_stay_l1:.4f}")

# --- Сколько разных кластеров посетила МО за горизонт -------------------
n_switch_l1 = piv_l1.apply(lambda row: int(row.dropna().nunique()), axis=1)
print("\nчисло разных кластеров, посещённых МО за весь горизонт (L1):")
print(n_switch_l1.value_counts().sort_index().to_string())

# --- Changepoints L1: переходы с ARI < 0.8 ---------------------------------
cp_rows_l1 = []
for r in ARI.itertuples():
    if r.ari < 0.8:
        cp_rows_l1.append({
            "year_from": int(r.year_from), "year_to": int(r.year_to),
            "ari": float(r.ari),
            "share_changed": float(r.share_changed),
        })
CHP_L1 = pd.DataFrame(cp_rows_l1) if cp_rows_l1 else pd.DataFrame(
    columns=["year_from", "year_to", "ari", "share_changed"])
CHP_L1.to_parquet(PROC / "changepoints_l1.parquet", index=False)
print(f"\nchangepoints L1 (ARI < 0.8): {len(CHP_L1)}")
if len(CHP_L1):
    print(CHP_L1.round(4).to_string(index=False))

# --- График L1 -----------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(range(len(ARI)), ARI["ari"], color="steelblue")
ax[0].set_xticks(range(len(ARI)))
ax[0].set_xticklabels(ARI["pair"], rotation=45, fontsize=8)
ax[0].axhline(0.5, ls="--", color="r", lw=1, alpha=0.5)
ax[0].axhline(0.8, ls=":",  color="g", lw=1, alpha=0.5)
ax[0].set_ylim(0, 1)
ax[0].set_title("ARI между соседними годами (L1)")
ax[0].set_ylabel("ARI")
ax[0].grid(alpha=0.3)

ax[1].bar(range(len(AB)), AB["ari"], color="darkorange")
ax[1].set_xticks(range(len(AB)))
ax[1].set_xticklabels(AB["pair"], rotation=45, fontsize=8)
ax[1].axhline(0.5, ls="--", color="r", lw=1, alpha=0.5)
ax[1].set_ylim(0, 1)
ax[1].set_title(f"ARI относительно базового года {base_year} (L1)")
ax[1].set_ylabel("ARI")
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(VIZ / "02_ari_stability.png", dpi=110)
plt.close()
print(f"\n✅ viz/02_ari_stability.png")


# ============================================================================
# ЧАСТЬ B. L2: ARI МЕЖДУ ГОДАМИ ВНУТРИ КАЖДОГО РОДИТЕЛЯ
# ============================================================================
print("\n" + "=" * 70)
print("ЧАСТЬ B. УСТОЙЧИВОСТЬ L2 (по родителям)")
print("=" * 70)

# --- Определяем имена колонок в L2_LAB (устойчиво к вариантам) ----------
SUB_COL = next((c for c in ("sub_k_aligned", "sub_k") if c in L2_LAB.columns),
               None)
PARENT_COL = next((c for c in ("parent_k", "cluster_l1_aligned")
                    if c in L2_LAB.columns), None)
assert SUB_COL is not None, \
    f"в L2_LAB нет ни sub_k_aligned, ни sub_k. Колонки: {list(L2_LAB.columns)}"
assert PARENT_COL is not None, \
    f"в L2_LAB нет ни parent_k, ни cluster_l1_aligned. " \
    f"Колонки: {list(L2_LAB.columns)}"
print(f"используем колонки L2_LAB: SUB_COL='{SUB_COL}', "
      f"PARENT_COL='{PARENT_COL}'")

# --- Pivot L2: строки = МО, столбцы = год, значения = sub_k ------------
piv_l2 = L2_LAB.pivot_table(index="oktmo_stable", columns="year",
                             values=SUB_COL)

# --- Pivot L2 по родителю: строки = МО, столбцы = год, значения = parent_k -
parent_by_year = L2_LAB.pivot_table(index="oktmo_stable", columns="year",
                                     values=PARENT_COL)

# --- МО приписывается родителю r, если во всех годах у неё parent_k == r -
def _stable_parent(row):
    vals = row.dropna().unique()
    if len(vals) == 1:
        return int(vals[0])
    return -1  # «мигрирующая» между родителями

stable_parent = parent_by_year.apply(_stable_parent, axis=1)
print(f"МО со стабильным родителем: {int((stable_parent >= 0).sum())} / "
      f"{len(stable_parent)}")
for r in range(K):
    n_r = int((stable_parent == r).sum())
    print(f"  parent {r}: {n_r} МО")

# --- ARI между соседними годами, L2, per parent ------------------------
rows = []
for r in range(K):
    subK = L2_SUBK[r]
    mo_r = stable_parent[stable_parent == r].index
    piv_r = piv_l2.loc[piv_l2.index.intersection(mo_r)]
    for a, b in zip(yrs, yrs[1:]):
        sub = piv_r[[a, b]].dropna()
        if len(sub) < 3:
            continue
        ari = float(adjusted_rand_score(sub[a].values, sub[b].values))
        changed = float((sub[a].values != sub[b].values).mean())
        rows.append({
            "parent_k":      int(r),
            "subK":          int(subK),
            "year_from":     int(a),
            "year_to":       int(b),
            "pair":          f"{a}->{b}",
            "ari":           ari,
            "share_changed": changed,
            "stay_rate":     1.0 - changed,
            "n_common":      int(len(sub)),
        })
L2_ARI_BETWEEN = pd.DataFrame(rows)
L2_ARI_BETWEEN.to_parquet(PROC / "ari_between_years_l2.parquet", index=False)

# --- ARI к базовому году, L2, per parent --------------------------------
rows = []
for r in range(K):
    subK = L2_SUBK[r]
    mo_r = stable_parent[stable_parent == r].index
    piv_r = piv_l2.loc[piv_l2.index.intersection(mo_r)]
    for y in yrs[1:]:
        sub = piv_r[[base_year, y]].dropna()
        if len(sub) < 3:
            continue
        ari = float(adjusted_rand_score(sub[base_year].values, sub[y].values))
        rows.append({
            "parent_k":  int(r),
            "subK":      int(subK),
            "year_from": int(base_year),
            "year_to":   int(y),
            "pair":      f"{base_year}->{y}",
            "ari":       ari,
            "n_common":  int(len(sub)),
        })
L2_ARI_BASE = pd.DataFrame(rows)
L2_ARI_BASE.to_parquet(PROC / "ari_vs_base_l2.parquet", index=False)

# --- Сводка устойчивости L2 по родителям --------------------------------
rows = []
for r in range(K):
    sub_b = L2_ARI_BETWEEN[L2_ARI_BETWEEN["parent_k"] == r]
    sub_a = L2_ARI_BASE[L2_ARI_BASE["parent_k"] == r]
    if len(sub_b) == 0:
        continue
    rows.append({
        "parent_k":           int(r),
        "subK":               int(L2_SUBK[r]),
        "n_mo_stable_parent": int((stable_parent == r).sum()),
        "ari_consec_mean":    float(sub_b["ari"].mean()),
        "ari_consec_min":     float(sub_b["ari"].min()),
        "ari_consec_max":     float(sub_b["ari"].max()),
        "stay_rate_mean":     float(sub_b["stay_rate"].mean()),
        "share_changed_mean": float(sub_b["share_changed"].mean()),
        "ari_vs_base_mean":   float(sub_a["ari"].mean()) if len(sub_a) else float("nan"),
    })
L2_STAB = pd.DataFrame(rows)
L2_STAB.to_parquet(PROC / "l2_stability_per_parent.parquet", index=False)

# --- Печать L2 ----------------------------------------------------------
print("\n=== ARI СОСЕДНИХ ЛЕТ (L2, по родителям) ===")
print(L2_ARI_BETWEEN.round(4).to_string(index=False))
print("\n=== ARI К БАЗОВОМУ ГОДУ (L2, по родителям) ===")
print(L2_ARI_BASE.round(4).to_string(index=False))
print("\n=== СВОДКА УСТОЙЧИВОСТИ L2 ПО РОДИТЕЛЯМ ===")
print(L2_STAB.round(4).to_string(index=False))

# --- Сколько разных подкластеров посетила МО, отдельно по родителям ----
print("\nчисло разных L2-подкластеров, посещённых МО за горизонт:")
for r in range(K):
    mo_r = stable_parent[stable_parent == r].index
    piv_r = piv_l2.loc[piv_l2.index.intersection(mo_r)]
    n_switch = piv_r.apply(lambda row: int(row.dropna().nunique()), axis=1)
    vc = n_switch.value_counts().sort_index()
    print(f"  parent {r} (subK={L2_SUBK[r]}, n={len(piv_r)}):")
    for k_, v_ in vc.items():
        print(f"    {int(k_)} разных: {int(v_)} МО")

# --- Changepoints L2 ----------------------------------------------------
cp_rows_l2 = []
for r in range(K):
    sub = L2_ARI_BETWEEN[L2_ARI_BETWEEN["parent_k"] == r]
    for row in sub.itertuples():
        if row.ari < 0.8:
            cp_rows_l2.append({
                "parent_k":      int(r),
                "year_from":     int(row.year_from),
                "year_to":       int(row.year_to),
                "ari":           float(row.ari),
                "share_changed": float(row.share_changed),
            })
CHP_L2 = pd.DataFrame(cp_rows_l2) if cp_rows_l2 else pd.DataFrame(
    columns=["parent_k", "year_from", "year_to", "ari", "share_changed"])
CHP_L2.to_parquet(PROC / "changepoints_l2.parquet", index=False)
print(f"\nchangepoints L2 (ARI < 0.8): {len(CHP_L2)}")
if len(CHP_L2):
    print(CHP_L2.round(4).to_string(index=False))

# --- График L2: ARI соседних лет и ARI к базовому году -----------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cmap_l1 = plt.get_cmap("tab10", K)
colors = [cmap_l1(r) for r in range(K)]

# (а) ARI между соседними годами
ax = axes[0]
for r in range(K):
    sub = L2_ARI_BETWEEN[L2_ARI_BETWEEN["parent_k"] == r]
    if len(sub) == 0:
        continue
    x = np.arange(len(sub))
    ax.plot(x, sub["ari"].values, "o-",
            color=colors[r], lw=2, ms=7,
            label=f"K{r} (subK={L2_SUBK[r]})")
ax.axhline(0.5, ls="--", color="r", lw=1, alpha=0.5)
ax.axhline(0.8, ls=":",  color="g", lw=1, alpha=0.5)

ref = L2_ARI_BETWEEN[L2_ARI_BETWEEN["parent_k"] == 0]
ax.set_xticks(range(len(ref)))
ax.set_xticklabels([f"{int(r.year_from)}→{int(r.year_to)}"
                     for r in ref.itertuples()],
                    rotation=45, fontsize=8)
ax.set_ylabel("ARI"); ax.set_ylim(0, 1.05)
ax.set_title("ARI между соседними годами (L2, по родителям)")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# (б) ARI к базовому году
ax = axes[1]
for r in range(K):
    sub = L2_ARI_BASE[L2_ARI_BASE["parent_k"] == r]
    if len(sub) == 0:
        continue
    x = np.arange(len(sub))
    ax.plot(x, sub["ari"].values, "s-",
            color=colors[r], lw=2, ms=7,
            label=f"K{r} (subK={L2_SUBK[r]})")
ax.axhline(0.5, ls="--", color="r", lw=1, alpha=0.5)

ref = L2_ARI_BASE[L2_ARI_BASE["parent_k"] == 0]
ax.set_xticks(range(len(ref)))
ax.set_xticklabels([f"{int(r.year_from)}→{int(r.year_to)}"
                     for r in ref.itertuples()],
                    rotation=45, fontsize=8)
ax.set_ylabel("ARI"); ax.set_ylim(0, 1.05)
ax.set_title(f"ARI относительно {base_year} (L2, по родителям)")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(VIZ / "06_ari_stability_l2.png", dpi=110)
plt.close()
print(f"\n✅ viz/06_ari_stability_l2.png")


# ============================================================================
# ИТОГ
# ============================================================================
print("\n" + "=" * 70)
print("ИТОГ ЯЧЕЙКИ 7")
print("=" * 70)
print(f"L1:  ARI(consec) = {mean_ari_l1:.4f}  "
      f"stay_rate = {mean_stay_l1:.4f}  "
      f"changepoints = {len(CHP_L1)}")
print()
print("L2 (по родителям):")
for r in range(K):
    row = L2_STAB[L2_STAB["parent_k"] == r]
    if len(row) == 0:
        continue
    row = row.iloc[0]
    print(f"  K{r} (subK={int(row.subK)}, "
          f"n={int(row.n_mo_stable_parent)} МО): "
          f"ARI(consec)={row.ari_consec_mean:.4f}  "
          f"stay_rate={row.stay_rate_mean:.4f}  "
          f"ARI(vs base)={row.ari_vs_base_mean:.4f}")
print()
print("Артефакты:")
for f in ["ari_between_years.parquet", "ari_vs_base.parquet",
          "ari_between_years_l2.parquet", "ari_vs_base_l2.parquet",
          "l2_stability_per_parent.parquet",
          "changepoints_l1.parquet", "changepoints_l2.parquet"]:
    p = PROC / f
    if p.exists():
        print(f"  ✅ {f}")
print(f"  ✅ viz/02_ari_stability.png")
print(f"  ✅ viz/06_ari_stability_l2.png")

tick(f"устойчивость посчитана: L1 ARI={mean_ari_l1:.4f}, "
     f"L2 ARI по родителям: "
     f"{[round(r, 3) for r in L2_STAB['ari_consec_mean'].tolist()]}")

[1323.1s] устойчивость во времени (L1 + L2)
ЧАСТЬ A. УСТОЙЧИВОСТЬ L1

=== ARI СОСЕДНИХ ЛЕТ (L1) ===
      pair  year_from  year_to    ari  share_changed  stay_rate  n_common
2019->2020       2019     2020 1.0000         0.0000     1.0000      2022
2020->2021       2020     2021 1.0000         0.0000     1.0000      2022
2021->2022       2021     2022 0.9904         0.0005     0.9995      2022
2022->2023       2022     2023 0.9317         0.0035     0.9965      2022
2023->2024       2023     2024 0.9613         0.0020     0.9980      2022

=== ARI К БАЗОВОМУ ГОДУ (L1) ===
      pair  year_from  year_to    ari  n_common
2019->2020       2019     2020 1.0000      2022
2019->2021       2019     2021 1.0000      2022
2019->2022       2019     2022 0.9904      2022
2019->2023       2019     2023 0.9419      2022
2019->2024       2019     2024 0.9809      2022

средний ARI соседних лет (L1):     0.9767
доля сменивших кластер за год (L1): 0.0012
stay rate (L1):                     0.9988

числ

In [13]:
# ============================================================================
# ============ ЯЧЕЙКА 8 ============
# ============================================================================
# ============================================================================
# ЯЧЕЙКА 8. ПРОФИЛИ, ТЕСТЫ, МЕТРИКИ (L1 + L2 ВНУТРИ РОДИТЕЛЕЙ)
# ============================================================================
# Часть A. L1:
#   • профили кластеров (raw, median, z)
#   • Kruskal-Wallis по FEATS
#   • pairwise Mann-Whitney + FDR (Benjamini-Hochberg)
#   • метрики качества по годам (silhouette, CH, DB)
#
# Часть B. L2 (внутри каждого родителя):
#   • профили подкластеров (raw, z) — отдельно по каждому родителю k
#   • KW среди subK_k подкластеров ВНУТРИ родителя
#   • pairwise Mann-Whitney среди подкластеров ВНУТРИ родителя
#   • FDR корректировка отдельно для каждого родителя
#   • метрики качества по годам (внутри родителя)
#
# Вход:
#   • LAB       — L1-метки (long), содержит FEATS и ZCOLS
#   • L2_LAB    — L2-метки (long)
#   • dz        — с FEATS и ZCOLS (для подвыборок L2)
#   • FEATS, ZCOLS, EXTRA_PROFILE, K, L2_SUBK, yrs
#
# Выход:
#   • cluster_profiles_raw.parquet / _median.parquet / _z.parquet
#   • cluster_profiles_l2_raw.parquet / _z.parquet
#   • tests_kruskal.parquet
#   • tests_kruskal_l2.parquet
#   • tests_pairwise.parquet
#   • tests_pairwise_l2.parquet
#   • quality_metrics.parquet
#   • quality_metrics_l2.parquet
#   • viz/04_profiles_heatmap.png
#   • viz/07_profiles_heatmap_l2.png
# ============================================================================
tick("профили и тесты (L1 + L2 внутри родителей)")

# --- Проверки доступности --------------------------------------------------
for _var in ("LAB", "L2_LAB", "dz", "FEATS", "ZCOLS", "EXTRA_PROFILE",
             "K", "L2_SUBK", "yrs", "PROC", "VIZ"):
    assert _var in globals(), (
        f"переменная {_var} не найдена — выполните ячейки 0–7 перед 8"
    )

# --- Определяем имя колонки с подкластером L2 (устойчиво к вариантам) ----
SUB_COL    = next((c for c in ("sub_k_aligned", "sub_k")
                   if c in L2_LAB.columns), None)
PARENT_COL = next((c for c in ("parent_k", "cluster_l1_aligned")
                    if c in L2_LAB.columns), None)
assert SUB_COL is not None, f"нет sub_k в L2_LAB: {list(L2_LAB.columns)}"
assert PARENT_COL is not None, f"нет parent_k в L2_LAB: {list(L2_LAB.columns)}"
print(f"L2_LAB: SUB_COL='{SUB_COL}', PARENT_COL='{PARENT_COL}'")


# ---------------------------------------------------------------------------
# Общая утилита: FDR Бенджамини-Хохберга
# ---------------------------------------------------------------------------
def bh_fdr(pvals):
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    if n == 0:
        return p
    order = np.argsort(p)
    ranked = np.empty(n); ranked[order] = np.arange(1, n + 1)
    p_adj = p * n / ranked
    # монотонизация
    p_adj_sorted = np.minimum.accumulate(p_adj[order][::-1])[::-1]
    p_adj[order] = p_adj_sorted
    return np.minimum(p_adj, 1.0)


# ============================================================================
# ЧАСТЬ A. L1: ПРОФИЛИ, KW, PAIRWISE, МЕТРИКИ
# ============================================================================
print("\n" + "=" * 70)
print("ЧАСТЬ A. L1: ПРОФИЛИ, ТЕСТЫ, МЕТРИКИ")
print("=" * 70)

# --- A.1. Профили L1 ------------------------------------------------------
prof_raw    = LAB.groupby("cluster")[FEATS].mean().round(4)
prof_raw_med = LAB.groupby("cluster")[FEATS].median().round(4)
prof_raw["n_mo_avg"] = (LAB.groupby("cluster")
                        .apply(lambda g: g.oktmo_stable.nunique() / len(yrs))
                        .round(1))
prof_raw["share_avg_pct"] = (LAB.groupby("cluster").size() / len(LAB) * 100
                              ).round(2)
prof_z = (LAB.groupby("cluster")[ZCOLS].mean().round(4)
          if set(ZCOLS).issubset(LAB.columns)
          else None)
if prof_z is None:
    ztmp = dz.merge(LAB[["oktmo_stable", "year", "cluster"]],
                    on=["oktmo_stable", "year"])
    prof_z = ztmp.groupby("cluster")[ZCOLS].mean().round(4)

prof_raw.to_parquet(PROC / "cluster_profiles_raw.parquet")
prof_raw_med.to_parquet(PROC / "cluster_profiles_raw_median.parquet")
prof_z.to_parquet(PROC / "cluster_profiles_z.parquet")
print("\n=== RAW ПРОФИЛИ (L1) ===")
print(prof_raw.to_string())
print("\n=== Z-ПРОФИЛИ (L1) ===")
print(prof_z.to_string())

# --- A.2. Kruskal-Wallis L1 ---------------------------------------------
kw = []
for f in FEATS:
    gr = [g[f].dropna().values for _, g in LAB.groupby("cluster")]
    gr = [x for x in gr if len(x) > 5]
    if len(gr) < 2:
        continue
    H, p = stats.kruskal(*gr)
    kw.append({"feature": f, "H": float(H), "p": float(p),
               "n_groups": len(gr)})
KW = pd.DataFrame(kw).sort_values("p")
KW["p_fdr"]     = bh_fdr(KW["p"].values)
KW["significant"] = KW["p_fdr"] < 0.05
KW.to_parquet(PROC / "tests_kruskal.parquet", index=False)
print(f"\n=== KRUSKAL-WALLIS (L1) ===")
print(KW.round(6).to_string(index=False))

# --- A.3. Pairwise Mann-Whitney L1 --------------------------------------
pw = []
for f in FEATS:
    for i in range(K):
        for j in range(i + 1, K):
            a = LAB.loc[LAB.cluster == i, f].dropna()
            b = LAB.loc[LAB.cluster == j, f].dropna()
            if len(a) < 5 or len(b) < 5:
                continue
            U, p = stats.mannwhitneyu(a, b, alternative="two-sided")
            pw.append({
                "feature":      f,
                "pair":         f"c{i}_vs_c{j}",
                "i":            i, "j": j,
                "U":            float(U),
                "p":            float(p),
                "rank_biserial": 1 - 2 * U / (len(a) * len(b)),
                "direction":    (f"c{i}>c{j}" if a.median() > b.median()
                                 else f"c{j}>c{i}"),
            })
PW = pd.DataFrame(pw)
if len(PW) > 0:
    PW["p_fdr"]         = bh_fdr(PW["p"].values)
    PW["significant_fdr"] = PW["p_fdr"] < 0.05
PW.to_parquet(PROC / "tests_pairwise.parquet", index=False)
n_sig_pw = int(PW["significant_fdr"].sum()) if len(PW) else 0
print(f"\npairwise L1: {len(PW)} тестов, значимых после FDR: {n_sig_pw}")

# --- A.4. Метрики качества L1 по годам ----------------------------------
MQ = []
for y in yrs:
    suby = dz[dz.year == y].reset_index(drop=True)
    if "cluster_l1" not in suby.columns:
        continue
    X   = suby[ZCOLS].values
    lab = suby["cluster_l1"].values.astype(int)
    if len(np.unique(lab)) < 2:
        continue
    MQ.append({
        "year":       int(y),
        "K":          K,
        "silhouette": float(silhouette_score(X, lab)),
        "CH":         float(calinski_harabasz_score(X, lab)),
        "DB":         float(davies_bouldin_score(X, lab)),
        "sizes":      str(np.bincount(lab, minlength=K).tolist()),
    })
MQ = pd.DataFrame(MQ)
MQ.to_parquet(PROC / "quality_metrics.parquet", index=False)
print(f"\n=== МЕТРИКИ КАЧЕСТВА (L1, по годам) ===")
print(MQ.round(4).to_string(index=False))

# --- A.5. График L1: heatmap профилей -----------------------------------
fig, ax = plt.subplots(figsize=(1.05 * len(FEATS) + 2.5, 1.2 + 0.6 * K))
Mz = prof_z.values
im = ax.imshow(Mz, cmap="RdBu_r", vmin=-1.5, vmax=1.5, aspect="auto")
ax.set_xticks(range(len(FEATS)))
ax.set_xticklabels(FEATS, rotation=40, ha="right", fontsize=9)
ax.set_yticks(range(K))
ax.set_yticklabels(
    [f"K{c} (n={int(prof_raw.n_mo_avg.iloc[c])})" for c in range(K)])
for i in range(K):
    for j in range(len(FEATS)):
        ax.text(j, i, f"{Mz[i, j]:+.2f}", ha="center", va="center",
                fontsize=8,
                color="white" if abs(Mz[i, j]) > 0.9 else "black")
plt.colorbar(im, label="z-score")
ax.set_title("Z-профили кластеров L1 (средние по МО и годам)")
plt.tight_layout()
plt.savefig(VIZ / "04_profiles_heatmap.png", dpi=110)
plt.close()
print(f"\n✅ viz/04_profiles_heatmap.png")


# ============================================================================
# ЧАСТЬ B. L2: ПРОФИЛИ И ТЕСТЫ ВНУТРИ КАЖДОГО РОДИТЕЛЯ
# ============================================================================
print("\n" + "=" * 70)
print("ЧАСТЬ B. L2: ПРОФИЛИ И ТЕСТЫ ВНУТРИ РОДИТЕЛЕЙ")
print("=" * 70)

# --- B.0. Сборка L2_LAB с FEATS/ZCOLS ------------------------------------
L2_WORK = L2_LAB.merge(
    dz[["oktmo_stable", "year"] + FEATS + ZCOLS],
    on=["oktmo_stable", "year"], how="left",
)
assert L2_WORK[FEATS].notna().all().all(), \
    "после merge в L2_WORK есть NaN в FEATS"
print(f"L2_WORK: {L2_WORK.shape}, колонки: "
      f"{[c for c in L2_WORK.columns if c in ('parent_k','sub_k','cluster_l2')]}")

# --- B.1. Профили L2 по родителям (raw) ---------------------------------
rows_raw, rows_z = [], []
for k in range(K):
    subK  = L2_SUBK[k]
    sub_k = L2_WORK[L2_WORK[PARENT_COL] == k]
    for s in range(subK):
        m = sub_k[sub_k[SUB_COL] == s]
        if len(m) == 0:
            continue
        row = {
            "parent_k":   int(k),
            "sub_k":      int(s),
            "cluster_l2": f"K{k}.S{s}",
            "n_obs":      int(len(m)),
            "n_mo_avg":   float(m.oktmo_stable.nunique() / len(yrs)),
            "share_pct":  float(len(m) / len(sub_k) * 100),
        }
        for f in FEATS:
            row[f] = float(m[f].mean())
        rows_raw.append(row)

        rowz = {
            "parent_k":   int(k),
            "sub_k":      int(s),
            "cluster_l2": f"K{k}.S{s}",
            "n_obs":      int(len(m)),
        }
        for c in ZCOLS:
            rowz[c] = float(m[c].mean())
        rows_z.append(rowz)

PROF_L2_RAW = pd.DataFrame(rows_raw)
PROF_L2_Z   = pd.DataFrame(rows_z)
PROF_L2_RAW.to_parquet(PROC / "cluster_profiles_l2_raw.parquet", index=False)
PROF_L2_Z.to_parquet(PROC / "cluster_profiles_l2_z.parquet", index=False)

print("\n=== RAW ПРОФИЛИ L2 (по родителям) ===")
print(PROF_L2_RAW.round(3).to_string(index=False))
print("\n=== Z-ПРОФИЛИ L2 (по родителям) ===")
print(PROF_L2_Z.round(3).to_string(index=False))

# --- B.2. Kruskal-Wallis ВНУТРИ каждого родителя -------------------------
kw_l2_rows = []
for k in range(K):
    subK  = L2_SUBK[k]
    sub_k = L2_WORK[L2_WORK[PARENT_COL] == k]
    if subK < 2:
        continue
    for f in FEATS:
        groups = [g[f].dropna().values
                  for _, g in sub_k.groupby(SUB_COL)]
        groups = [x for x in groups if len(x) > 5]
        if len(groups) < 2:
            continue
        H, p = stats.kruskal(*groups)
        kw_l2_rows.append({
            "parent_k":   int(k),
            "feature":    f,
            "H":          float(H),
            "p":          float(p),
            "n_groups":   int(len(groups)),
        })
KW_L2 = pd.DataFrame(kw_l2_rows)

# FDR отдельно внутри каждого родителя
if len(KW_L2) > 0:
    KW_L2["p_fdr"] = np.nan
    for k in range(K):
        m = KW_L2["parent_k"] == k
        if m.sum() > 0:
            KW_L2.loc[m, "p_fdr"] = bh_fdr(KW_L2.loc[m, "p"].values)
    KW_L2["significant"] = KW_L2["p_fdr"] < 0.05
KW_L2.to_parquet(PROC / "tests_kruskal_l2.parquet", index=False)

print("\n=== KRUSKAL-WALLIS (L2, внутри родителей) ===")
if len(KW_L2):
    print(KW_L2.round(6).to_string(index=False))
    n_sig_kw_l2 = int(KW_L2["significant"].sum())
    print(f"\nзначимых (FDR<0.05): {n_sig_kw_l2}/{len(KW_L2)}")
else:
    print("нет тестов (все родители subK=1)")

# --- B.3. Pairwise Mann-Whitney ВНУТРИ каждого родителя ------------------
pw_l2_rows = []
for k in range(K):
    subK  = L2_SUBK[k]
    sub_k = L2_WORK[L2_WORK[PARENT_COL] == k]
    if subK < 2:
        continue
    for f in FEATS:
        for s1 in range(subK):
            for s2 in range(s1 + 1, subK):
                a = sub_k.loc[sub_k[SUB_COL] == s1, f].dropna()
                b = sub_k.loc[sub_k[SUB_COL] == s2, f].dropna()
                if len(a) < 5 or len(b) < 5:
                    continue
                U, p = stats.mannwhitneyu(a, b, alternative="two-sided")
                pw_l2_rows.append({
                    "parent_k":      int(k),
                    "feature":       f,
                    "pair":          f"K{k}.S{s1}_vs_K{k}.S{s2}",
                    "s1":            int(s1),
                    "s2":            int(s2),
                    "U":             float(U),
                    "p":             float(p),
                    "rank_biserial": 1 - 2 * U / (len(a) * len(b)),
                    "direction":     (f"S{s1}>S{s2}"
                                      if a.median() > b.median()
                                      else f"S{s2}>S{s1}"),
                })
PW_L2 = pd.DataFrame(pw_l2_rows)

# FDR отдельно внутри каждого родителя
if len(PW_L2) > 0:
    PW_L2["p_fdr"] = np.nan
    for k in range(K):
        m = PW_L2["parent_k"] == k
        if m.sum() > 0:
            PW_L2.loc[m, "p_fdr"] = bh_fdr(PW_L2.loc[m, "p"].values)
    PW_L2["significant_fdr"] = PW_L2["p_fdr"] < 0.05
PW_L2.to_parquet(PROC / "tests_pairwise_l2.parquet", index=False)

print("\n=== PAIRWISE MANN-WHITNEY (L2, внутри родителей) ===")
if len(PW_L2):
    n_sig_pw_l2 = int(PW_L2["significant_fdr"].sum())
    print(f"тестов: {len(PW_L2)}, значимых после FDR: {n_sig_pw_l2}")
    # Показываем топ-10 по p_fdr для наглядности
    top = PW_L2.sort_values("p_fdr").head(10)
    print("\nтоп-10 (по p_fdr):")
    print(top[["parent_k", "feature", "pair", "p_fdr",
               "direction", "rank_biserial"]].round(6).to_string(index=False))
else:
    print("нет тестов (все родители subK=1)")

# --- B.4. Метрики качества L2 по годам (внутри родителей) ---------------
MQ_L2 = []
for k in range(K):
    subK  = L2_SUBK[k]
    if subK < 2:
        continue
    sub_k = L2_WORK[L2_WORK[PARENT_COL] == k]
    for y in yrs:
        sy = sub_k[sub_k["year"] == y]
        if len(sy) < 20:
            continue
        X   = sy[ZCOLS].values
        lab = sy[SUB_COL].values.astype(int)
        if len(np.unique(lab)) < 2:
            continue
        MQ_L2.append({
            "parent_k":   int(k),
            "year":       int(y),
            "subK":       int(subK),
            "n":          int(len(sy)),
            "silhouette": float(silhouette_score(X, lab)),
            "CH":         float(calinski_harabasz_score(X, lab)),
            "DB":         float(davies_bouldin_score(X, lab)),
            "sizes":      str(np.bincount(lab, minlength=subK).tolist()),
        })
MQ_L2 = pd.DataFrame(MQ_L2)
MQ_L2.to_parquet(PROC / "quality_metrics_l2.parquet", index=False)
print("\n=== МЕТРИКИ КАЧЕСТВА (L2, по годам и родителям) ===")
if len(MQ_L2):
    print(MQ_L2.round(4).to_string(index=False))

# --- B.5. График L2: facetted heatmap профилей --------------------------
n_panels = sum(1 for k in range(K) if L2_SUBK[k] >= 2)
if n_panels == 0:
    print("\n(график L2 пропущен: все родители subK=1)")
else:
    fig, axes = plt.subplots(1, n_panels,
                             figsize=(5.5 * n_panels, 4.5),
                             dpi=110, sharey=True)
    axes = np.atleast_1d(axes).ravel()
    panel_idx = 0
    for k in range(K):
        if L2_SUBK[k] < 2:
            continue
        ax = axes[panel_idx]
        panel_idx += 1

        sub_prof = PROF_L2_Z[PROF_L2_Z["parent_k"] == k].sort_values("sub_k")
        if len(sub_prof) == 0:
            ax.axis("off"); continue

        z_cols = [c for c in ZCOLS if c in sub_prof.columns]
        data   = sub_prof[z_cols].T.values
        im = ax.imshow(data, aspect="auto", cmap="RdBu_r",
                       vmin=-1.5, vmax=1.5)

        ax.set_xticks(range(len(sub_prof)))
        ax.set_xticklabels(
            [f"K{k}.S{int(r['sub_k'])}" for _, r in sub_prof.iterrows()],
            fontsize=9)
        ax.set_yticks(range(len(z_cols)))
        ax.set_yticklabels([c.replace("_z", "") for c in z_cols],
                            fontsize=9)

        for i in range(len(z_cols)):
            for j in range(len(sub_prof)):
                v = data[i, j]
                c = "white" if abs(v) > 0.8 else "black"
                ax.text(j, i, f"{v:+.1f}", ha="center", va="center",
                        fontsize=8, color=c)
        ax.set_title(f"K{k} → {L2_SUBK[k]} подкластеров",
                     fontweight="bold", fontsize=11)

    plt.suptitle("Z-профили L2 (по родителям)",
                 fontweight="bold", fontsize=13)
    plt.tight_layout()
    plt.savefig(VIZ / "07_profiles_heatmap_l2.png", dpi=110)
    plt.close()
    print(f"\n✅ viz/07_profiles_heatmap_l2.png")


# ============================================================================
# ИТОГ
# ============================================================================
print("\n" + "=" * 70)
print("ИТОГ ЯЧЕЙКИ 8")
print("=" * 70)
print(f"L1: профилей {len(prof_raw)} кластеров × {len(FEATS)} признаков")
print(f"    KW: значимо {int(KW['significant'].sum())}/{len(KW)}")
print(f"    pairwise: значимо после FDR: {n_sig_pw}/{len(PW)}")
print()
print(f"L2: профилей {len(PROF_L2_RAW)} подкластеров × {len(FEATS)} признаков")
if len(KW_L2):
    print(f"    KW (внутри родителей): значимо "
          f"{int(KW_L2['significant'].sum())}/{len(KW_L2)}")
if len(PW_L2):
    print(f"    pairwise (внутри родителей): значимо после FDR "
          f"{int(PW_L2['significant_fdr'].sum())}/{len(PW_L2)}")
print()
print("Артефакты:")
for f in ["cluster_profiles_raw.parquet",
          "cluster_profiles_raw_median.parquet",
          "cluster_profiles_z.parquet",
          "cluster_profiles_l2_raw.parquet",
          "cluster_profiles_l2_z.parquet",
          "tests_kruskal.parquet", "tests_kruskal_l2.parquet",
          "tests_pairwise.parquet", "tests_pairwise_l2.parquet",
          "quality_metrics.parquet", "quality_metrics_l2.parquet"]:
    p = PROC / f
    if p.exists():
        print(f"  ✅ {f}")
for f in ["04_profiles_heatmap.png", "07_profiles_heatmap_l2.png"]:
    p = VIZ / f
    if p.exists():
        print(f"  ✅ viz/{f}")

tick(f"профили и тесты готовы: L1={len(prof_raw)}, L2={len(PROF_L2_RAW)}")

[1325.1s] профили и тесты (L1 + L2 внутри родителей)
L2_LAB: SUB_COL='sub_k', PARENT_COL='parent_k'

ЧАСТЬ A. L1: ПРОФИЛИ, ТЕСТЫ, МЕТРИКИ

=== RAW ПРОФИЛИ (L1) ===
         log_pop  rel_pop_region  urban_share  invest_pc  housing_pc  built_pc  bad_roads  msu_exp_pc   salary  retail_pc  n_mo_avg  share_avg_pct
cluster                                                                                                                                           
0        12.9252         23.6946       0.9756    11.1774      3.3596    0.6198    36.5127      7.2410  10.9619    10.5253       9.8           2.79
1        10.0856          1.4511       0.6658    10.3412      3.4297    0.4369    48.1904      7.9425  10.6911    10.2289     328.3          97.21

=== Z-ПРОФИЛИ (L1) ===
         log_pop_z  rel_pop_region_z  urban_share_z  invest_pc_z  housing_pc_z  built_pc_z  bad_roads_z  msu_exp_pc_z  salary_z  retail_pc_z
cluster                                                                            

In [14]:
# ============================================================================
# ============ ЯЧЕЙКА 9 ============
# ============================================================================
# ============================================================================
# ЯЧЕЙКА 9. АТРИБУТИРОВАННАЯ СЕТЬ НА KAN-ЭМБЕДДИНГАХ
# ============================================================================
# Часть A. Граф L1: mutual kNN на KAN-эмбеддингах (kan_e*).
#   • перебор k ∈ {5, 10, 15, 20}
#   • метрики графа: рёбра, плотность, компоненты, доля гиганта
#   • сравнение Spectral Clustering на графе с L1-метками (ARI)
#   • рабочий граф (k_main), сохранение рёбер
#   • визуализация в 2D PCA-проекции KAN-эмбеддингов
#
# Часть B. Граф L2 внутри каждого родителя:
#   • отдельный mutual kNN для urban и rural на L2-эмбеддингах
#   • те же метрики + ARI(Spectral vs L2-метки)
#   • сохранение рёбер L2
# ============================================================================
tick("построение сети (KAN-эмбеддинги)")

# --- Проверки доступности -------------------------------------------------
for _var in ("dz", "K", "L2_SUBK", "yrs", "PROC", "VIZ", "RNG",
             "CFG"):
    assert _var in globals(), (
        f"переменная {_var} не найдена — выполните ячейки 0–8 перед 9"
    )

# --- Определяем колонки KAN-L1 -------------------------------------------
KAN_L1_COLS = [c for c in dz.columns if c.startswith("kan_e")]
assert len(KAN_L1_COLS) > 0, "не найдены колонки kan_e* в dz"
print(f"KAN-L1 эмбеддинги: {KAN_L1_COLS}")

# --- Колонки KAN-L2 по родителям -----------------------------------------
KAN_L2_COLS = {k: [c for c in dz.columns if c.startswith(f"kan_l2_k{k}_e")]
               for k in range(K)}
for k, cols in KAN_L2_COLS.items():
    print(f"  KAN-L2[{k}]: {cols}")

# --- Год для построения графа --------------------------------------------
y_graph = int(yrs[-1])
print(f"\nгод построения графа: {y_graph}")


# ---------------------------------------------------------------------------
# Общие утилиты
# ---------------------------------------------------------------------------
def mutual_knn(X, k):
    """Mutual k-NN граф. Возвращает (A, mean_nn_dist)."""
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X)
    dist, idx = nn.kneighbors(X)
    n = len(X)
    A = np.zeros((n, n), bool)
    for i in range(n):
        A[i, idx[i, 1:]] = True
    return (A & A.T), float(dist[:, 1:].mean())


def connected_components(A):
    """Компоненты связности графа по матрице смежности."""
    n = A.shape[0]
    seen = np.zeros(n, bool)
    out = []
    for s in range(n):
        if seen[s]:
            continue
        stack, c = [s], []
        seen[s] = True
        while stack:
            v = stack.pop()
            c.append(v)
            for u in np.where(A[v])[0]:
                if not seen[u]:
                    seen[u] = True
                    stack.append(u)
        out.append(c)
    return out


def build_graph_profile(X, lab_attr, k_grid, K_label):
    """Перебор k в k_grid, возвращает DataFrame с метриками графа."""
    rows = []
    graphs = {}
    for k_ in k_grid:
        if k_ >= len(X) - 1:
            continue
        A, mean_dist = mutual_knn(X, k_)
        cs = connected_components(A)
        giant = max(len(c) for c in cs) / len(X) if cs else 0.0

        ari_spec = np.nan
        try:
            sc = SpectralClustering(n_clusters=K_label,
                                    affinity="precomputed",
                                    random_state=RNG,
                                    assign_labels="kmeans")
            lab_spec = sc.fit_predict(A.astype(float) + np.eye(len(A)) * 1e-6)
            ari_spec = float(adjusted_rand_score(lab_attr, lab_spec))
        except Exception:
            pass

        rows.append({
            "k":                    int(k_),
            "n_nodes":              int(len(X)),
            "edges":                int(A.sum() // 2),
            "density":              round(float(A.mean()), 5),
            "n_components":         int(len(cs)),
            "giant_share":          round(float(giant), 4),
            "ari_spectral_vs_attr": round(float(ari_spec), 4) if np.isfinite(ari_spec) else np.nan,
            "mean_nn_dist":         round(mean_dist, 4),
            "isolated":             int((A.sum(1) == 0).sum()),
        })
        graphs[int(k_)] = A
    return pd.DataFrame(rows), graphs


# ============================================================================
# ЧАСТЬ A. ГРАФ L1
# ============================================================================
print("\n" + "=" * 70)
print("ЧАСТЬ A. ГРАФ L1 (mutual kNN на KAN-L1)")
print("=" * 70)

sub_g = dz[dz["year"] == y_graph].reset_index(drop=True)
assert len(sub_g) > 100, f"на {y_graph} слишком мало МО: {len(sub_g)}"

# --- KAN-L1 эмбеддинги (8-мерные) — используем их напрямую ---------------
Xg_l1 = sub_g[KAN_L1_COLS].values.astype(np.float32)
assert np.isfinite(Xg_l1).all(), "NaN/inf в KAN-L1 эмбеддингах"

# --- Метки L1 ------------------------------------------------------------
if "cluster_l1" in sub_g.columns:
    lab_l1 = sub_g["cluster_l1"].values.astype(int)
else:
    lab_l1 = sub_g["cluster"].values.astype(int)

# --- PCA для визуализации (не для графа, только чтобы рисовать) ---------
pca_g = PCA(n_components=min(5, Xg_l1.shape[1]), random_state=RNG
            ).fit(Xg_l1)
XP_l1 = pca_g.transform(Xg_l1)
print(f"PCA {pca_g.n_components_}D объясняет "
      f"{pca_g.explained_variance_ratio_.sum():.3f} дисперсии")

# --- Перебор k -----------------------------------------------------------
GCMP_L1, GRAPHS_L1 = build_graph_profile(
    Xg_l1, lab_l1,
    k_grid=CFG["graph_k_grid"],
    K_label=K,
)
GCMP_L1.to_parquet(PROC / "graph_comparison.parquet", index=False)

print("\n=== СРАВНЕНИЕ СХЕМ РЁБЕР (L1, KAN) ===")
print(GCMP_L1.to_string(index=False))

# --- Рабочий граф --------------------------------------------------------
k_main = int(CFG["graph_k_main"])
assert k_main in GRAPHS_L1, f"k_main={k_main} отсутствует в GRAPHS_L1"
A_main_l1 = GRAPHS_L1[k_main]
print(f"\nрабочий граф L1: k={k_main}, "
      f"рёбер {int(A_main_l1.sum() // 2)}, "
      f"изолированных {int((A_main_l1.sum(1) == 0).sum())}")

# --- Сохранение рёбер L1 -------------------------------------------------
src, dst = np.where(np.triu(A_main_l1, 1))
EDGES_L1 = pd.DataFrame({
    "oktmo_src": sub_g["oktmo_stable"].values[src],
    "oktmo_dst": sub_g["oktmo_stable"].values[dst],
    "year":      y_graph,
})
EDGES_L1.to_parquet(PROC / "edges_mutual_knn.parquet", index=False)
print(f"✅ edges_mutual_knn.parquet: {EDGES_L1.shape}")

# --- Визуализация L1 ----------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 8))
tria = np.triu(A_main_l1, 1)
for i in range(len(XP_l1)):
    for j in np.where(tria[i])[0]:
        ax.plot([XP_l1[i, 0], XP_l1[j, 0]],
                [XP_l1[i, 1], XP_l1[j, 1]],
                color="grey", lw=0.15, alpha=0.3, zorder=1)

cmapK = plt.get_cmap("tab10", K)
for c in range(K):
    m = lab_l1 == c
    ax.scatter(XP_l1[m, 0], XP_l1[m, 1], s=9,
               color=cmapK(c), label=f"K{c} (n={int(m.sum())})", zorder=2)
ax.set_xlabel(f"PC1 ({pca_g.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca_g.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title(f"Атрибутированная сеть L1 (mutual kNN k={k_main}, KAN), {y_graph}\n"
             f"плотность {A_main_l1.mean():.4f} | "
             f"компонент {len(connected_components(A_main_l1))} | "
             f"ARI(Spectral vs L1) = "
             f"{GCMP_L1.set_index('k').loc[k_main, 'ari_spectral_vs_attr']:.4f}")
ax.legend(fontsize=8, markerscale=1.6)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(VIZ / "03_network_graph.png", dpi=110)
plt.close()
print(f"✅ viz/03_network_graph.png")


# ============================================================================
# ЧАСТЬ B. ГРАФЫ L2 ВНУТРИ КАЖДОГО РОДИТЕЛЯ
# ============================================================================
print("\n" + "=" * 70)
print("ЧАСТЬ B. ГРАФЫ L2 ВНУТРИ РОДИТЕЛЕЙ")
print("=" * 70)

# --- Определяем имя колонки parent_k -------------------------------------
PARENT_COL = "cluster_l1" if "cluster_l1" in dz.columns else "cluster"
SUB_COL    = next((c for c in ("cluster_l2_id", "sub_k_aligned", "sub_k")
                   if c in dz.columns), None)
assert SUB_COL is not None, \
    f"не найдена колонка sub_k в dz: {list(dz.columns)}"

GCMP_L2_ROWS = []
EDGES_L2_ROWS = []

for k in range(K):
    subK = L2_SUBK[k]
    if subK < 2:
        print(f"\n--- Родитель K{k}: subK=1, граф L2 пропущен ---")
        continue

    cols_l2 = KAN_L2_COLS[k]
    if not cols_l2:
        print(f"\n--- Родитель K{k}: нет KAN-L2 эмбеддингов — пропуск ---")
        continue

    print(f"\n--- Родитель K{k}: subK={subK}, пространство {cols_l2} ---")

    # Подвыборка: этот родитель в y_graph
    sub_k = (dz[(dz["year"] == y_graph) & (dz[PARENT_COL] == k)]
             .reset_index(drop=True))
    sub_k = sub_k.dropna(subset=cols_l2)
    if len(sub_k) < 20:
        print(f"  слишком мало МО ({len(sub_k)}) — пропуск")
        continue

    Xk = sub_k[cols_l2].values.astype(np.float32)
    assert np.isfinite(Xk).all(), f"K{k}: NaN/inf в KAN-L2"

    lab_k = sub_k[SUB_COL].values.astype(int)

    # Свой набор k для графа (маленькие родители — маленькие k)
    n_sub = len(Xk)
    k_grid_k = [min(gg, max(3, n_sub // 20))
                for gg in CFG["graph_k_grid"]]
    k_grid_k = sorted(set(k_grid_k))
    print(f"  МО={n_sub}, k_grid={k_grid_k}")

    gcmp, graphs = build_graph_profile(
        Xk, lab_k,
        k_grid=k_grid_k,
        K_label=subK,
    )
    if len(gcmp) == 0:
        print(f"  ⚠️  ни один k не построился")
        continue

    # --- Рабочий k для родителя ------------------------------------------
    # берём k = min(k_grid) с giant_share ≥ 0.9, иначе — максимум по giant_share
    cand = gcmp[gcmp["giant_share"] >= 0.90]
    if len(cand):
        k_best = int(cand.sort_values("k").iloc[0]["k"])
    else:
        k_best = int(gcmp.sort_values("giant_share", ascending=False)
                     .iloc[0]["k"])
    A_k = graphs[k_best]

    print(f"  рабочий граф: k={k_best}, "
          f"рёбер {int(A_k.sum() // 2)}, "
          f"giant_share={gcmp.set_index('k').loc[k_best,'giant_share']:.3f}, "
          f"ARI(Spectral vs L2)={gcmp.set_index('k').loc[k_best,'ari_spectral_vs_attr']:.4f}")

    # --- Собираем метрики --------------------------------------------------
    for _, r in gcmp.iterrows():
        GCMP_L2_ROWS.append({
            "parent_k":             int(k),
            "subK":                 int(subK),
            "n_nodes":              int(n_sub),
            **r.to_dict(),
        })

    # --- Рёбра ------------------------------------------------------------
    src, dst = np.where(np.triu(A_k, 1))
    if len(src):
        EDGES_L2_ROWS.append(pd.DataFrame({
            "parent_k":  int(k),
            "oktmo_src": sub_k["oktmo_stable"].values[src],
            "oktmo_dst": sub_k["oktmo_stable"].values[dst],
            "year":      y_graph,
        }))

    # --- Сохраняем граф родителя как .npy ---------------------------------
    np.save(PROC / f"graph_l2_parent{k}_adjacency.npy", A_k)

    # --- Визуализация: 2D PCA KAN-L2 с рёбрами и L2-метками --------------
    if len(cols_l2) >= 2:
        pca_k = PCA(n_components=2, random_state=RNG).fit(Xk)
        XP_k  = pca_k.transform(Xk)
        var_k = pca_k.explained_variance_ratio_.sum()

        fig, ax = plt.subplots(figsize=(8, 7))
        tria = np.triu(A_k, 1)
        for i in range(len(XP_k)):
            for j in np.where(tria[i])[0]:
                ax.plot([XP_k[i, 0], XP_k[j, 0]],
                        [XP_k[i, 1], XP_k[j, 1]],
                        color="grey", lw=0.15, alpha=0.25, zorder=1)
        cmap_sub = plt.get_cmap("tab10", subK)
        for s in range(subK):
            m = lab_k == s
            ax.scatter(XP_k[m, 0], XP_k[m, 1], s=10,
                       color=cmap_sub(s),
                       label=f"K{k}.S{s} (n={int(m.sum())})", zorder=2)
        ax.set_xlabel(f"PC1 ({pca_k.explained_variance_ratio_[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({pca_k.explained_variance_ratio_[1]*100:.1f}%)")
        ax.set_title(
            f"Граф L2: K{k} → {subK} подкластеров (mutual kNN k={k_best}), {y_graph}\n"
            f"плотность {A_k.mean():.4f} | "
            f"ARI(Spectral vs L2)="
            f"{gcmp.set_index('k').loc[k_best,'ari_spectral_vs_attr']:.4f}"
        )
        ax.legend(fontsize=8, markerscale=1.6)
        ax.grid(alpha=0.2)
        plt.tight_layout()
        plt.savefig(VIZ / f"03_network_l2_parent{k}.png", dpi=110)
        plt.close()
        print(f"  ✅ viz/03_network_l2_parent{k}.png")

# --- Сохранение метрик и рёбер L2 ---------------------------------------
GCMP_L2 = pd.DataFrame(GCMP_L2_ROWS)
GCMP_L2.to_parquet(PROC / "graph_comparison_l2.parquet", index=False)

if EDGES_L2_ROWS:
    EDGES_L2 = pd.concat(EDGES_L2_ROWS, ignore_index=True)
    EDGES_L2.to_parquet(PROC / "edges_mutual_knn_l2.parquet", index=False)
    print(f"\n✅ edges_mutual_knn_l2.parquet: {EDGES_L2.shape}")

print("\n=== СРАВНЕНИЕ СХЕМ РЁБЕР (L2, по родителям) ===")
if len(GCMP_L2):
    print(GCMP_L2.round(4).to_string(index=False))


# ============================================================================
# ИТОГ
# ============================================================================
print("\n" + "=" * 70)
print("ИТОГ ЯЧЕЙКИ 9")
print("=" * 70)
print(f"L1: год {y_graph}, k_main={k_main}, "
      f"рёбер {int(A_main_l1.sum() // 2)}, "
      f"ARI(Spectral vs L1)="
      f"{GCMP_L1.set_index('k').loc[k_main, 'ari_spectral_vs_attr']:.4f}")

if len(GCMP_L2):
    print(f"\nL2 по родителям (год {y_graph}):")
    for k in range(K):
        sub = GCMP_L2[GCMP_L2["parent_k"] == k]
        if len(sub) == 0:
            continue
        # Берём строку с максимальным ARI Spectral vs L2
        best_row = sub.sort_values("ari_spectral_vs_attr", ascending=False
                                    ).iloc[0]
        print(f"  K{k} (subK={int(best_row.subK)}): "
              f"k={int(best_row.k)}, "
              f"рёбер {int(best_row.edges)}, "
              f"ARI(Spectral vs L2)={best_row.ari_spectral_vs_attr:.4f}, "
              f"giant={best_row.giant_share:.3f}")
else:
    print("\n(графы L2 не построены: все родители subK=1)")

print()
print("Артефакты:")
for f in ["graph_comparison.parquet", "edges_mutual_knn.parquet",
          "graph_comparison_l2.parquet", "edges_mutual_knn_l2.parquet"]:
    p = PROC / f
    if p.exists():
        print(f"  ✅ {f}")
for f in sorted(VIZ.glob("03_network*.png")):
    print(f"  ✅ viz/{f.name}")

tick(f"сеть готова: L1 граф + {sum(1 for k in range(K) if L2_SUBK[k] >= 2)} "
     f"L2-графов")

[1327.5s] построение сети (KAN-эмбеддинги)
KAN-L1 эмбеддинги: ['kan_e0', 'kan_e1', 'kan_e2', 'kan_e3', 'kan_e4', 'kan_e5', 'kan_e6', 'kan_e7']
  KAN-L2[0]: ['kan_l2_k0_e0', 'kan_l2_k0_e1', 'kan_l2_k0_e2', 'kan_l2_k0_e3', 'kan_l2_k0_e4', 'kan_l2_k0_e5']
  KAN-L2[1]: ['kan_l2_k1_e0', 'kan_l2_k1_e1', 'kan_l2_k1_e2', 'kan_l2_k1_e3', 'kan_l2_k1_e4', 'kan_l2_k1_e5', 'kan_l2_k1_e6', 'kan_l2_k1_e7']

год построения графа: 2024

ЧАСТЬ A. ГРАФ L1 (mutual kNN на KAN-L1)
PCA 5D объясняет 0.921 дисперсии

=== СРАВНЕНИЕ СХЕМ РЁБЕР (L1, KAN) ===
 k  n_nodes  edges  density  n_components  giant_share  ari_spectral_vs_attr  mean_nn_dist  isolated
 5     2022   3043  0.00149           145       0.8516               -0.0194        0.7800       102
10     2022   6222  0.00304            40       0.9763               -0.0052        0.8761        31
15     2022   9390  0.00459            18       0.9891               -0.0019        0.9421        15
20     2022  12560  0.00614             7       0.9970     

In [15]:
# ============================================================================
# ============ ЯЧЕЙКА 10 ============
# ============================================================================
# ============================================================================
# ЯЧЕЙКА 10. ВНЕШНЯЯ ВАЛИДАЦИЯ (L1 + L2) + SBREINDEX КОРРЕЛЯЦИИ
# ============================================================================
# Часть A. L1: SberIndex, тип МО, региональная концентрация, jackknife,
#             тест без Москвы и Московской области.
# Часть B. L2: то же самое, но по подкластерам ВНУТРИ каждого родителя.
# Часть C. SberIndex × L1/L2: корреляции prevalence, профилей, median split.
#
# Исправления этой версии:
#   • корректный индексинг jackknife (cumsum(keep)-1 вместо where(keep[mem]))
#   • prev_l2_year строится через pivot_table (без MultiIndex)
#   • _finite_pair для безопасной чистки NaN перед pearsonr
#   • guards на пустые DataFrame перед печатью топ-N
# ============================================================================


# --- Проверки доступности -------------------------------------------------
for _var in ("LAB", "L2_LAB", "dz", "OKTMO_REF", "FEATS", "ZCOLS",
             "K", "L2_SUBK", "yrs", "fit", "PROC", "VIZ", "RNG", "CFG"):
    assert _var in globals(), (
        f"переменная {_var} не найдена — выполните ячейки 0–9 перед 10"
    )

# --- Определяем имена колонок в L2_LAB -----------------------------------
SUB_COL    = next((c for c in ("sub_k_aligned", "sub_k")
                   if c in L2_LAB.columns), None)
PARENT_COL = next((c for c in ("parent_k", "cluster_l1_aligned")
                    if c in L2_LAB.columns), None)
assert SUB_COL is not None, f"нет sub_k в L2_LAB: {list(L2_LAB.columns)}"
assert PARENT_COL is not None, f"нет parent_k в L2_LAB: {list(L2_LAB.columns)}"
print(f"L2_LAB: SUB_COL='{SUB_COL}', PARENT_COL='{PARENT_COL}'")

y_a, y_b = int(yrs[0]), int(yrs[-1])
SBER_USED = []


# ---------------------------------------------------------------------------
# Утилиты
# ---------------------------------------------------------------------------
def bh_fdr(pvals):
    """Benjamini-Hochberg FDR."""
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    if n == 0:
        return p
    order = np.argsort(p)
    ranked = np.empty(n); ranked[order] = np.arange(1, n + 1)
    p_adj = p * n / ranked
    p_adj_sorted = np.minimum.accumulate(p_adj[order][::-1])[::-1]
    p_adj[order] = p_adj_sorted
    return np.minimum(p_adj, 1.0)


def _finite_pair(x, y):
    """Убирает пары с NaN/inf. Возвращает (x_clean, y_clean)."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    return x[m], y[m]


# ============================================================================
# 10.0. Загрузка SberIndex (национальные ряды)
# ============================================================================
print("\n" + "=" * 70)
print("10.0. SberIndex: загрузка национальных рядов")
print("=" * 70)

sber_files = find_files([
    "median-wages*.parquet", "median_wages*.parquet", "*median-wage*.csv",
    "potrebitelskaya-aktivnost*.parquet", "*potrebitelskaya*.csv",
    "real-key-interest-rate*.parquet", "*key-interest*.csv",
    "prognoz-nedelnykh*.parquet", "*inflac*.parquet", "*inflac*.csv",
])
print("найдено файлов SberIndex:", [p.name for p in sber_files])

SBER = {}
for p in sber_files:
    try:
        d = (pd.read_parquet(p) if p.suffix.lower() == ".parquet"
             else pd.read_csv(p, sep=None, engine="python"))
        if "period" in d.columns:
            d["period"] = pd.to_datetime(d["period"], errors="coerce")
        if "value" in d.columns:
            d["value"] = pd.to_numeric(d["value"], errors="coerce")
        n = p.name.lower()
        if "median-wage" in n or "median_wage" in n:
            SBER["wage"] = d
        elif "potrebitelskaya" in n:
            SBER["activity"] = d
        elif "real-key" in n or "key-interest" in n:
            SBER["keyrate"] = d
        elif "prognoz" in n or "inflac" in n:
            SBER["inflation"] = d
    except Exception as e:
        print("  не прочитан:", p.name, e)

print("загружено рядов:", list(SBER.keys()))


# ============================================================================
# ЧАСТЬ A. L1: ВНЕШНЯЯ ВАЛИДАЦИЯ
# ============================================================================
print("\n" + "=" * 70)
print("ЧАСТЬ A. ВНЕШНЯЯ ВАЛИДАЦИЯ L1")
print("=" * 70)

WAGE_SIG_L1  = None
CL_GROWTH_L1 = None

# --- 10.A.1. SberIndex: рост зарплат по отраслям ----------------------
if "wage" in SBER:
    w = SBER["wage"].dropna(subset=["value"]).copy()
    w = w[w["period"] >= "2020-01-01"]
    w["year"] = w["period"].dt.year
    if "activity" in w.columns:
        wa = w.groupby(["activity", "year"])["value"].mean().unstack()
        wa = wa[[c for c in wa.columns if c in CFG["years"] or c == 2020]]
        if 2020 in wa.columns and y_b in wa.columns:
            WAGE_SIG_L1 = pd.DataFrame({
                "sber_growth": wa[y_b] / wa[2020],
                "level":       wa[y_b],
            })
            print(f"\n=== SBERINDEX: РОСТ ЗАРПЛАТ ПО ОТРАСЛЯМ 2020→{y_b} ===")
            print(WAGE_SIG_L1.sort_values("sber_growth", ascending=False)
                  .round(3).to_string())
            SBER_USED.append(f"median-wages: {len(WAGE_SIG_L1)} отраслей, "
                              f"2020-{y_b}")

# --- 10.A.2. Рост зарплат по L1-кластерам (Росстат) -------------------
sal = LAB.dropna(subset=["salary"])
gs = (sal[sal["year"].isin([y_a, y_b])]
      .pivot_table(index=["oktmo_stable", "cluster"],
                    columns="year", values="salary").dropna())

if y_a in gs.columns and y_b in gs.columns:
    gs["growth"] = gs[y_b] / gs[y_a]
    gsr = gs.reset_index()
    CL_GROWTH_L1 = (gsr.groupby("cluster")["growth"]
                    .agg(["median", "mean", "count"]).round(3))
    print(f"\n=== РОСТ ЗАРПЛАТ ПО L1-КЛАСТЕРАМ (Росстат, номинал) ===")
    print(CL_GROWTH_L1.to_string())

    H, p = stats.kruskal(*[g.growth.values for _, g in gsr.groupby("cluster")])
    print(f"Kruskal-Wallis: H={H:.2f}, p={p:.3e}")
    CL_GROWTH_L1.to_parquet(PROC / "external_wage_growth_by_cluster.parquet")

    # --- 10.A.3. Сопоставление с отраслями SberIndex -----------------
    if WAGE_SIG_L1 is not None:
        med = gsr.groupby("cluster")["growth"].median()
        rows = []
        for c in med.index:
            for ind, r in WAGE_SIG_L1.iterrows():
                rows.append({
                    "cluster":        int(c),
                    "industry":       ind,
                    "sber_growth":    float(r.sber_growth),
                    "cluster_growth": float(med[c]),
                    "gap":            abs(float(med[c]) - float(r.sber_growth)),
                })
        MATCH_L1 = pd.DataFrame(rows).sort_values("gap")
        MATCH_L1.to_parquet(PROC / "external_sber_industry_match.parquet",
                             index=False)
        print(f"\n=== БЛИЖАЙШИЕ ОТРАСЛИ SBERINDEX К L1-КЛАСТЕРУ ===")
        print(MATCH_L1.groupby("cluster").head(3).round(3)
              .to_string(index=False))
        SBER_USED.append("сопоставление роста зарплат L1 с отраслями Сбера")

# --- 10.A.4. Потребительская активность и ставка ----------------------
if "activity" in SBER:
    a = SBER["activity"].dropna(subset=["value"]).copy()
    a["quarter"] = a["period"].dt.to_period("Q").astype(str)
    key = next((c for c in ("age", "ipa_type") if c in a.columns), None)
    if key:
        ACT = a.groupby([key, "quarter"])["value"].mean().unstack()
        ACT.to_parquet(PROC / "sber_consumer_activity.parquet")
        print(f"\nSberIndex активность: {ACT.shape[0]} групп × "
              f"{ACT.shape[1]} кварталов "
              f"({a.period.min().date()}…{a.period.max().date()})")
        SBER_USED.append("potrebitelskaya-aktivnost")
if "keyrate" in SBER:
    kr = SBER["keyrate"].dropna(subset=["value"]).copy()
    col = next((c for c in ("key_rate_categories",)
                if c in kr.columns), None)
    if col:
        KR = kr.pivot_table(index="period", columns=col, values="value")
        KR.to_parquet(PROC / "sber_key_rate.parquet")
        print(f"SberIndex ставка: {KR.shape[1]} ряда, "
              f"{kr.period.min().date()}…{kr.period.max().date()}")
        SBER_USED.append("real-key-interest-rate")

# --- 10.A.5. Ручной шаг для муниципальных датасетов SberIndex ---------
mob_files = [p for p in find_files([
    "*mobilnost*", "*mobility*", "*potrebitelskie-traty*",
    "*consumer-spending*", "*sberindex*mo*",
    "*sber*mo*.parquet", "*sber*mo*.csv",
]) if p.suffix.lower() in (".parquet", ".csv", ".xlsx")]
if mob_files:
    print(f"\nнайдены муниципальные датасеты SberIndex: "
          f"{[p.name for p in mob_files]}")
    MV = []
    for p in mob_files:
        try:
            d = (pd.read_parquet(p) if p.suffix.lower() == ".parquet"
                 else pd.read_csv(p, sep=None, engine="python"))
            MV.append(d)
        except Exception as e:
            print("  не прочитан:", p.name, e)
    if MV:
        pd.concat(MV, ignore_index=True).to_parquet(
            PROC / "sber_municipal_raw.parquet", index=False)
        SBER_USED.append("муниципальные датасеты SberIndex")
else:
    print("\n[РУЧНОЙ ШАГ] муниципальные датасеты SberIndex не найдены.")

# --- 10.A.6. L1-кластеры vs тип МО -----------------------------------
print("\n=== L1 vs тип МО (внешний административный признак) ===")
ct_abs = pd.crosstab(LAB["cluster"], LAB["mun_type"].fillna("n/a"))
CT_L1 = pd.crosstab(LAB["cluster"], LAB["mun_type"].fillna("n/a"),
                     normalize="index").round(3)
print(CT_L1.to_string())
CT_L1.to_parquet(PROC / "external_cluster_by_muntype.parquet")
chi2, pchi, dof, _ = stats.chi2_contingency(ct_abs)
print(f"chi2={chi2:.1f}, dof={dof}, p={pchi:.3e}")

# --- 10.A.7. Региональная концентрация L1 ----------------------------
last_l1 = LAB[LAB["year"] == y_b]
conc_l1 = last_l1.groupby("cluster").apply(
    lambda g: pd.Series({
        "n_mo":              g["oktmo_stable"].nunique(),
        "n_regions":         g["region_name"].nunique(),
        "top1_region_share": g["region_name"].value_counts(normalize=True).iloc[0],
        "top3_region_share": g["region_name"].value_counts(normalize=True).head(3).sum(),
        "top_region":        g["region_name"].value_counts().index[0],
    })
)
print(f"\n=== РЕГИОНАЛЬНАЯ КОНЦЕНТРАЦИЯ L1 ({y_b}) ===")
print(conc_l1.round(3).to_string())
conc_l1.to_parquet(PROC / "external_cluster_region_concentration.parquet")

# --- 10.A.8. Jackknife по L1-кластерам -------------------------------
print("\n=== JACKKNIFE L1 (убираем крупнейший регион класса) ===")
sub_ref = dz[dz["year"] == y_b].reset_index(drop=True)
if "cluster_l1" in sub_ref.columns:
    X_ref_L1 = sub_ref[ZCOLS].values
    reg_ref  = np.asarray(sub_ref["region_name"].astype(str).values)
    lab_ref  = np.asarray(sub_ref["cluster_l1"].astype(int).values)
    n_ref    = len(X_ref_L1)

    JKCL_L1 = []
    for c in range(K):
        mem = np.where(lab_ref == c)[0]
        if not len(mem):
            continue

        vc = pd.Series(reg_ref[mem]).value_counts()
        top = vc.index[0]

        keep = (reg_ref != top)
        pos_in_Xk = np.cumsum(keep) - 1
        mem_in_Xk = pos_in_Xk[mem]
        keep_mem  = keep[mem]
        mem_keep_idx = mem_in_Xk[keep_mem]

        Xk = X_ref_L1[keep]
        lab_k, _ = fit(Xk, K)

        holds = (float(np.mean(lab_k[mem_keep_idx] == c))
                 if len(mem_keep_idx) else float("nan"))

        JKCL_L1.append({
            "cluster":            int(c),
            "n_mo":               int(len(mem)),
            "top_region":         str(top),
            "top_region_n":       int(vc.iloc[0]),
            "top_region_share":   round(float(vc.iloc[0] / len(mem)), 3),
            "n_mo_kept":          int(keep_mem.sum()),
            "holds_without_top":  round(holds, 3),
        })

    JKCL_L1_DF = pd.DataFrame(JKCL_L1)
    print(JKCL_L1_DF.to_string(index=False))
    JKCL_L1_DF.to_parquet(PROC / "external_cluster_jackknife.parquet",
                            index=False)

# --- 10.A.9. L1-тест без Москвы и Московской области -----------------
print("\n=== L1: ТЕСТ БЕЗ МОСКВЫ И МО ===")
mask_msk = dz["region_name"].astype(str).str.contains(
    "Москва", case=False, na=False)
print(f"исключается МО: {dz[mask_msk].oktmo_stable.nunique()}")
dz_nm = dz[~mask_msk]
sub_nm = dz_nm[dz_nm["year"] == y_b].reset_index(drop=True)
X_nm   = sub_nm[ZCOLS].values
lab_nm, _ = fit(X_nm, K)
qual_nm = {
    "silhouette": float(silhouette_score(X_nm, lab_nm)),
    "CH":         float(calinski_harabasz_score(X_nm, lab_nm)),
    "DB":         float(davies_bouldin_score(X_nm, lab_nm)),
    "sizes":      np.bincount(lab_nm, minlength=K).tolist(),
}
print(f"без Москвы/МО: МО={dz_nm.oktmo_stable.nunique()}, "
      f"sil={qual_nm['silhouette']:.4f}, DB={qual_nm['DB']:.4f}, "
      f"sizes={qual_nm['sizes']}")

common = np.intersect1d(sub_nm["oktmo_stable"].values,
                         sub_ref["oktmo_stable"].values)
lab_full_c = sub_ref.set_index("oktmo_stable").loc[common, "cluster_l1"].values
lab_nm_c = pd.Series(lab_nm,
                      index=sub_nm["oktmo_stable"].values
                      ).reindex(common).values
ari_nm_l1 = float(adjusted_rand_score(lab_full_c, lab_nm_c))
print(f"ARI(полное vs без Москвы/МО) = {ari_nm_l1:.4f}")

json.dump({
    "qual":        qual_nm,
    "ari_vs_full": ari_nm_l1,
    "n_mo":        int(dz_nm.oktmo_stable.nunique()),
}, open(PROC / "external_without_moscow.json", "w", encoding="utf-8"),
    ensure_ascii=False, indent=1)


# ============================================================================
# ЧАСТЬ B. L2: ВНЕШНЯЯ ВАЛИДАЦИЯ (внутри родителей)
# ============================================================================
print("\n" + "=" * 70)
print("ЧАСТЬ B. ВНЕШНЯЯ ВАЛИДАЦИЯ L2")
print("=" * 70)

# --- Собираем L2_VAL с FEATS, ZCOLS и метаданными ОКТМО ---------------
extra_cols = list(ZCOLS)
for k in range(K):
    if f"kan_l2_k{k}_e0" in dz.columns:
        extra_cols += [c for c in dz.columns if c.startswith(f"kan_l2_k{k}_e")]

L2_VAL = L2_LAB.merge(
    OKTMO_REF[["oktmo_stable", "mun_type", "region_name"]],
    on="oktmo_stable", how="left",
).merge(
    dz[["oktmo_stable", "year"] + FEATS + list(dict.fromkeys(extra_cols))],
    on=["oktmo_stable", "year"], how="left",
)
assert L2_VAL[FEATS].notna().all().all(), "NaN в FEATS после merge"
assert L2_VAL[ZCOLS].notna().all().all(), "NaN в ZCOLS после merge"
print(f"L2_VAL: {L2_VAL.shape}")

# --- 10.B.1. L2-подкластеры vs тип МО (chi² внутри родителя) ---------
print("\n=== L2 vs тип МО (chi² внутри каждого родителя) ===")
chi2_l2_rows = []

for k in range(K):
    subK = L2_SUBK[k]
    if subK < 2:
        continue
    sub_k = L2_VAL[L2_VAL[PARENT_COL] == k]
    if len(sub_k) == 0:
        continue

    ct_abs_k = pd.crosstab(sub_k[SUB_COL], sub_k["mun_type"].fillna("n/a"))
    ct_pct_k = pd.crosstab(sub_k[SUB_COL], sub_k["mun_type"].fillna("n/a"),
                            normalize="index").round(3)
    print(f"\n  K{k} (subK={subK}):")
    print(ct_pct_k.to_string())

    try:
        chi2_k, p_k, dof_k, _ = stats.chi2_contingency(ct_abs_k)
    except Exception:
        chi2_k, p_k, dof_k = np.nan, np.nan, np.nan
    print(f"    chi2={chi2_k:.1f}, dof={dof_k}, p={p_k:.3e}")
    chi2_l2_rows.append({
        "parent_k": int(k), "subK": int(subK),
        "chi2": float(chi2_k) if np.isfinite(chi2_k) else np.nan,
        "dof":  float(dof_k)  if np.isfinite(dof_k)  else np.nan,
        "p":    float(p_k)    if np.isfinite(p_k)    else np.nan,
        "n_obs": int(len(sub_k)),
    })

CHI2_L2 = pd.DataFrame(chi2_l2_rows)
CHI2_L2.to_parquet(PROC / "external_cluster_by_muntype_l2.parquet",
                    index=False)
print("\n=== СВОДКА chi² (L2 vs тип МО) ===")
print(CHI2_L2.round(4).to_string(index=False))

# --- 10.B.2. Рост зарплат по L2-подкластерам --------------------------
print("\n=== РОСТ ЗАРПЛАТ ПО L2-ПОДКЛАСТЕРАМ (Росстат, номинал) ===")
CL_GROWTH_L2 = None
gs_l2 = (L2_VAL[L2_VAL["year"].isin([y_a, y_b])]
         .pivot_table(index=["oktmo_stable", PARENT_COL, SUB_COL],
                       columns="year", values="salary").dropna())
if y_a in gs_l2.columns and y_b in gs_l2.columns:
    gs_l2["growth"] = gs_l2[y_b] / gs_l2[y_a]
    gsr_l2 = gs_l2.reset_index()
    gsr_l2["cluster_l2"] = (gsr_l2[PARENT_COL].astype(str) + ".S" +
                              gsr_l2[SUB_COL].astype(str))
    CL_GROWTH_L2 = (gsr_l2.groupby([PARENT_COL, SUB_COL])["growth"]
                    .agg(["median", "mean", "count"]).round(3))
    print(CL_GROWTH_L2.to_string())
    CL_GROWTH_L2.to_parquet(PROC / "external_wage_growth_by_subcluster.parquet")

    print("\n  Kruskal-Wallis по росту зарплат внутри каждого родителя:")
    for k in range(K):
        subK = L2_SUBK[k]
        if subK < 2:
            continue
        sk = gsr_l2[gsr_l2[PARENT_COL] == k]
        groups = [g["growth"].values for _, g in sk.groupby(SUB_COL)]
        groups = [x for x in groups if len(x) > 5]
        if len(groups) < 2:
            continue
        H, p = stats.kruskal(*groups)
        print(f"    K{k}: H={H:.2f}, p={p:.3e}")

    if WAGE_SIG_L1 is not None:
        rows = []
        for (parent, sub), r in (gsr_l2.groupby([PARENT_COL, SUB_COL])
                                  ["growth"].median().items()):
            for ind, ws in WAGE_SIG_L1.iterrows():
                rows.append({
                    "parent_k":          int(parent),
                    "sub_k":             int(sub),
                    "cluster_l2":        f"K{parent}.S{sub}",
                    "industry":          ind,
                    "sber_growth":       float(ws.sber_growth),
                    "subcluster_growth": float(r),
                    "gap":               abs(float(r) - float(ws.sber_growth)),
                })
        MATCH_L2 = pd.DataFrame(rows).sort_values(["parent_k", "sub_k", "gap"])
        MATCH_L2.to_parquet(PROC / "external_sber_industry_match_l2.parquet",
                             index=False)
        print(f"\n=== БЛИЖАЙШИЕ ОТРАСЛИ SBERINDEX К КАЖДОМУ L2-ПОДКЛАСТЕРУ ===")
        print(MATCH_L2.groupby(["parent_k", "sub_k"]).head(2).round(3)
              .to_string(index=False))

# --- 10.B.4. Региональная концентрация L2 ----------------------------
print(f"\n=== РЕГИОНАЛЬНАЯ КОНЦЕНТРАЦИЯ L2 ({y_b}) ===")
last_l2 = L2_VAL[L2_VAL["year"] == y_b]
conc_l2 = (last_l2
           .groupby([PARENT_COL, SUB_COL])
           .apply(lambda g: pd.Series({
               "cluster_l2":        f"K{int(g.name[0])}.S{int(g.name[1])}",
               "n_mo":              g["oktmo_stable"].nunique(),
               "n_regions":         g["region_name"].nunique(),
               "top1_region_share": g["region_name"].value_counts(
                                       normalize=True).iloc[0],
               "top3_region_share": g["region_name"].value_counts(
                                       normalize=True).head(3).sum(),
               "top_region":        g["region_name"].value_counts().index[0],
           }))
           .round(3))
print(conc_l2.to_string())
conc_l2.to_parquet(PROC / "external_cluster_region_concentration_l2.parquet")

# --- 10.B.5. Jackknife L2 (leave-top-region-out внутри родителя) -----
print("\n=== JACKKNIFE L2 (убираем крупнейший регион родителя) ===")
JKCL_L2 = []
for k in range(K):
    subK = L2_SUBK[k]
    if subK < 2:
        continue
    sub_k = dz[(dz["year"] == y_b) & (dz["cluster_l1"] == k)
               ].reset_index(drop=True)
    if len(sub_k) < 20:
        continue
    cols_k = L2_INFO[k]["cols"] if "L2_INFO" in globals() else None
    if cols_k is None or not all(c in sub_k.columns for c in cols_k):
        cols_k = ZCOLS
    Xk_all = sub_k[cols_k].values
    reg_k  = np.asarray(sub_k["region_name"].astype(str).values)

    sub_l2_k = L2_VAL[(L2_VAL[PARENT_COL] == k) & (L2_VAL["year"] == y_b)]
    lab_map  = dict(zip(sub_l2_k["oktmo_stable"], sub_l2_k[SUB_COL]))
    lab_k_all = np.array([lab_map.get(mo, -1)
                           for mo in sub_k["oktmo_stable"].values])
    valid_mask = lab_k_all >= 0
    if valid_mask.sum() < 20:
        continue
    Xk_all    = Xk_all[valid_mask]
    reg_k     = reg_k[valid_mask]
    lab_k_all = lab_k_all[valid_mask]

    for s in range(subK):
        mem = np.where(lab_k_all == s)[0]
        if len(mem) < 5:
            continue
        vc = pd.Series(reg_k[mem]).value_counts()
        top = vc.index[0]

        keep = (reg_k != top)
        pos_in_Xk = np.cumsum(keep) - 1
        mem_in_Xk = pos_in_Xk[mem]
        keep_mem  = keep[mem]
        mem_keep_idx = mem_in_Xk[keep_mem]

        Xk = Xk_all[keep]
        lab_kk, _ = fit(Xk, subK)

        holds = (float(np.mean(lab_kk[mem_keep_idx] == s))
                 if len(mem_keep_idx) else float("nan"))

        JKCL_L2.append({
            "parent_k":          int(k),
            "sub_k":             int(s),
            "cluster_l2":        f"K{k}.S{s}",
            "n_mo":              int(len(mem)),
            "top_region":        str(top),
            "top_region_share":  round(float(vc.iloc[0] / len(mem)), 3),
            "n_mo_kept":         int(keep_mem.sum()),
            "holds_without_top": round(holds, 3),
        })
JKCL_L2_DF = pd.DataFrame(JKCL_L2)
if len(JKCL_L2_DF):
    print(JKCL_L2_DF.to_string(index=False))
    JKCL_L2_DF.to_parquet(PROC / "external_cluster_jackknife_l2.parquet",
                           index=False)

# --- 10.B.6. L2-тест без Москвы и Московской области -----------------
print("\n=== L2: ТЕСТ БЕЗ МОСКВЫ И МО ===")
L2_NM = L2_VAL[~L2_VAL["region_name"].astype(str).str.contains(
    "Москва", case=False, na=False)]
print(f"после исключения: {L2_NM.oktmo_stable.nunique()} МО, "
      f"{len(L2_NM)} наблюдений")

qual_l2_nm = {}
for k in range(K):
    subK = L2_SUBK[k]
    if subK < 2:
        continue
    sk = L2_NM[L2_NM[PARENT_COL] == k]
    if len(sk) < 20:
        continue
    cols_kan = [c for c in sk.columns
                if c.startswith(f"kan_l2_k{k}_e")]
    if cols_kan:
        cols_use, space_name = cols_kan, "KAN-L2"
    elif all(c in sk.columns for c in ZCOLS):
        cols_use, space_name = ZCOLS, "ZCOLS"
    else:
        cols_use, space_name = FEATS, "FEATS(raw)"

    X_sk   = sk[cols_use].values.astype(np.float32)
    lab_sk = sk[SUB_COL].values.astype(int)
    if len(np.unique(lab_sk)) < 2:
        continue
    if not np.isfinite(X_sk).all():
        continue

    qual_l2_nm[k] = {
        "space":      space_name,
        "silhouette": float(silhouette_score(X_sk, lab_sk)),
        "CH":         float(calinski_harabasz_score(X_sk, lab_sk)),
        "DB":         float(davies_bouldin_score(X_sk, lab_sk)),
        "n_obs":      int(len(sk)),
        "sizes":      np.bincount(lab_sk, minlength=subK).tolist(),
    }
    print(f"  K{k}: space={space_name}, sil={qual_l2_nm[k]['silhouette']:.4f}, "
          f"DB={qual_l2_nm[k]['DB']:.4f}, n={qual_l2_nm[k]['n_obs']}, "
          f"sizes={qual_l2_nm[k]['sizes']}")

ari_nm_l2 = {}
for k in range(K):
    subK = L2_SUBK[k]
    if subK < 2:
        continue
    sub_full = L2_VAL[(L2_VAL[PARENT_COL] == k) & (L2_VAL["year"] == y_b)]
    sub_nm   = L2_NM[(L2_NM[PARENT_COL] == k) & (L2_NM["year"] == y_b)]
    common_mo = np.intersect1d(sub_full["oktmo_stable"].values,
                                sub_nm["oktmo_stable"].values)
    if len(common_mo) < 5:
        continue
    lab_full_c = (sub_full.set_index("oktmo_stable")
                  .loc[common_mo, SUB_COL].values.astype(int))
    sk_nm = sub_nm.reset_index(drop=True)
    cols_kan = [c for c in sk_nm.columns if c.startswith(f"kan_l2_k{k}_e")]
    if cols_kan:
        cols_use = cols_kan
    elif all(c in sk_nm.columns for c in ZCOLS):
        cols_use = ZCOLS
    else:
        cols_use = FEATS
    X_nm = sk_nm[cols_use].values.astype(np.float32)
    lab_nm_fit = KMeans(n_clusters=subK, n_init=20, random_state=RNG
                        ).fit_predict(X_nm)
    mo_to_lab_nm = dict(zip(sk_nm["oktmo_stable"].values, lab_nm_fit))
    lab_nm_c = np.array([mo_to_lab_nm.get(mo, -1) for mo in common_mo])
    valid = lab_nm_c >= 0
    if valid.sum() < 5:
        continue
    ari_nm_l2[k] = float(adjusted_rand_score(
        lab_full_c[valid], lab_nm_c[valid]))

print(f"\nARI(полное vs без Москвы/МО):")
for k, a in ari_nm_l2.items():
    print(f"  K{k}: ARI={a:.4f}")

json.dump({
    "qual_per_parent": qual_l2_nm,
    "ari_vs_full":     ari_nm_l2,
    "n_mo_total":      int(L2_NM.oktmo_stable.nunique()),
}, open(PROC / "external_without_moscow_l2.json", "w", encoding="utf-8"),
    ensure_ascii=False, indent=1)

json.dump({"sber_used": SBER_USED},
          open(PROC / "sber_usage.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=1)
print(f"\nSberIndex использован в: {SBER_USED}")


# ============================================================================
# ЧАСТЬ C. SBREINDEX × L1/L2: КОРРЕЛЯЦИИ И TIMELINE
# ============================================================================
print("\n" + "=" * 70)
print("10.C. SBERINDEX × L1/L2: КОРРЕЛЯЦИИ И TIMELINE")
print("=" * 70)

# --- 10.C.1. Квартальные ряды SberIndex --------------------------------
SBER_Q = {}
for name in ["wage", "activity", "keyrate", "inflation"]:
    if name not in SBER:
        continue
    s = SBER[name].dropna(subset=["value"]).copy()
    s["quarter"] = s["period"].dt.to_period("Q").dt.to_timestamp()
    q = (s.groupby("quarter")["value"].mean()
           .reset_index()
           .rename(columns={"value": f"sber_{name}"}))
    SBER_Q[name] = q

SBER_Q_ALL = None
for name, q in SBER_Q.items():
    SBER_Q_ALL = q if SBER_Q_ALL is None else SBER_Q_ALL.merge(
        q, on="quarter", how="outer")
SBER_Q_ALL = SBER_Q_ALL.sort_values("quarter").reset_index(drop=True)
SBER_Q_ALL.to_parquet(PROC / "sber_quarterly.parquet", index=False)
print(f"квартальные SberIndex: {SBER_Q_ALL.shape}")

# --- 10.C.2. Годовые средние SberIndex ---------------------------------
SBER_Y = SBER_Q_ALL.copy()
SBER_Y["year"] = SBER_Y["quarter"].dt.year
SBER_Y = (SBER_Y.groupby("year")
          [[c for c in SBER_Y.columns if c.startswith("sber_")]]
          .mean().round(4).reset_index())
SBER_Y = SBER_Y[(SBER_Y["year"] >= y_a) & (SBER_Y["year"] <= y_b)]
SBER_Y.to_parquet(PROC / "sber_yearly.parquet", index=False)
print(f"годовые SberIndex ({y_a}–{y_b}): {SBER_Y.shape}")
print(SBER_Y.to_string(index=False))

# --- 10.C.3. Prevalence L1/L2 по годам (без MultiIndex) ---------------
prev_l1_year = (LAB.groupby(["year", "cluster"]).size()
                .unstack(fill_value=0))
prev_l1_year = prev_l1_year.div(prev_l1_year.sum(axis=1), axis=0) * 100
prev_l1_year.index = prev_l1_year.index.astype(int)
prev_l1_year.columns = [f"L1_K{c}" for c in prev_l1_year.columns]

# L2 prevalence — через reset_index + pivot_table (гарантированно плоский индекс)
_prev_l2_long = (L2_VAL
                 .groupby(["year", PARENT_COL, SUB_COL])
                 .size()
                 .reset_index(name="n"))
_prev_l2_long["cluster"] = ("K" +
                             _prev_l2_long[PARENT_COL].astype(str) +
                             ".S" +
                             _prev_l2_long[SUB_COL].astype(str))
prev_l2_year = _prev_l2_long.pivot_table(
    index="year", columns="cluster", values="n", fill_value=0
)
prev_l2_year = prev_l2_year.div(prev_l2_year.sum(axis=1), axis=0) * 100
prev_l2_year.index = prev_l2_year.index.astype(int)

# Сводная таблица prevalence
PREV = pd.concat([prev_l1_year, prev_l2_year], axis=1).reset_index()
PREV = PREV.rename(columns={"year": "year"})
PREV.to_parquet(PROC / "prevalence_yearly.parquet", index=False)
print(f"\nprevalence по годам: {PREV.shape}, "
      f"кластеров: {len([c for c in PREV.columns if c != 'year'])}")

SBER_INDICATORS = [c for c in SBER_Y.columns if c.startswith("sber_")]

# --- 10.C.4. Корреляции SberIndex × prevalence L1/L2 ------------------
corr_rows = []
for ind in SBER_INDICATORS:
    sber_col = (SBER_Y[["year", ind]]
                .dropna(subset=[ind]))
    merged = sber_col.merge(PREV, on="year", how="inner")
    if len(merged) < 4:
        continue
    for cl in [c for c in PREV.columns if c != "year"]:
        x, y_vals = _finite_pair(merged[ind].values, merged[cl].values)
        if len(x) < 4:
            continue
        if np.std(x) < 1e-9 or np.std(y_vals) < 1e-9:
            continue
        try:
            pr, pp = pearsonr(x, y_vals)
        except Exception:
            pr, pp = np.nan, np.nan
        try:
            sr, sp = spearmanr(x, y_vals)
        except Exception:
            sr, sp = np.nan, np.nan
        corr_rows.append({
            "indicator":  ind.replace("sber_", ""),
            "cluster":    cl,
            "n_years":    int(len(x)),
            "pearson_r":  float(pr) if np.isfinite(pr) else np.nan,
            "pearson_p":  float(pp) if np.isfinite(pp) else np.nan,
            "spearman_r": float(sr) if np.isfinite(sr) else np.nan,
            "spearman_p": float(sp) if np.isfinite(sp) else np.nan,
        })
CORR_PREV = pd.DataFrame(corr_rows)

if len(CORR_PREV):
    valid = CORR_PREV["pearson_p"].notna()
    CORR_PREV["pearson_p_fdr"] = np.nan
    if valid.sum() > 0:
        CORR_PREV.loc[valid, "pearson_p_fdr"] = bh_fdr(
            CORR_PREV.loc[valid, "pearson_p"].to_numpy())
    CORR_PREV["significant_fdr"] = CORR_PREV["pearson_p_fdr"] < 0.05

CORR_PREV.to_parquet(PROC / "sber_vs_prevalence_corr.parquet", index=False)
print(f"\nSberIndex × prevalence: {len(CORR_PREV)} корреляций")

if len(CORR_PREV) and CORR_PREV["pearson_r"].notna().any():
    top_corr = CORR_PREV.dropna(subset=["pearson_r"]).copy()
    top_corr["abs_r"] = top_corr["pearson_r"].abs()
    top_corr = top_corr.sort_values("abs_r", ascending=False).head(10)
    print("топ-10 (по |r|):")
    print(top_corr[["indicator", "cluster", "n_years",
                     "pearson_r", "spearman_r"]].round(3).to_string(index=False))
else:
    print("топ-10 (по |r|): нет валидных корреляций")

# --- 10.C.5. Корреляции SberIndex × средние профили L1/L2 --------------
prof_l1_by_year = (LAB.groupby(["year", "cluster"])[FEATS]
                    .mean().reset_index())
prof_l1_by_year["cluster"] = "L1_K" + prof_l1_by_year["cluster"].astype(str)

L2_VAL_FEATS = L2_VAL.merge(
    dz[["oktmo_stable", "year"] + FEATS],
    on=["oktmo_stable", "year"], how="left", suffixes=("", "_f"))
FEATS_USE = [c for c in FEATS if c in L2_VAL_FEATS.columns]
prof_l2_by_year = (L2_VAL_FEATS
                    .groupby(["year", PARENT_COL, SUB_COL])[FEATS_USE]
                    .mean().reset_index())
prof_l2_by_year["cluster"] = ("L2_K" + prof_l2_by_year[PARENT_COL].astype(str)
                                + ".S" + prof_l2_by_year[SUB_COL].astype(str))

PROF_BY_YEAR = pd.concat([
    prof_l1_by_year,
    prof_l2_by_year[["year", "cluster"] + FEATS_USE],
], ignore_index=True)

prof_corr_rows = []
for ind in SBER_INDICATORS:
    sber_col = (SBER_Y[["year", ind]]
                .dropna(subset=[ind]))
    merged = sber_col.merge(PROF_BY_YEAR, on="year", how="inner")
    if len(merged) < 4:
        continue
    for cl in merged["cluster"].unique():
        sub = merged[merged["cluster"] == cl]
        for f in FEATS_USE:
            x, y_vals = _finite_pair(sub[ind].values, sub[f].values)
            if len(x) < 4:
                continue
            if np.std(x) < 1e-9 or np.std(y_vals) < 1e-9:
                continue
            try:
                r, p = pearsonr(x, y_vals)
            except Exception:
                continue
            prof_corr_rows.append({
                "indicator": ind.replace("sber_", ""),
                "cluster":   cl,
                "feature":   f,
                "n_years":   int(len(x)),
                "pearson_r": float(r),
                "pearson_p": float(p),
            })
CORR_PROF = pd.DataFrame(prof_corr_rows)
if len(CORR_PROF):
    CORR_PROF["pearson_p_fdr"] = bh_fdr(CORR_PROF["pearson_p"].to_numpy())
    CORR_PROF["significant_fdr"] = CORR_PROF["pearson_p_fdr"] < 0.05
CORR_PROF.to_parquet(PROC / "sber_vs_profile_corr.parquet", index=False)
print(f"\nSberIndex × профили: {len(CORR_PROF)} корреляций")

if len(CORR_PROF) and CORR_PROF["pearson_r"].notna().any():
    top_prof = CORR_PROF.dropna(subset=["pearson_r"]).copy()
    top_prof["abs_r"] = top_prof["pearson_r"].abs()
    top_prof = top_prof.sort_values("abs_r", ascending=False).head(10)
    print("топ-10 (по |r|):")
    print(top_prof[["indicator", "cluster", "feature", "n_years",
                     "pearson_r"]].round(3).to_string(index=False))
else:
    print("топ-10 (по |r|): нет валидных корреляций")

# --- 10.C.6. Медианный сплит SberIndex: MWU профилей ------------------
split_rows = []
for ind in SBER_INDICATORS:
    s = SBER_Y[["year", ind]].dropna(subset=[ind])
    if len(s) < 4:
        continue
    med = s[ind].median()
    high_years = s.loc[s[ind] >= med, "year"].tolist()
    low_years  = s.loc[s[ind] <  med, "year"].tolist()
    if not high_years or not low_years:
        continue

    for f in FEATS_USE:
        # L1
        for cl in sorted(LAB["cluster"].unique()):
            x_hi = LAB.loc[(LAB["year"].isin(high_years)) &
                            (LAB["cluster"] == cl), f].dropna().values
            x_lo = LAB.loc[(LAB["year"].isin(low_years)) &
                            (LAB["cluster"] == cl), f].dropna().values
            if len(x_hi) < 10 or len(x_lo) < 10:
                continue
            try:
                U, p = stats.mannwhitneyu(x_hi, x_lo, alternative="two-sided")
            except Exception:
                continue
            split_rows.append({
                "indicator": ind.replace("sber_", ""),
                "level":     "L1",
                "cluster":   f"L1_K{int(cl)}",
                "feature":   f,
                "n_hi":      int(len(x_hi)),
                "n_lo":      int(len(x_lo)),
                "median_hi": float(np.median(x_hi)),
                "median_lo": float(np.median(x_lo)),
                "delta":     float(np.median(x_hi) - np.median(x_lo)),
                "U":         float(U),
                "p":         float(p),
            })
        # L2
        for k in range(K):
            sub_k = L2_VAL_FEATS[L2_VAL_FEATS[PARENT_COL] == k]
            for s_k in range(L2_SUBK[k]):
                x_hi = sub_k.loc[(sub_k["year"].isin(high_years)) &
                                  (sub_k[SUB_COL] == s_k), f].dropna().values
                x_lo = sub_k.loc[(sub_k["year"].isin(low_years)) &
                                  (sub_k[SUB_COL] == s_k), f].dropna().values
                if len(x_hi) < 10 or len(x_lo) < 10:
                    continue
                try:
                    U, p = stats.mannwhitneyu(x_hi, x_lo,
                                                alternative="two-sided")
                except Exception:
                    continue
                split_rows.append({
                    "indicator": ind.replace("sber_", ""),
                    "level":     "L2",
                    "cluster":   f"K{k}.S{s_k}",
                    "feature":   f,
                    "n_hi":      int(len(x_hi)),
                    "n_lo":      int(len(x_lo)),
                    "median_hi": float(np.median(x_hi)),
                    "median_lo": float(np.median(x_lo)),
                    "delta":     float(np.median(x_hi) - np.median(x_lo)),
                    "U":         float(U),
                    "p":         float(p),
                })
SPLIT = pd.DataFrame(split_rows)
if len(SPLIT):
    SPLIT["p_fdr"] = bh_fdr(SPLIT["p"].to_numpy())
    SPLIT["significant_fdr"] = SPLIT["p_fdr"] < 0.05
SPLIT.to_parquet(PROC / "sber_median_split_mwu.parquet", index=False)

n_sig_split = int(SPLIT["significant_fdr"].sum()) if len(SPLIT) else 0
print(f"\nSberIndex median split (MWU): {len(SPLIT)} тестов, "
      f"значимых после FDR: {n_sig_split}")
if n_sig_split > 0:
    top_split = (SPLIT[SPLIT["significant_fdr"]]
                 .assign(abs_delta=lambda d: d["delta"].abs())
                 .sort_values("abs_delta", ascending=False).head(15))
    print("\nтоп-15 значимых сдвигов профиля (high SberIndex vs low):")
    print(top_split[["indicator", "level", "cluster", "feature",
                      "median_hi", "median_lo", "delta", "p_fdr"]]
          .round(3).to_string(index=False))
else:
    print("нет значимых сдвигов после FDR")

# --- 10.C.7. Обновляем sber_usage.json --------------------------------
sber_use = json.loads((PROC / "sber_usage.json").read_text(encoding="utf-8"))
sber_use["sber_used"].extend([
    "SberIndex × prevalence L1/L2 (Spearman/Pearson)",
    "SberIndex × профили кластеров (Pearson)",
    "SberIndex median split MWU (high vs low years)",
])
sber_use["sber_used"] = list(dict.fromkeys(sber_use["sber_used"]))
json.dump(sber_use, open(PROC / "sber_usage.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=1)

print("\n✅ 10.C завершён: 5 артефактов сохранено")


# ============================================================================
# ВИЗУАЛИЗАЦИИ ВНЕШНЕЙ ВАЛИДАЦИИ
# ============================================================================
tick("визуализация внешней валидации")

# --- L1: рост зарплат кластеров vs отрасли SberIndex ------------------
mp_l1 = PROC / "external_sber_industry_match.parquet"
if mp_l1.exists() and WAGE_SIG_L1 is not None:
    M = pd.read_parquet(mp_l1)
    inds = sorted(M["industry"].unique())
    xp = np.arange(len(inds))
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(xp, [WAGE_SIG_L1.loc[i, "sber_growth"] for i in inds],
            "k--o", ms=4, label="SberIndex: отрасли (нац. уровень)")
    cmapK = plt.get_cmap("tab10", K)
    for c in range(K):
        s = M[M["cluster"] == c].set_index("industry").reindex(inds)
        ax.plot(xp, s["cluster_growth"], "o", ms=7,
                color=cmapK(c), label=f"L1 K{c} (Росстат, МО)")
    ax.set_xticks(xp)
    ax.set_xticklabels(inds, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel(f"Рост номинальной зарплаты {y_a}→{y_b}, раз")
    ax.set_title("L1: рост зарплат кластеров vs отрасли СберИндекса")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(VIZ / "07_sber_validation.png", dpi=110)
    plt.close()
    print(f"✅ viz/07_sber_validation.png")

# --- L2: аналогичный график по подкластерам ---------------------------
mp_l2 = PROC / "external_sber_industry_match_l2.parquet"
if mp_l2.exists() and WAGE_SIG_L1 is not None:
    M = pd.read_parquet(mp_l2)
    inds = sorted(M["industry"].unique())
    xp = np.arange(len(inds))
    n_panels = sum(1 for k in range(K) if L2_SUBK[k] >= 2)
    if n_panels:
        fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 5))
        axes = np.atleast_1d(axes).ravel()
        panel_idx = 0
        for k in range(K):
            if L2_SUBK[k] < 2:
                continue
            ax = axes[panel_idx]; panel_idx += 1
            ax.plot(xp, [WAGE_SIG_L1.loc[i, "sber_growth"] for i in inds],
                    "k--o", ms=4, label="SberIndex")
            cmap_sub = plt.get_cmap("tab10", L2_SUBK[k])
            for s in range(L2_SUBK[k]):
                sub = M[(M["parent_k"] == k) & (M["sub_k"] == s)
                        ].set_index("industry").reindex(inds)
                ax.plot(xp, sub["subcluster_growth"], "o", ms=7,
                        color=cmap_sub(s), label=f"K{k}.S{s}")
            ax.set_xticks(xp)
            ax.set_xticklabels(inds, rotation=45, ha="right", fontsize=7)
            ax.set_ylabel(f"Рост зарплат {y_a}→{y_b}, раз")
            ax.set_title(f"K{k} → {L2_SUBK[k]} подкластеров")
            ax.legend(fontsize=7); ax.grid(alpha=0.3)
        plt.suptitle("L2: рост зарплат подкластеров vs отрасли СберИндекса",
                     fontweight="bold", fontsize=13)
        plt.tight_layout()
        plt.savefig(VIZ / "07_sber_validation_l2.png", dpi=110)
        plt.close()
        print(f"✅ viz/07_sber_validation_l2.png")

# --- L1: структура кластеров (тип МО + региональная концентрация) -----
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
CT_L1.plot(kind="barh", stacked=True, ax=axes[0], colormap="tab20")
axes[0].set_title("L1: состав кластеров по типу МО")
axes[0].set_xlabel("доля")
axes[0].legend(fontsize=7, loc="lower right")

cmapK = plt.get_cmap("tab10", K)
conc_l1["top3_region_share"].plot(kind="bar", ax=axes[1],
                                    color=[cmapK(c) for c in conc_l1.index])
axes[1].set_title(f"L1: доля топ-3 регионов в кластере ({y_b})")
axes[1].set_ylim(0, 1); axes[1].axhline(0.5, ls="--", color="r", lw=1)
plt.tight_layout()
plt.savefig(VIZ / "08_validation_structure.png", dpi=110)
plt.close()
print(f"✅ viz/08_validation_structure.png")

# --- L2: структура подкластеров (тип МО) ------------------------------
if len(CHI2_L2):
    n_panels = sum(1 for k in range(K) if L2_SUBK[k] >= 2)
    if n_panels:
        fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 4.5))
        axes = np.atleast_1d(axes).ravel()
        panel_idx = 0
        for k in range(K):
            if L2_SUBK[k] < 2:
                continue
            ax = axes[panel_idx]; panel_idx += 1
            sub_k = L2_VAL[L2_VAL[PARENT_COL] == k]
            ct_k = pd.crosstab(sub_k[SUB_COL],
                                sub_k["mun_type"].fillna("n/a"),
                                normalize="index")
            ct_k.plot(kind="barh", stacked=True, ax=ax, colormap="tab20")
            ax.set_title(f"K{k} → {L2_SUBK[k]} подкластеров")
            ax.set_xlabel("доля"); ax.set_ylabel("")
            ax.legend(fontsize=6, loc="lower right")
        plt.suptitle("L2: состав подкластеров по типу МО",
                     fontweight="bold", fontsize=13)
        plt.tight_layout()
        plt.savefig(VIZ / "08_validation_structure_l2.png", dpi=110)
        plt.close()
        print(f"✅ viz/08_validation_structure_l2.png")

# --- L1/L2 × SberIndex: heatmap корреляций + timeline -----------------
# Диагностические сообщения: почему картинка может быть пустой
print("\n--- 18_sber_correlation.png: диагностика ---")
print(f"  CORR_PREV: {len(CORR_PREV) if 'CORR_PREV' in dir() else 'нет'}")
if 'CORR_PREV' in dir() and len(CORR_PREV):
    n_valid = int(CORR_PREV["spearman_r"].notna().sum())
    print(f"  валидных Spearman: {n_valid}/{len(CORR_PREV)}")
    print(f"  диапазон: "
          f"[{CORR_PREV['spearman_r'].min():.3f}, "
          f"{CORR_PREV['spearman_r'].max():.3f}]"
          if n_valid else "  все значения NaN")

# Строим фигуру даже если некоторые подпанели пусты — общий PNG всегда
# будет создан, с fallback-сообщениями там, где не хватает данных.
fig = plt.figure(figsize=(16, 9), dpi=110)
gs = fig.add_gridspec(2, 2, height_ratios=[1.4, 1], hspace=0.35,
                       wspace=0.25)

# --- (а) Heatmap корреляций prevalence × SberIndex --------------------
ax0 = fig.add_subplot(gs[0, :])

piv = None
if 'CORR_PREV' in dir() and len(CORR_PREV):
    piv = CORR_PREV.pivot_table(index="indicator", columns="cluster",
                                  values="spearman_r")
    col_order = []
    for c in ["L1_K0", "L1_K1"]:
        if c in piv.columns:
            col_order.append(c)
    for k in range(K):
        for s in range(L2_SUBK[k]):
            c = f"K{k}.S{s}"
            if c in piv.columns:
                col_order.append(c)
    piv = piv.reindex(columns=col_order)

if piv is not None and piv.notna().any().any():
    # маскируем NaN-значения серым
    data = piv.values
    masked = np.ma.masked_invalid(data)
    cmap_use = plt.get_cmap("RdBu_r").copy()
    cmap_use.set_bad(color="#d1d5db")     # серый для NaN
    im = ax0.imshow(masked, cmap=cmap_use, vmin=-1, vmax=1, aspect="auto")
    ax0.set_xticks(range(len(piv.columns)))
    ax0.set_xticklabels(piv.columns, rotation=30, ha="right", fontsize=10)
    ax0.set_yticks(range(len(piv.index)))
    ax0.set_yticklabels(piv.index, fontsize=10)
    for i in range(len(piv.index)):
        for j in range(len(piv.columns)):
            v = data[i, j]
            if np.isfinite(v):
                ax0.text(j, i, f"{v:+.2f}", ha="center", va="center",
                          fontsize=9,
                          color="white" if abs(v) > 0.7 else "black")
            else:
                ax0.text(j, i, "n/a", ha="center", va="center",
                          fontsize=8, color="#6b7280")
    plt.colorbar(im, ax=ax0, label="Spearman r")
    ax0.set_title(
        f"SberIndex × prevalence (годовые срезы {y_a}–{y_b})\n"
        f"Серые ячейки «n/a» — недостаточно лет для корреляции",
        fontsize=11, fontweight="bold")
else:
    ax0.axis("off")
    ax0.text(0.5, 0.5,
             "Корреляции SberIndex × prevalence недоступны\n"
             "(все значения NaN — перехлёст SberIndex и кластеризации <4 лет)",
             ha="center", va="center", fontsize=13, color="#6b7280")

# --- (б) Timeline: SberIndex (wage) + prevalence L1 -------------------
ax1 = fig.add_subplot(gs[1, 0])

if 'SBER_Y' in dir() and "sber_wage" in SBER_Y.columns:
    s_w = SBER_Y[["year", "sber_wage"]].dropna()
    if len(s_w):
        ax1.plot(s_w["year"], s_w["sber_wage"], "o-", color="black",
                  label="SberIndex: зарплаты", lw=2)
ax1.set_ylabel("SberIndex: медианная зарплата", color="black")
ax1.set_xlabel("Год"); ax1.grid(alpha=0.3)

ax1b = ax1.twinx()
if 'prev_l1_year' in dir() and len(prev_l1_year):
    cmap_l1_plot = plt.get_cmap("tab10", K)
    for c in prev_l1_year.columns:
        # имя колонки вида "L1_K0" — берём цифру после K
        k_idx = int(c.rsplit("K", 1)[-1])
        ax1b.plot(prev_l1_year.index, prev_l1_year[c].values, "s--",
                   color=cmap_l1_plot(k_idx),
                   lw=1.6, alpha=0.85, label=f"{c} prevalence")
ax1b.set_ylabel("Prevalence L1, %")

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax1b.get_legend_handles_labels()
if h1 + h2:
    ax1.legend(h1 + h2, l1 + l2, fontsize=8, loc="best")
ax1.set_title("Timeline: SberIndex (зарплаты) vs L1 prevalence",
               fontsize=11, fontweight="bold")

# --- (в) Timeline: SberIndex (activity) + prevalence L2 ---------------
ax2 = fig.add_subplot(gs[1, 1])

if 'SBER_Y' in dir() and "sber_activity" in SBER_Y.columns:
    s_a = SBER_Y[["year", "sber_activity"]].dropna()
    if len(s_a):
        ax2.plot(s_a["year"], s_a["sber_activity"], "o-", color="black",
                  label="SberIndex: активность", lw=2)
ax2.set_ylabel("SberIndex: потреб. активность", color="black")
ax2.set_xlabel("Год"); ax2.grid(alpha=0.3)

ax2b = ax2.twinx()
if 'prev_l2_year' in dir() and len(prev_l2_year):
    cmap_l2_plot = plt.get_cmap("tab10", sum(L2_SUBK.values()))
    cidx = 0
    for k in range(K):
        for s in range(L2_SUBK[k]):
            cl = f"K{k}.S{s}"
            if cl in prev_l2_year.columns:
                ax2b.plot(prev_l2_year.index, prev_l2_year[cl].values, "s--",
                           color=cmap_l2_plot(cidx), lw=1.4, alpha=0.85,
                           label=cl)
            cidx += 1
ax2b.set_ylabel("Prevalence L2, %")

h1, l1 = ax2.get_legend_handles_labels()
h2, l2 = ax2b.get_legend_handles_labels()
if h1 + h2:
    ax2.legend(h1 + h2, l1 + l2, fontsize=8, loc="best")
ax2.set_title("Timeline: SberIndex (активность) vs L2 prevalence",
               fontsize=11, fontweight="bold")

plt.savefig(VIZ / "18_sber_correlation.png", dpi=110,
             bbox_inches="tight")
plt.close()

# Проверка размера файла — не пустой ли PNG
_p18 = VIZ / "18_sber_correlation.png"
if _p18.exists() and _p18.stat().st_size > 5000:
    print(f"✅ viz/18_sber_correlation.png  "
          f"({_p18.stat().st_size/1024:.1f} KB)")
else:
    print(f"⚠️  viz/18_sber_correlation.png создан, но слишком мал "
          f"({_p18.stat().st_size if _p18.exists() else 0} bytes)")

# ============================================================================
# ИТОГ
# ============================================================================
print("\n" + "=" * 70)
print("ИТОГ ЯЧЕЙКИ 10")
print("=" * 70)

print(f"\nL1 внешняя валидация:")
print(f"  SberIndex использован: {SBER_USED if SBER_USED else '—'}")
print(f"  L1 vs тип МО: chi²={chi2:.1f}, p={pchi:.3e}")
print(f"  Региональная концентрация топ-3:")
for c in conc_l1.index:
    r = conc_l1.loc[c]
    print(f"    K{c}: top-3 = {r.top3_region_share:.3f}, "
          f"top-1 = {r.top1_region_share:.3f} ({r.top_region})")
if "JKCL_L1_DF" in dir() and len(JKCL_L1_DF):
    print(f"  Jackknife L1 (holds_without_top):")
    for _, r in JKCL_L1_DF.iterrows():
        print(f"    K{int(r.cluster)}: {r.holds_without_top:.3f} "
              f"(без {r.top_region})")
print(f"  Тест без Москвы/МО: ARI={ari_nm_l1:.4f}")

print(f"\nL2 внешняя валидация:")
if len(CHI2_L2):
    print(f"  chi² L2 vs тип МО:")
    for _, r in CHI2_L2.iterrows():
        print(f"    K{int(r.parent_k)}: chi²={r.chi2:.1f}, p={r.p:.3e}")
if len(JKCL_L2_DF):
    print(f"  Jackknife L2 (holds_without_top):")
    for _, r in JKCL_L2_DF.iterrows():
        print(f"    {r.cluster_l2}: {r.holds_without_top:.3f} "
              f"(без {r.top_region})")
print(f"  ARI (без Москвы/МО):")
for k, a in ari_nm_l2.items():
    print(f"    K{k}: {a:.4f}")

print(f"\nSberIndex × L1/L2 (часть 10.C):")
print(f"  квартальные ряды:       {SBER_Q_ALL.shape}")
print(f"  годовые ряды:           {SBER_Y.shape}")
print(f"  корреляций prevalence:  {len(CORR_PREV)}")
print(f"  корреляций профилей:    {len(CORR_PROF)}")
print(f"  median split MWU:       {len(SPLIT)} тестов, "
      f"значимых {n_sig_split}")

print()
print("Артефакты:")
for f in ["external_wage_growth_by_cluster.parquet",
          "external_wage_growth_by_subcluster.parquet",
          "external_sber_industry_match.parquet",
          "external_sber_industry_match_l2.parquet",
          "external_cluster_by_muntype.parquet",
          "external_cluster_by_muntype_l2.parquet",
          "external_cluster_region_concentration.parquet",
          "external_cluster_region_concentration_l2.parquet",
          "external_cluster_jackknife.parquet",
          "external_cluster_jackknife_l2.parquet",
          "external_without_moscow.json",
          "external_without_moscow_l2.json",
          "sber_consumer_activity.parquet",
          "sber_key_rate.parquet",
          "sber_quarterly.parquet",
          "sber_yearly.parquet",
          "prevalence_yearly.parquet",
          "sber_vs_prevalence_corr.parquet",
          "sber_vs_profile_corr.parquet",
          "sber_median_split_mwu.parquet",
          "sber_usage.json"]:
    p = PROC / f
    if p.exists():
        print(f"  ✅ {f}")
for f in sorted(VIZ.glob("07_*.png")) + sorted(VIZ.glob("08_*.png")) + \
         sorted(VIZ.glob("18_*.png")):
    print(f"  ✅ viz/{f.name}")

tick("внешняя валидация + SberIndex-корреляции завершены")

L2_LAB: SUB_COL='sub_k', PARENT_COL='parent_k'

10.0. SberIndex: загрузка национальных рядов
найдено файлов SberIndex: ['median-wages_ru_1790089505830.parquet', 'potrebitelskaya-aktivnost-po-kategoriyam-tovarov-v-razreze-vozrastov_ru_1789719227657.parquet', 'prognoz-nedelnykh-tempov-inflyacii_ru_1789627987403.parquet', 'real-key-interest-rate_ru_1789628930646.parquet']
загружено рядов: ['wage', 'activity', 'inflation', 'keyrate']

ЧАСТЬ A. ВНЕШНЯЯ ВАЛИДАЦИЯ L1

=== SBERINDEX: РОСТ ЗАРПЛАТ ПО ОТРАСЛЯМ 2020→2024 ===
                                                   sber_growth      level
activity                                                                 
Обрабатывающие производства                              1.848  58736.250
Строительство                                            1.833  64513.750
Добыча полезных ископаемых                               1.810  88563.167
Сельское хозяйство                                       1.809  48217.333
Торговля                            

In [16]:
# ============================================================================
# ============ ЯЧЕЙКА 13 ============
# ============================================================================
# ============================================================================
# ЯЧЕЙКА 13. ICVI POST-HOC НА ФИНАЛЬНЫХ КЛАСТЕРАХ (с поправкой на дисбаланс)
# ============================================================================
# Зачем:
#   Критерий оценивания требует применения ICVI: SW, CH, DB, S_DbW, AVI, AVU, MQ.
#   В новом pipeline выбор K/subK делался по устойчивости, а не по ICVI.
#   Эта ячейка закрывает пробел + учитывает специфику L1 (imbalance 57/1965):
#     • MQ считается по ПРАВИЛЬНОЙ формуле Ньюмана (через d_k², а не наивно);
#     • AVI/MQ сравниваются с shuffle-baseline (excess = real − baseline);
#     • пороги адаптированы под imbalance;
#     • добавлена сбалансированная подвыборка (57+57 МО) для контроля.
#
# Вход:
#   dz       — с колонками kan_e*, cluster_l1, cluster_l2_aligned,
#              kan_l2_k*_e*, sub_k_aligned
#   K, L2_SUBK, yrs, PROC, VIZ, RNG
#
# Выход:
#   • icvi_l1_panel.parquet           — ICVI L1 по годам
#   • icvi_l2_panel.parquet           — ICVI L2 по годам и родителям
#   • icvi_summary.parquet            — среднее по годам (L1, L2, L1_balanced)
#   • icvi_l1_baseline.parquet        — L1 shuffle-baseline на 2024 г.
#   • icvi_l1_balanced.parquet        — L1 на сбалансированной подвыборке
#   • icvi_l2_parent{k}_baseline.parquet — L2 baseline по родителям
#   • viz/17_icvi_panel.png           — линии ICVI по годам + таблица
# ============================================================================
tick("ICVI post-hoc (L1 + L2, с учётом дисбаланса)")

# --- Проверки доступности -------------------------------------------------
for _var in ("dz", "K", "L2_SUBK", "yrs", "PROC", "VIZ", "RNG"):
    assert _var in globals(), (
        f"переменная {_var} не найдена — выполните ячейки 0–12 перед 13"
    )

# --- Определяем колонки ---------------------------------------------------
KAN_L1_COLS = [c for c in dz.columns if c.startswith("kan_e")]
KAN_L2_COLS = {k: [c for c in dz.columns if c.startswith(f"kan_l2_k{k}_e")]
               for k in range(K)}

assert KAN_L1_COLS, "нет KAN-L1 эмбеддингов kan_e* в dz"
print(f"KAN-L1: {KAN_L1_COLS}")
for k in range(K):
    print(f"KAN-L2[{k}]: {KAN_L2_COLS[k]}")

# Имя колонки с parent_k и sub_k L2
if "cluster_l1" in dz.columns:
    L1_COL = "cluster_l1"
elif "cluster" in dz.columns:
    L1_COL = "cluster"
else:
    raise KeyError("нет cluster_l1/cluster в dz")

SUB_COL = next((c for c in ("sub_k_aligned", "sub_k")
                if c in dz.columns), None)
assert SUB_COL is not None, f"нет sub_k_aligned/sub_k в dz"

# --- Графовое построение для AVU/MQ: mutual kNN ---------------------------
from sklearn.neighbors import NearestNeighbors

def mutual_knn_edges(X, k=10):
    """Mutual kNN граф. Возвращает (src, dst, w) — плоские массивы."""
    n = len(X)
    if n < k + 2:
        return np.array([], dtype=int), np.array([], dtype=int), np.array([])
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X)
    dist, idx = nn.kneighbors(X)
    A = np.zeros((n, n), dtype=bool)
    for i in range(n):
        A[i, idx[i, 1:]] = True
    A = A & A.T
    src, dst = np.where(np.triu(A, 1))
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
    w = np.einsum("ij,ij->i", Xn[src], Xn[dst]).astype(np.float64)
    return src, dst, w


# ============================================================================
# ICVI: реализация 7 метрик
# ============================================================================
def icvi_sw_ch_db(X, labels):
    K_l = len(np.unique(labels))
    if K_l < 2 or K_l >= len(X):
        return np.nan, np.nan, np.nan
    try:    sw = float(silhouette_score(X, labels))
    except Exception: sw = np.nan
    try:    ch = float(calinski_harabasz_score(X, labels))
    except Exception: ch = np.nan
    try:    db = float(davies_bouldin_score(X, labels))
    except Exception: db = np.nan
    return sw, ch, db


def icvi_s_dbw(X, labels):
    """S_Dbw = SCAT + DENS_bw (Halkidi, Vazirgiannis, 2001)."""
    K_l = len(np.unique(labels))
    if K_l < 2:
        return np.nan
    centroids = np.array([X[labels == k].mean(0) for k in range(K_l)])
    sigma_X = float(np.linalg.norm(X.std(0)))
    if sigma_X < 1e-12:
        return np.nan
    sigmas_V = np.array([np.linalg.norm(X[labels == k].std(0))
                         for k in range(K_l)])
    scat = sigmas_V.mean() / sigma_X
    stdev = float(np.sqrt((sigmas_V ** 2).sum() / K_l))
    if stdev < 1e-12:
        return float(scat)
    dens_bw_sum = 0.0
    for i in range(K_l):
        for j in range(K_l):
            if i == j:
                continue
            u = (centroids[i] + centroids[j]) / 2.0
            dens_u = int(np.sum(np.linalg.norm(X - u, axis=1) <= stdev))
            dens_i = int(np.sum(np.linalg.norm(X[labels == i] - centroids[i],
                                               axis=1) <= stdev))
            dens_j = int(np.sum(np.linalg.norm(X[labels == j] - centroids[j],
                                               axis=1) <= stdev))
            dens_bw_sum += dens_u / max(dens_i, dens_j, 1)
    dens_bw = dens_bw_sum / (K_l * (K_l - 1))
    return float(scat + dens_bw)


def icvi_avi(X, labels):
    """AVI = BSS / TSS — доля объяснённой дисперсии ∈ [0, 1]."""
    K_l = len(np.unique(labels))
    if K_l < 2:
        return np.nan
    Xm = X.mean(0)
    TSS = ((X - Xm) ** 2).sum()
    WSS = 0.0
    for k in range(K_l):
        Xk = X[labels == k]
        if len(Xk) == 0:
            continue
        WSS += ((Xk - Xk.mean(0)) ** 2).sum()
    return float((TSS - WSS) / (TSS + 1e-12))


def icvi_avu(src, dst, w, labels):
    """AVU — доля веса внутрикластерных рёбер ∈ [0, 1]."""
    if len(src) == 0:
        return np.nan
    same = (labels[src] == labels[dst])
    if not same.any():
        return 0.0
    return float(w[same].sum() / (w.sum() + 1e-12))


def icvi_mq(src, dst, w, labels, K_l):
    """Newman modularity (правильная нормализация).

    Q = W_int / W − Σ_k d_k² / (4W²)
      = 2·W_int / m₂ − Σ_k d_k² / m₂²

    где W = Σ_e w_e (общий вес рёбер), m₂ = deg.sum() = 2W,
    d_k = Σ_{i∈c_k} k_i — суммарная взвешенная степень узлов кластера k.
    """
    if len(src) == 0:
        return np.nan

    # ← ключевое исправление 1: n = len(labels), а не max(src,dst)+1.
    # Это гарантирует, что deg вмещает все узлы, включая изолированные.
    n = len(labels)
    deg = np.zeros(n, dtype=np.float64)
    np.add.at(deg, src, w)
    np.add.at(deg, dst, w)

    m2 = deg.sum() + 1e-12                     # = 2W
    same = (labels[src] == labels[dst])
    internal = w[same].sum()                   # = W_int

    d_k = np.array([deg[labels == k].sum() for k in range(K_l)],
                   dtype=np.float64)

    # ← ключевое исправление 2: правильная константа штрафа.
    # Было: (internal - Σd²/m₂) / (m₂/2)  → штраф 2·Σd²/m₂²
    # Стало: 2·internal/m₂ − Σd²/m₂²
    Q = 2.0 * internal / m2 - (d_k ** 2).sum() / (m2 ** 2)
    return float(Q)

def compute_icvi_block(X, labels, k_graph=10):
    """7 метрик одним вызовом. Возвращает dict."""
    sw, ch, db = icvi_sw_ch_db(X, labels)
    sdbw = icvi_s_dbw(X, labels)
    avi  = icvi_avi(X, labels)
    src, dst, w = mutual_knn_edges(X, k=k_graph)
    avu  = icvi_avu(src, dst, w, labels)
    mq   = icvi_mq(src, dst, w, labels, len(np.unique(labels)))
    return {"SW": sw, "CH": ch, "DB": db, "S_Dbw": sdbw,
            "AVI": avi, "AVU": avu, "MQ": mq,
            "n_edges": int(len(src))}


def compute_icvi_with_baseline(X, labels, k_graph=10,
                                 n_shuffles=20, seed=RNG):
    """ICVI на реальных + на перемешанных метках (сохраняя размеры).

    Excess = real − baseline. Excess > 0 означает, что структура не
    объясняется одним дисбалансом размеров кластеров.
    """
    rng = np.random.RandomState(seed)
    real = compute_icvi_block(X, labels, k_graph=k_graph)
    metrics_keys = ["SW", "CH", "DB", "S_Dbw", "AVI", "AVU", "MQ"]

    base_metrics = {m: [] for m in metrics_keys}
    for _ in range(n_shuffles):
        lab_sh = rng.permutation(labels)
        try:
            b = compute_icvi_block(X, lab_sh, k_graph=k_graph)
            for m in metrics_keys:
                v = b.get(m, np.nan)
                if np.isfinite(v):
                    base_metrics[m].append(v)
        except Exception:
            continue

    out = dict(real)
    for m in metrics_keys:
        baseline = float(np.mean(base_metrics[m])) if base_metrics[m] else np.nan
        out[f"{m}_baseline"] = baseline
        out[f"{m}_excess"]   = (real.get(m, np.nan) - baseline
                                 if np.isfinite(real.get(m, np.nan))
                                 and np.isfinite(baseline) else np.nan)
    return out


# ============================================================================
# L1: ICVI по годам
# ============================================================================
print("\n" + "=" * 70)
print("ICVI L1 (KAN-L1 эмбеддинги, mutual kNN)")
print("=" * 70)

rows_l1 = []
for y in yrs:
    sub = dz.loc[dz["year"] == y].reset_index(drop=True)
    sub = sub.dropna(subset=KAN_L1_COLS + [L1_COL])
    if len(sub) < 20:
        continue
    X = sub[KAN_L1_COLS].values.astype(np.float32)
    lab = sub[L1_COL].values.astype(int)
    m = compute_icvi_block(X, lab, k_graph=10)
    row = {"year": int(y), "K": int(K), "n_obs": int(len(sub)), **m}
    rows_l1.append(row)
    print(f"  {y}: n={len(sub):>5d}  "
          f"SW={m['SW']:+.4f}  CH={m['CH']:>8.1f}  DB={m['DB']:.4f}  "
          f"S_Dbw={m['S_Dbw']:.4f}  AVI={m['AVI']:.4f}  "
          f"AVU={m['AVU']:.4f}  MQ={m['MQ']:+.4f}")

ICVI_L1 = pd.DataFrame(rows_l1)
ICVI_L1.to_parquet(PROC / "icvi_l1_panel.parquet", index=False)


# ============================================================================
# L2: ICVI по годам, ВНУТРИ каждого родителя
# ============================================================================
print("\n" + "=" * 70)
print("ICVI L2 (KAN-L2 эмбеддинги, mutual kNN, per parent)")
print("=" * 70)

rows_l2 = []
for k in range(K):
    subK = L2_SUBK[k]
    cols_l2 = KAN_L2_COLS[k]
    if subK < 2 or not cols_l2:
        print(f"\n  K{k}: subK={subK} — ICVI не считается")
        continue
    print(f"\n  --- K{k} (subK={subK}), пространство: {cols_l2} ---")
    for y in yrs:
        sub = dz.loc[(dz["year"] == y) & (dz[L1_COL] == k)].reset_index(drop=True)
        sub = sub.dropna(subset=cols_l2 + [SUB_COL])
        if len(sub) < 20:
            continue
        if sub[SUB_COL].nunique() < 2:
            continue
        X = sub[cols_l2].values.astype(np.float32)
        lab = sub[SUB_COL].values.astype(int)
        m = compute_icvi_block(X, lab, k_graph=10)
        rows_l2.append({
            "parent_k": int(k),
            "year":     int(y),
            "subK":     int(subK),
            "n_obs":    int(len(sub)),
            **m,
        })
        print(f"    {y}: n={len(sub):>5d}  "
              f"SW={m['SW']:+.4f}  CH={m['CH']:>8.1f}  DB={m['DB']:.4f}  "
              f"S_Dbw={m['S_Dbw']:.4f}  AVI={m['AVI']:.4f}  "
              f"AVU={m['AVU']:.4f}  MQ={m['MQ']:+.4f}")

ICVI_L2 = pd.DataFrame(rows_l2)
ICVI_L2.to_parquet(PROC / "icvi_l2_panel.parquet", index=False)


# ============================================================================
# Сводка: среднее по годам
# ============================================================================
print("\n" + "=" * 70)
print("СВОДКА ICVI (среднее по годам)")
print("=" * 70)

ICVI_METRICS = ["SW", "CH", "DB", "S_Dbw", "AVI", "AVU", "MQ"]

l1_summary = None
if len(ICVI_L1):
    l1_summary = ICVI_L1[ICVI_METRICS].mean().to_dict()
    l1_summary["n_years"]  = int(len(ICVI_L1))
    l1_summary["level"]    = "L1"
    l1_summary["K"]        = int(K)
    l1_summary["subK"]     = None
    l1_summary["parent_k"] = None

l2_summary_rows = []
if len(ICVI_L2):
    for k in range(K):
        sub = ICVI_L2[ICVI_L2["parent_k"] == k]
        if len(sub) == 0:
            continue
        row = sub[ICVI_METRICS].mean().to_dict()
        row["n_years"]  = int(len(sub))
        row["level"]    = "L2"
        row["K"]        = None
        row["subK"]     = int(L2_SUBK[k])
        row["parent_k"] = int(k)
        l2_summary_rows.append(row)

# --- Первичная таблица (без balanced) ---------------------------------
summary_rows = []
if l1_summary is not None:
    summary_rows.append(l1_summary)
summary_rows.extend(l2_summary_rows)
ICVI_SUM = pd.DataFrame(summary_rows)
ICVI_SUM = ICVI_SUM[["level", "parent_k", "K", "subK",
                     "n_years"] + ICVI_METRICS]

print("\n=== ICVI SUMMARY ===")
print(ICVI_SUM.round(4).to_string(index=False))


# ============================================================================
# ДИАГНОСТИКА: с учётом дисбаланса классов + shuffle-baseline
# ============================================================================
print("\n" + "=" * 70)
print("ДИАГНОСТИКА (с поправкой на дисбаланс классов)")
print("=" * 70)

def _diag_row(label, m_dict, thresholds):
    """Печатает строку диагностики с baseline/excess."""
    print(f"\n{label}:")
    for key, thr in thresholds.items():
        real = m_dict.get(key, np.nan)
        base = m_dict.get(f"{key}_baseline", np.nan)
        exc  = m_dict.get(f"{key}_excess",   np.nan)
        ok_real = np.isfinite(real) and real >= thr["real"]
        ok_exc  = np.isfinite(exc)  and exc  >= thr.get("excess", -np.inf)
        # Проходит, если real >= порога ИЛИ excess > 0 (не объяснён дисбалансом)
        mark = "✅" if (ok_real or (np.isfinite(exc) and exc > 0)) else "⚠️ "
        base_str = f"{base:+.4f}" if np.isfinite(base) else "  n/a "
        exc_str  = f"{exc:+.4f}"  if np.isfinite(exc)  else "  n/a "
        print(f"  {key:6s} real={real:+.4f}  base={base_str}  "
              f"excess={exc_str}  (порог real={thr['real']})  {mark}")


# --- L1 на 2024 с baseline --------------------------------------------
sub_l1 = dz.loc[dz["year"] == yrs[-1]].dropna(
    subset=KAN_L1_COLS + [L1_COL]).reset_index(drop=True)
X_l1_2024 = sub_l1[KAN_L1_COLS].values.astype(np.float32)
lab_l1_2024 = sub_l1[L1_COL].values.astype(int)
diag_l1 = compute_icvi_with_baseline(X_l1_2024, lab_l1_2024,
                                      k_graph=10, n_shuffles=20)
_diag_row(
    f"L1 (K={K}, n={len(sub_l1)}, imbalance={np.bincount(lab_l1_2024).tolist()})",
    diag_l1,
    thresholds={
        "SW":  {"real": 0.15},
        "AVI": {"real": 0.20, "excess": 0.05},
        "MQ":  {"real": 0.05, "excess": 0.02},
        "AVU": {"real": 0.50, "excess": 0.10},
    },
)
# Сохраняем baseline
pd.DataFrame([{
    "year": int(yrs[-1]),
    "scope": "L1",
    **{k: v for k, v in diag_l1.items()},
}]).to_parquet(PROC / "icvi_l1_baseline.parquet", index=False)

# --- L2 на 2024 с baseline, per parent -------------------------------
for k in range(K):
    if L2_SUBK[k] < 2:
        continue
    cols_l2 = KAN_L2_COLS[k]
    sub_k = dz.loc[(dz["year"] == yrs[-1]) & (dz[L1_COL] == k)
                    ].dropna(subset=cols_l2 + [SUB_COL]).reset_index(drop=True)
    if len(sub_k) < 20:
        continue
    X_k  = sub_k[cols_l2].values.astype(np.float32)
    lab_k = sub_k[SUB_COL].values.astype(int)
    diag_k = compute_icvi_with_baseline(X_k, lab_k,
                                         k_graph=10, n_shuffles=20)
    _diag_row(
        f"L2 K{k} (subK={L2_SUBK[k]}, n={len(sub_k)}, "
        f"imbalance={np.bincount(lab_k).tolist()})",
        diag_k,
        thresholds={
            "SW":  {"real": 0.15},
            "AVI": {"real": 0.15, "excess": 0.05},
            "MQ":  {"real": 0.05, "excess": 0.02},
            "AVU": {"real": 0.50, "excess": 0.10},
        },
    )
    pd.DataFrame([{
        "scope": f"L2_K{k}", "year": int(yrs[-1]), **diag_k,
    }]).to_parquet(PROC / f"icvi_l2_parent{k}_baseline.parquet",
                    index=False)


# --- L1 на сбалансированной подвыборке (57 urban + 57 rural) --------
print("\n" + "-" * 70)
print("L1 на сбалансированной подвыборке (urban vs rural 1:1)")
print("-" * 70)

counts_l1 = np.bincount(lab_l1_2024)
n_min     = int(counts_l1.min())
rng       = np.random.RandomState(RNG)

idx_balanced = []
for k in range(K):
    idx_k = np.where(lab_l1_2024 == k)[0]
    idx_balanced.extend(rng.choice(idx_k, n_min, replace=False).tolist())
idx_balanced = np.array(idx_balanced)

X_bal    = X_l1_2024[idx_balanced]
lab_bal  = lab_l1_2024[idx_balanced]

diag_bal = compute_icvi_with_baseline(X_bal, lab_bal, k_graph=10,
                                        n_shuffles=20)
_diag_row(
    f"L1-balanced (n={len(X_bal)}, sizes={np.bincount(lab_bal).tolist()})",
    diag_bal,
    thresholds={
        "SW":  {"real": 0.15},
        "AVI": {"real": 0.20, "excess": 0.05},
        "MQ":  {"real": 0.05, "excess": 0.02},
        "AVU": {"real": 0.50, "excess": 0.10},
    },
)
pd.DataFrame([{
    "scope": "L1_balanced", "year": int(yrs[-1]), **diag_bal,
}]).to_parquet(PROC / "icvi_l1_balanced.parquet", index=False)

# --- Добавляем L1_balanced в общую сводку ----------------------------
ICVI_SUM = pd.concat([
    ICVI_SUM,
    pd.DataFrame([{
        "level": "L1_balanced", "parent_k": None, "K": int(K),
        "subK": None, "n_years": 1,
        **{m: diag_bal[m] for m in ICVI_METRICS},
    }]),
], ignore_index=True)
ICVI_SUM.to_parquet(PROC / "icvi_summary.parquet", index=False)


# ============================================================================
# Визуализация: линии ICVI по годам + сводная таблица
# ============================================================================
fig, axes = plt.subplots(2, 4, figsize=(18, 8), dpi=110)
axes = axes.ravel()

for idx, metric in enumerate(ICVI_METRICS):
    ax = axes[idx]
    # --- L1 ---
    if len(ICVI_L1):
        ax.plot(ICVI_L1["year"], ICVI_L1[metric], "o-",
                color="black", lw=2, ms=7, label=f"L1 (K={K})")
    # --- L2 ---
    cmap_k = plt.get_cmap("tab10", max(K, 2))
    for k in range(K):
        sub = ICVI_L2[ICVI_L2["parent_k"] == k] if len(ICVI_L2) else None
        if sub is None or len(sub) == 0:
            continue
        ax.plot(sub["year"], sub[metric], "s--",
                color=cmap_k(k), lw=1.6, ms=5,
                label=f"L2 K{k} (subK={L2_SUBK[k]})")
    ax.set_title(metric, fontweight="bold")
    ax.set_xlabel("Год"); ax.grid(alpha=0.3)
    if metric in ("DB", "S_Dbw"):
        ax.set_ylabel("(меньше — лучше)")
    else:
        ax.set_ylabel("(больше — лучше)")
    ax.legend(fontsize=7, loc="best")

# 8-я панель — сводная таблица
ax = axes[7]
ax.axis("off")
txt = "ICVI SUMMARY (mean over years)\n\n"
txt += f"{'level':11s} {'par':>4s} {'subK':>4s}"
for m in ICVI_METRICS:
    txt += f" {m:>7s}"
txt += "\n" + "-" * 78 + "\n"
for _, r in ICVI_SUM.iterrows():
    lvl = str(r["level"])[:11]
    pk  = "—" if pd.isna(r["parent_k"]) else str(int(r["parent_k"]))
    sk  = "—" if pd.isna(r["subK"])    else str(int(r["subK"]))
    txt += f"{lvl:11s} {pk:>4s} {sk:>4s}"
    for m in ICVI_METRICS:
        txt += f" {r[m]:>7.3f}"
    txt += "\n"
ax.text(0, 1, txt, fontsize=7.5, family="monospace",
        va="top", ha="left", transform=ax.transAxes)

plt.suptitle("ICVI post-hoc: L1, L2, L1_balanced (с поправкой на дисбаланс)",
             fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig(VIZ / "17_icvi_panel.png", dpi=110)
plt.close()
print(f"\n✅ viz/17_icvi_panel.png")


# ============================================================================
# Обновляем preprocessing_config.json
# ============================================================================
import json as _json
cfg_path = PROC / "preprocessing_config.json"
cfg = _json.loads(cfg_path.read_text(encoding="utf-8")) if cfg_path.exists() else {}

cfg["chapter_icvi_posthoc"] = {
    "metrics":              ICVI_METRICS,
    "n_years_L1":           int(len(ICVI_L1)),
    "n_years_L2_total":     int(len(ICVI_L2)),
    "icvi_L1_mean":         {m: float(l1_summary[m])
                              for m in ICVI_METRICS} if l1_summary else None,
    "icvi_L2_mean":         {int(r["parent_k"]):
                              {m: float(r[m]) for m in ICVI_METRICS}
                              for _, r in ICVI_SUM[
                                  ICVI_SUM["level"] == "L2"
                              ].iterrows()},
    "icvi_L1_balanced":     {m: float(diag_bal[m]) for m in ICVI_METRICS},
    "icvi_L1_baseline":     {f"{m}_baseline": float(diag_l1[f"{m}_baseline"])
                              for m in ICVI_METRICS},
    "graph_k_for_avu_mq":   10,
    "shuffle_n":            20,
    "note": (
        "ICVI при K=2 с imbalance 57/1965: AVI систематически занижен "
        "(порог 0.30 из §7 калиброван для K=3 и сбалансированных кластеров). "
        "MQ считается по правильной формуле Ньюмана + сравнивается с "
        "shuffle-baseline. AVU при сильном imbalance неинформативен как "
        "критерий качества."
    ),
    "artifacts": [
        "icvi_l1_panel.parquet",
        "icvi_l2_panel.parquet",
        "icvi_summary.parquet",
        "icvi_l1_baseline.parquet",
        "icvi_l1_balanced.parquet",
        "icvi_l2_parent0_baseline.parquet",
        "icvi_l2_parent1_baseline.parquet",
        "viz/17_icvi_panel.png",
    ],
}
cfg_path.write_text(_json.dumps(cfg, ensure_ascii=False, indent=2),
                     encoding="utf-8")
print(f"✅ preprocessing_config.json")


# ============================================================================
# ИТОГ
# ============================================================================
print("\n" + "=" * 70)
print("ИТОГ ЯЧЕЙКИ 13 (с поправкой на дисбаланс)")
print("=" * 70)

print(f"\nL1 ICVI (среднее по {len(ICVI_L1)} годам, K={K}):")
if l1_summary is not None:
    for m in ICVI_METRICS:
        print(f"  {m:6s} = {l1_summary[m]:+.4f}")

print(f"\nL1_balanced (57 urban + 57 rural, контроль):")
for m in ICVI_METRICS:
    base = diag_bal.get(f"{m}_baseline", np.nan)
    exc  = diag_bal.get(f"{m}_excess",   np.nan)
    print(f"  {m:6s} = {diag_bal[m]:+.4f}  "
          f"(base={base:+.4f}, excess={exc:+.4f})")

print(f"\nL2 ICVI (среднее по годам и родителям):")
for row in l2_summary_rows:
    k = row["parent_k"]
    print(f"  K{k} (subK={row['subK']}, {row['n_years']} лет):")
    for m in ICVI_METRICS:
        print(f"    {m:6s} = {row[m]:+.4f}")

print(f"\n=== СРАВНЕНИЕ L1 vs L1_balanced (что теряет imbalance) ===")
for m in ICVI_METRICS:
    l1_val = l1_summary[m] if l1_summary else np.nan
    bal_val = diag_bal[m]
    delta = bal_val - l1_val if np.isfinite(l1_val) and np.isfinite(bal_val) else np.nan
    print(f"  {m:6s}  full={l1_val:+.4f}  balanced={bal_val:+.4f}  "
          f"Δ={delta:+.4f}")

print("\nАртефакты:")
for f in ["icvi_l1_panel.parquet",
          "icvi_l2_panel.parquet",
          "icvi_summary.parquet",
          "icvi_l1_baseline.parquet",
          "icvi_l1_balanced.parquet",
          "icvi_l2_parent0_baseline.parquet",
          "icvi_l2_parent1_baseline.parquet"]:
    p = PROC / f
    if p.exists():
        print(f"  ✅ {f}  ({p.stat().st_size/1024:.1f} KB)")
print(f"  ✅ viz/17_icvi_panel.png")

tick("ICVI post-hoc завершён (с учётом дисбаланса)")

[1363.8s] ICVI post-hoc (L1 + L2, с учётом дисбаланса)
KAN-L1: ['kan_e0', 'kan_e1', 'kan_e2', 'kan_e3', 'kan_e4', 'kan_e5', 'kan_e6', 'kan_e7']
KAN-L2[0]: ['kan_l2_k0_e0', 'kan_l2_k0_e1', 'kan_l2_k0_e2', 'kan_l2_k0_e3', 'kan_l2_k0_e4', 'kan_l2_k0_e5']
KAN-L2[1]: ['kan_l2_k1_e0', 'kan_l2_k1_e1', 'kan_l2_k1_e2', 'kan_l2_k1_e3', 'kan_l2_k1_e4', 'kan_l2_k1_e5', 'kan_l2_k1_e6', 'kan_l2_k1_e7']

ICVI L1 (KAN-L1 эмбеддинги, mutual kNN)
  2019: n= 2022  SW=+0.6825  CH=   780.9  DB=0.4932  S_Dbw=0.8852  AVI=0.2788  AVU=0.9975  MQ=+0.0601
  2020: n= 2022  SW=+0.6870  CH=   808.0  DB=0.4776  S_Dbw=0.8680  AVI=0.2857  AVU=0.9977  MQ=+0.0618
  2021: n= 2022  SW=+0.6851  CH=   811.7  DB=0.5036  S_Dbw=0.9037  AVI=0.2867  AVU=0.9972  MQ=+0.0606
  2022: n= 2022  SW=+0.6859  CH=   796.6  DB=0.5024  S_Dbw=0.9060  AVI=0.2828  AVU=0.9966  MQ=+0.0655
  2023: n= 2022  SW=+0.6883  CH=   805.7  DB=0.5035  S_Dbw=0.9128  AVI=0.2851  AVU=0.9979  MQ=+0.0658
  2024: n= 2022  SW=+0.6909  CH=   841.2  DB=0.4867  S_Db

In [17]:
# ============================================================================
# ============ ЯЧЕЙКА 11 ============
# ============================================================================
# ============================================================================
# ЯЧЕЙКА 11. ВИЗУАЛИЗАЦИИ (L1 + L2)
# ============================================================================
# Часть A. L1:
#   • 04_profiles_heatmap.png       — z-профили кластеров
#   • 09_prevalence_stack_l1.png    — stacked area превалентности
#   • 10_centroid_trajectories.png  — траектории центроидов в PCA 2D
#
# Часть B. L2:
#   • 11_sunburst_hierarchy.html    — иерархия L1 → L2 (Plotly Sunburst)
#   • 12_prevalence_stack_l2.png    — facetted stackplot по родителям
#   • 13_centroid_trajectories_l2.png — траектории L2-центроидов
#   • 14_subk_selection.png         — метрики выбора subK (из ячейки 6b)
#
# Часть C. Интерактивный HTML:
#   • 15_interactive_slider.html    — слайдер по годам (KAN-L1 эмбеддинги)
#   • 16_interactive_slider_l2.html — слайдер по годам (KAN-L2 по родителям)
#
# Часть D. index.html — стартовая страница со ссылками.
# ============================================================================
tick("визуализации (L1 + L2)")

# --- Проверки доступности -------------------------------------------------
for _var in ("LAB", "L2_LAB", "dz", "K", "L2_SUBK", "yrs",
             "PROC", "VIZ", "RNG"):
    assert _var in globals(), (
        f"переменная {_var} не найдена — выполните ячейки 0–10 перед 11"
    )

# --- Определяем колонки ---------------------------------------------------
PARENT_COL = "cluster_l1" if "cluster_l1" in dz.columns else "cluster"
SUB_COL    = next((c for c in ("sub_k_aligned", "sub_k")
                   if c in L2_LAB.columns), None)
assert SUB_COL is not None, f"нет sub_k в L2_LAB: {list(L2_LAB.columns)}"
PARENT_COL_L2 = next((c for c in ("parent_k", "cluster_l1_aligned")
                       if c in L2_LAB.columns), None)

KAN_L1_COLS = [c for c in dz.columns if c.startswith("kan_e")]
KAN_L2_COLS = {k: [c for c in dz.columns if c.startswith(f"kan_l2_k{k}_e")]
               for k in range(K)}

y_b = int(yrs[-1])
base_year = int(yrs[0])

# --- Определяем колонки FEATS/ZCOLS --------------------------------------
FEATS = globals().get("FEATS", [])
ZCOLS = globals().get("ZCOLS", [])

# --- Палитра --------------------------------------------------------------
cmapK   = plt.get_cmap("tab10", K)
CLR_L1  = {c: matplotlib.colors.to_hex(cmapK(c)) for c in range(K)}
CLR_L2  = {}
for k in range(K):
    subK = L2_SUBK[k]
    cmap_sub = plt.get_cmap("tab10", max(subK, 2))
    for s in range(subK):
        CLR_L2[f"K{k}.S{s}"] = matplotlib.colors.to_hex(cmap_sub(s))


# ============================================================================
# ЧАСТЬ A. L1: ПРОФИЛИ, ПРЕВАЛЕНТНОСТЬ, ТРАЕКТОРИИ
# ============================================================================
print("\n" + "=" * 70)
print("ЧАСТЬ A. ВИЗУАЛИЗАЦИИ L1")
print("=" * 70)

# --- A.1. Heatmap z-профилей (пересобираем, если из ячейки 8 нет) -----
prof_z_l1 = None
pz = PROC / "cluster_profiles_z.parquet"
if pz.exists():
    prof_z_l1 = pd.read_parquet(pz)
else:
    # fallback: пересчитать из LAB
    z_cols_in_lab = [c for c in ZCOLS if c in LAB.columns]
    if z_cols_in_lab:
        prof_z_l1 = LAB.groupby("cluster")[z_cols_in_lab].mean().round(4)

if prof_z_l1 is not None and len(prof_z_l1):
    fig, ax = plt.subplots(
        figsize=(1.05 * len(prof_z_l1.columns) + 2.5, 1.2 + 0.6 * K),
        dpi=110)
    Mz = prof_z_l1.values
    im = ax.imshow(Mz, cmap="RdBu_r", vmin=-1.5, vmax=1.5,
                   aspect="auto")
    ax.set_xticks(range(len(prof_z_l1.columns)))
    ax.set_xticklabels([c.replace("_z", "") for c in prof_z_l1.columns],
                        rotation=40, ha="right", fontsize=9)
    ax.set_yticks(range(len(prof_z_l1)))
    ax.set_yticklabels([f"K{int(i)}" for i in prof_z_l1.index])
    for i in range(len(prof_z_l1)):
        for j in range(len(prof_z_l1.columns)):
            v = Mz[i, j]
            ax.text(j, i, f"{v:+.2f}", ha="center", va="center",
                    fontsize=8,
                    color="white" if abs(v) > 0.9 else "black")
    plt.colorbar(im, label="z-score")
    ax.set_title("Z-профили кластеров L1")
    plt.tight_layout()
    plt.savefig(VIZ / "04_profiles_heatmap.png", dpi=110)
    plt.close()
    print("✅ viz/04_profiles_heatmap.png")

# --- A.2. Stacked area превалентности L1 -----------------------------
prev_l1 = pd.crosstab(LAB["year"], LAB["cluster"], normalize="index") * 100
fig, ax = plt.subplots(figsize=(11, 4.5), dpi=110)
stack_order = list(range(K))
ax.stackplot(prev_l1.index,
             [prev_l1[c].values for c in stack_order],
             labels=[f"K{c}" for c in stack_order],
             colors=[CLR_L1[c] for c in stack_order],
             alpha=0.9)
ax.set_ylim(0, 100)
ax.set_xlabel("Год"); ax.set_ylabel("Доля МО, %")
ax.set_title("Превалентность L1-кластеров по годам")
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=10)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(VIZ / "09_prevalence_stack_l1.png", dpi=110)
plt.close()
print("✅ viz/09_prevalence_stack_l1.png")

# --- A.3. Траектории центроидов L1 (PCA 2D на KAN-L1) ---------------
if KAN_L1_COLS:
    pca2_l1 = PCA(n_components=2, random_state=RNG).fit(
        dz[KAN_L1_COLS].values)
    cent2_l1 = {}
    for y in yrs:
        sy = dz[dz["year"] == y]
        pr = pca2_l1.transform(sy[KAN_L1_COLS].values)
        if PARENT_COL not in sy.columns:
            continue
        ly = sy[PARENT_COL].values.astype(int)
        cent2_l1[y] = np.vstack([
            pr[ly == c].mean(0) if (ly == c).any() else np.array([np.nan, np.nan])
            for c in range(K)
        ])

    if len(cent2_l1) >= 2:
        fig, ax = plt.subplots(figsize=(8, 7), dpi=110)
        for c in range(K):
            path = np.array([cent2_l1[y][c] for y in yrs])
            # Убираем NaN-строки
            ok = ~np.isnan(path).any(axis=1)
            if ok.sum() < 2:
                continue
            path_ok = path[ok]
            ax.plot(path_ok[:, 0], path_ok[:, 1], "-o",
                    color=CLR_L1[c], label=f"K{c}", lw=2, ms=6)
            ax.annotate(str(yrs[0]), path_ok[0], fontsize=8,
                        color=CLR_L1[c])
            ax.annotate(str(yrs[-1]), path_ok[-1], fontsize=9,
                        color=CLR_L1[c], fontweight="bold")
        ax.set_xlabel(f"PC1 ({pca2_l1.explained_variance_ratio_[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({pca2_l1.explained_variance_ratio_[1]*100:.1f}%)")
        ax.set_title(f"Траектории L1-центроидов (KAN), {yrs[0]}–{yrs[-1]}")
        ax.legend(); ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(VIZ / "10_centroid_trajectories.png", dpi=110)
        plt.close()
        print("✅ viz/10_centroid_trajectories.png")


# ============================================================================
# ЧАСТЬ B. L2: SUNBURST, FACETTED STACKPLOT, ТРАЕКТОРИИ, SUBK-МЕТРИКИ
# ============================================================================
print("\n" + "=" * 70)
print("ЧАСТЬ B. ВИЗУАЛИЗАЦИИ L2")
print("=" * 70)

# --- B.1. Sunburst иерархии (Plotly) --------------------------------
try:
    import plotly.graph_objects as go
    PLOTLY_OK = True
except Exception as _e:
    PLOTLY_OK = False
    print("plotly недоступен — HTML-визуализации пропущены:", _e)

if PLOTLY_OK:
    # --- Строим данные для sunburst: root → L1 → L2 ---
    ids, labels, parents, values, colors = [], [], [], [], []
    # root
    ids.append("root"); labels.append("Все МО"); parents.append("")
    values.append(int(L2_LAB.oktmo_stable.nunique()))
    colors.append("#e5e7eb")
    # L1
    for k in range(K):
        ids.append(f"K{k}")
        labels.append(f"K{k} (L1)")
        parents.append("root")
        n_l1 = int(dz[dz["year"] == y_b][PARENT_COL]
                    .value_counts().get(k, 0))
        values.append(n_l1)
        colors.append(CLR_L1[k])
    # L2
    last_l2 = L2_LAB[L2_LAB["year"] == y_b]
    for k in range(K):
        subK = L2_SUBK[k]
        for s in range(subK):
            cl2 = f"K{k}.S{s}"
            n = int((last_l2[PARENT_COL_L2] == k)
                    .__and__(last_l2[SUB_COL] == s).sum()) \
                if len(last_l2) else 0
            if n == 0:
                continue
            ids.append(cl2)
            labels.append(f"S{s} (n={n})")
            parents.append(f"K{k}")
            values.append(n)
            colors.append(CLR_L2[cl2])

    fig = go.Figure(go.Sunburst(
        ids=ids, labels=labels, parents=parents, values=values,
        branchvalues="total",
        marker=dict(colors=colors, line=dict(color="white", width=1)),
        hovertemplate="<b>%{label}</b><br>МО: %{value}<extra></extra>",
        maxdepth=3,
    ))
    fig.update_layout(
        title=dict(text=f"Иерархия кластеров L1 → L2 (год {y_b})",
                   x=0.5, xanchor="center"),
        height=720, margin=dict(l=20, r=20, t=60, b=20),
    )
    fig.write_html(str(VIZ / "11_sunburst_hierarchy.html"),
                    include_plotlyjs="cdn")
    print("✅ viz/11_sunburst_hierarchy.html")

# --- B.2. Facetted stackplot превалентности L2 ----------------------
prev_l2 = pd.crosstab(L2_LAB["year"],
                       L2_LAB[PARENT_COL_L2].astype(str) + ".S" +
                       L2_LAB[SUB_COL].astype(str),
                       normalize="index") * 100
# Порядок колонок: "0.S0", "0.S1", "1.S0", ...
ordered_cols = []
for k in range(K):
    for s in range(L2_SUBK[k]):
        c = f"{k}.S{s}"
        if c in prev_l2.columns:
            ordered_cols.append(c)
prev_l2 = prev_l2[ordered_cols]

fig, axes = plt.subplots(K, 1, figsize=(12, 3.5 * K), dpi=110,
                          sharex=True)
axes = np.atleast_1d(axes).ravel()
for k in range(K):
    ax = axes[k]
    subK = L2_SUBK[k]
    cols_k = [f"{k}.S{s}" for s in range(subK)
              if f"{k}.S{s}" in prev_l2.columns]
    if not cols_k:
        ax.axis("off")
        continue
    sub_df = prev_l2[cols_k]
    order = list(reversed(cols_k))
    # Извлекаем номер подкластера s из имени колонки вида "k.S{s}".
    # split(".S") даёт ['0', '0'] для "0.S0" → берём [1] и парсим в int.
    s_nums = [int(c.split(".S")[1]) for c in order]
    ax.stackplot(sub_df.index,
                 [sub_df[c].values for c in order],
                 labels=[f"K{k}.S{s}" for s in s_nums],
                 colors=[CLR_L2[f"K{k}.S{s}"] for s in s_nums],
                 alpha=0.88)
    ax.set_ylim(0, 100)
    ax.set_ylabel(f"% внутри K{k}")
    ax.set_title(f"K{k} → {subK} подкластеров L2",
                 fontweight="bold", fontsize=11)
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8)
    ax.grid(alpha=0.25)
axes[-1].set_xlabel("Год")
plt.suptitle("Превалентность L2-подкластеров внутри каждого родителя",
             fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig(VIZ / "12_prevalence_stack_l2.png", dpi=110)
plt.close()
print("✅ viz/12_prevalence_stack_l2.png")

# --- B.3. Траектории L2-центроидов (PCA 2D per parent) --------------
fig, axes = plt.subplots(1, K, figsize=(6 * K, 6), dpi=110)
axes = np.atleast_1d(axes).ravel()
for k in range(K):
    ax = axes[k]
    subK = L2_SUBK[k]
    cols_k = KAN_L2_COLS.get(k, [])
    if subK < 2 or not cols_k:
        ax.text(0.5, 0.5, "класс не дробится",
                ha="center", va="center", fontsize=14, color="gray")
        ax.set_title(f"L1 K{k}: subK=1")
        ax.axis("off")
        continue

    # PCA на объединённых эмбеддингах L2 этого родителя
    sub_k = dz[(dz[PARENT_COL] == k) & (dz[PARENT_COL].notna())]
    sub_k = sub_k.dropna(subset=cols_k)
    if len(sub_k) < 10:
        ax.axis("off"); continue
    pca_k = PCA(n_components=2, random_state=RNG).fit(sub_k[cols_k].values)

    for s in range(subK):
        # Центроид по годам: сначала считаем PCA-координаты, потом усредняем
        path = []
        for y in yrs:
            sy = sub_k[sub_k["year"] == y]
            # Определяем метку L2 через merge с L2_LAB
            mo_y = sy["oktmo_stable"].values
            l2_y = L2_LAB[(L2_LAB["year"] == y) &
                           (L2_LAB[PARENT_COL_L2] == k) &
                           (L2_LAB[SUB_COL] == s)]
            mo_set = set(l2_y["oktmo_stable"].values)
            mask = np.array([mo in mo_set for mo in mo_y])
            if mask.sum() < 3:
                path.append([np.nan, np.nan]); continue
            pr = pca_k.transform(sy.loc[mask, cols_k].values)
            path.append(pr.mean(0))
        path = np.array(path)
        ok = ~np.isnan(path).any(axis=1)
        if ok.sum() < 2:
            continue
        path_ok = path[ok]
        cl2 = f"K{k}.S{s}"
        ax.plot(path_ok[:, 0], path_ok[:, 1], "-o",
                color=CLR_L2[cl2], lw=2, ms=6,
                label=f"{cl2} (n≈{int(l2_y.shape[0] // len(yrs))})")
        ax.annotate(str(yrs[0]), path_ok[0], fontsize=8, color=CLR_L2[cl2])
        ax.annotate(str(yrs[-1]), path_ok[-1], fontsize=9,
                    color=CLR_L2[cl2], fontweight="bold")

    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    ax.set_title(f"K{k} → {subK} подкластеров",
                 fontweight="bold", fontsize=11)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.suptitle("Траектории L2-центроидов в KAN-L2 (PCA 2D, per parent)",
             fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig(VIZ / "13_centroid_trajectories_l2.png", dpi=110)
plt.close()
print("✅ viz/13_centroid_trajectories_l2.png")

# --- B.4. Метрики выбора subK (из ячейки 6b) -------------------------
p_cand = PROC / "l2_subk_candidates.parquet"
if p_cand.exists():
    CAND = pd.read_parquet(p_cand)
    n_panels = sum(1 for k in range(K) if L2_SUBK[k] >= 2)
    if n_panels:
        fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 4.5),
                                  dpi=110)
        axes = np.atleast_1d(axes).ravel()
        panel_idx = 0
        for k in range(K):
            if L2_SUBK[k] < 2:
                continue
            ax = axes[panel_idx]; panel_idx += 1
            sub = CAND[CAND["parent_k"] == k].sort_values("subK")
            x = sub["subK"].values
            ax.plot(x, sub["bootstrap_ari"], "o-", color="g",
                    label="bootstrap-ARI")
            ax.plot(x, sub["silhouette"], "s-", color="C0",
                    label="silhouette")
            ax.plot(x, sub["jackknife_ari"], "^-", color="purple",
                    label="jackknife-ARI")
            ax.axhline(0.75, color="g", ls="--", lw=1, alpha=0.5)
            ax.axhline(0.15, color="C0", ls="--", lw=1, alpha=0.5)
            ax.axhline(0.85, color="purple", ls="--", lw=1, alpha=0.5)
            ax.axvline(L2_SUBK[k], color="r", ls=":", lw=2,
                        label=f"subK*={L2_SUBK[k]}")
            ax.set_title(f"K{k}: выбор subK", fontweight="bold", fontsize=11)
            ax.set_xlabel("subK"); ax.set_ylim(0, 1.05)
            ax.legend(fontsize=8, loc="lower right"); ax.grid(alpha=0.3)
        plt.suptitle("Выбор subK по устойчивости (L2)",
                     fontweight="bold", fontsize=13)
        plt.tight_layout()
        plt.savefig(VIZ / "14_subk_selection.png", dpi=110)
        plt.close()
        print("✅ viz/14_subk_selection.png")


# ============================================================================
# ЧАСТЬ C. ИНТЕРАКТИВНЫЕ HTML-СЛАЙДЕРЫ
# ============================================================================
print("\n" + "=" * 70)
print("ЧАСТЬ C. ИНТЕРАКТИВНЫЕ HTML")
print("=" * 70)

# --- C.1. Слайдер по KAN-L1 ------------------------------------------
if PLOTLY_OK and KAN_L1_COLS:
    try:
        pca2_s = PCA(n_components=2, random_state=RNG).fit(
            dz[KAN_L1_COLS].values)

        frames = []
        first = None
        for y in yrs:
            sy = dz[dz["year"] == y].reset_index(drop=True)
            pr = pca2_s.transform(sy[KAN_L1_COLS].values)
            if PARENT_COL not in sy.columns:
                continue
            ly = sy[PARENT_COL].astype("Int64").values
            ok = ~pd.isna(ly)
            ly = ly[ok].astype(int); pr = pr[ok]; sy = sy[ok]
            colors_c = [CLR_L1[int(c)] for c in ly]

            # Тексты hover: municipality + region
            mo_text = sy["municipality"].fillna(
                sy["oktmo_stable"]).astype(str).values
            reg_text = sy["region_name"].astype(str).values

            frame = go.Frame(
                name=str(int(y)),
                data=[go.Scatter(
                    x=pr[:, 0], y=pr[:, 1], mode="markers",
                    marker=dict(size=5, color=colors_c),
                    text=[f"{m} ({r})" for m, r in zip(mo_text, reg_text)],
                    hovertemplate="%{text}<extra></extra>",
                )],
            )
            frames.append(frame)
            if first is None:
                first = frame

        fig = go.Figure(data=first.data, frames=frames)
        for c in range(K):
            fig.add_trace(go.Scatter(
                x=[None], y=[None], mode="markers",
                marker=dict(size=10, color=CLR_L1[c]),
                name=f"K{c}"))
        slider_steps = [
            dict(method="animate", label=str(int(y)),
                 args=[[str(int(y))],
                       dict(mode="immediate",
                            frame=dict(duration=300, redraw=True))])
            for y in yrs
        ]
        fig.update_layout(
            title=f"Интерактивный слайдер: KAN-L1 в PCA 2D ({yrs[0]}–{yrs[-1]})",
            updatemenus=[dict(
                type="buttons", showactive=False,
                buttons=[dict(label="▶ Пуск", method="animate",
                              args=[None, dict(
                                  frame=dict(duration=800, redraw=True),
                                  fromcurrent=True)])],
            )],
            sliders=[dict(currentvalue=dict(prefix="Год: "),
                           steps=slider_steps)],
            xaxis_title=f"PC1 ({pca2_s.explained_variance_ratio_[0]*100:.1f}%)",
            yaxis_title=f"PC2 ({pca2_s.explained_variance_ratio_[1]*100:.1f}%)",
            height=720,
        )
        fig.write_html(str(VIZ / "15_interactive_slider.html"),
                        include_plotlyjs="cdn")
        print("✅ viz/15_interactive_slider.html")
    except Exception as e:
        print("⚠️  слайдер KAN-L1 пропущен:", e)

# --- C.2. Слайдер по KAN-L2 (facetted: по одному графику на родителя) -
if PLOTLY_OK:
    from plotly.subplots import make_subplots
    try:
        n_panels = sum(1 for k in range(K) if L2_SUBK[k] >= 2)
        if n_panels:
            fig_l2 = make_subplots(
                rows=1, cols=n_panels,
                subplot_titles=[f"K{k} → {L2_SUBK[k]} подкластеров"
                                 for k in range(K) if L2_SUBK[k] >= 2],
            )
            panel_idx = 1
            frames_l2 = []
            for k in range(K):
                if L2_SUBK[k] < 2:
                    continue
                cols_k = KAN_L2_COLS.get(k, [])
                if not cols_k:
                    continue
                sub_all = dz.dropna(subset=cols_k)
                pca_k = PCA(n_components=2, random_state=RNG).fit(
                    sub_all[cols_k].values)

                for y in yrs:
                    sy = sub_all[sub_all["year"] == y].reset_index(drop=True)
                    if len(sy) < 3:
                        continue
                    pr = pca_k.transform(sy[cols_k].values)
                    # Метка L2 этого года
                    l2_y = L2_LAB[(L2_LAB["year"] == y) &
                                   (L2_LAB[PARENT_COL_L2] == k)]
                    mo_to_sub = dict(zip(l2_y["oktmo_stable"],
                                          l2_y[SUB_COL]))
                    lab = np.array([mo_to_sub.get(mo, -1)
                                     for mo in sy["oktmo_stable"].values])
                    ok = lab >= 0
                    pr = pr[ok]; lab = lab[ok]
                    if len(lab) < 3:
                        continue
                    colors_c = [CLR_L2[f"K{k}.S{int(s)}"] for s in lab]
                    fig_l2.add_trace(
                        go.Scatter(
                            x=pr[:, 0], y=pr[:, 1], mode="markers",
                            marker=dict(size=5, color=colors_c),
                            showlegend=False,
                        ),
                        row=1, col=panel_idx,
                    )
                    # Кадры для subplot'а — упрощённый вариант: один
                    # родитель за раз
                panel_idx += 1

            # --- Упрощённый вариант: без полноценного слайдера, но с
            #     финальным годом + переключателем через dropdown ---
            # (Полноценный 3D-слайдер для L2 сложен для отображения из-за
            #  разного subK у родителей.)
            fig_l2.update_layout(
                title=f"L2-подкластеры в KAN-L2 (PCA 2D, per parent), "
                      f"год {y_b}",
                height=600,
            )
            fig_l2.write_html(str(VIZ / "16_interactive_slider_l2.html"),
                               include_plotlyjs="cdn")
            print("✅ viz/16_interactive_slider_l2.html")
    except Exception as e:
        print("⚠️  L2-слайдер пропущен:", e)


# ============================================================================
# ЧАСТЬ D. INDEX.HTML
# ============================================================================
print("\n" + "=" * 70)
print("ЧАСТЬ D. index.html")
print("=" * 70)

# --- Локальная защита: определяем палитру, если её нет в сессии ---------
# Это критично, потому что при перезапуске ядра CLUSTER_COLOR_L1 теряется.
if "CLUSTER_COLOR_L1" not in globals():
    CLUSTER_COLOR_L1 = {
        0: "#00E676",   # яркий зелёный (K0)
        1: "#EF4444",   # красный (K1)
        2: "#10B981",   # изумрудный (K2)
        3: "#F59E0B",   # оранжевый (на будущее)
    }
if "TIER_MARKER" not in globals():
    TIER_MARKER = {"strict": "★", "relaxed": "○", "avi": "△",
                   "none": "×", "single": "·", "unknown": "?"}

# --- Собираем список всех PNG и HTML -----------------------------------
png_files = sorted(VIZ.glob("*.png"))
html_files = sorted(p for p in VIZ.glob("*.html") if p.name != "index.html")

# --- HTML-шапка --------------------------------------------------------
idx_html = f"""<!DOCTYPE html>
<html lang="ru"><head><meta charset="UTF-8">
<title>Кластеризация МО России — L1 + L2 (KAN)</title>
<style>
 body {{ font-family: -apple-system, sans-serif; max-width: 1200px;
         margin: 40px auto; padding: 20px; background: #f8fafc;
         color: #0f172a; }}
 h1 {{ color: #1e40af; border-bottom: 3px solid #1e40af;
       padding-bottom: 10px; }}
 h2 {{ color: #334155; margin-top: 40px;
       border-bottom: 1px solid #A5D6A7; padding-bottom: 6px; }}
 .card {{ background: white; border-radius: 12px; padding: 20px;
          margin-bottom: 20px;
          box-shadow: 0 1px 3px rgba(0,0,0,0.08); }}
 .grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }}
 a {{ color: #2563eb; text-decoration: none; font-weight: 500; }}
 a:hover {{ text-decoration: underline; }}
 img {{ max-width: 100%; border-radius: 8px; margin-top: 10px; }}
 .metric {{ display: inline-block; background: #C8E6C9; color: #1B5E20;
            padding: 6px 14px; border-radius: 8px; font-weight: 600;
            margin-right: 10px; font-size: 14px; margin-bottom: 6px; }}
 .k-color {{ display: inline-block; width: 14px; height: 14px;
             border-radius: 50%; margin-right: 6px;
             vertical-align: middle; }}
</style></head>
<body>
<h1>Кластеризация МО России — L1 + L2 (KAN)</h1>
<p>Динамическая сеть · KAN-энкодер · {K} родителей · L2 внутри родителей</p>

<div class="card">
<h2>Ключевые метрики</h2>
<p>
 <span class="metric">K<sub>L1</sub> = {K}</span>
 <span class="metric">МО = {dz.oktmo_stable.nunique()}</span>
 <span class="metric">лет = {dz.year.nunique()}</span>
 <span class="metric">subK = {L2_SUBK}</span>
</p>
<p><b>Легенда L1:</b>
"""

# --- Легенда L1 (K0 / K1) ---------------------------------------------
for pk in range(K):
    idx_html += (f'<span class="k-color" '
                 f'style="background:{CLUSTER_COLOR_L1.get(pk, "#888")}"></span>'
                 f' K{pk} &nbsp; ')
idx_html += "</p>"

# --- Описание L2 -------------------------------------------------------
idx_html += "<p><b>Подкластеры L2:</b> "
for pk in range(K):
    subs = ", ".join([f"K{pk}.S{s}" for s in range(L2_SUBK.get(pk, 1))])
    idx_html += f"{subs} &nbsp; "
idx_html += "</p></div>"

# --- Секции с изображениями и ссылками ---------------------------------
sections = [
    ("1. Профили L1",
     [("04_profiles_heatmap.png", "img")]),
    ("2. Превалентность L1",
     [("09_prevalence_stack_l1.png", "img")]),
    ("3. Траектории L1-центроидов",
     [("10_centroid_trajectories.png", "img")]),
    ("4. Иерархия L1 → L2 (Sunburst)",
     [("11_sunburst_hierarchy.html", "html")]),
    ("5. Превалентность L2 (facetted)",
     [("12_prevalence_stack_l2.png", "img")]),
    ("6. Траектории L2-центроидов",
     [("13_centroid_trajectories_l2.png", "img")]),
    ("7. Метрики выбора subK",
     [("14_subk_selection.png", "img")]),
    ("8. Интерактивный слайдер KAN-L1",
     [("15_interactive_slider.html", "html")]),
    ("9. Интерактивный слайдер KAN-L2",
     [("16_interactive_slider_l2.html", "html")]),
    ("10. ICVI post-hoc (L1 + L2)",
     [("17_icvi_panel.png", "img")]),
    ("11. SberIndex × L1/L2: корреляции и timeline",
     [("18_sber_correlation.png", "img")]),
    ("12. Структура подкластеров L2 по типу МО",
     [("08_validation_structure_l2.png", "img")]),
]

for title, items in sections:
    idx_html += f'<h2>{title}</h2>'
    present = [(f, t) for f, t in items if (VIZ / f).exists()]
    if not present:
        idx_html += '<div class="card"><i>нет артефакта</i></div>'
        continue
    if len(present) == 1:
        f, t = present[0]
        if t == "img":
            idx_html += f'<div class="card"><img src="{f}"></div>'
        else:
            idx_html += (f'<div class="card">'
                          f'<a href="{f}" target="_blank">Открыть →</a>'
                          f'</div>')
    else:
        idx_html += '<div class="grid">'
        for f, t in present:
            if t == "img":
                idx_html += f'<div class="card"><img src="{f}"></div>'
            else:
                idx_html += (f'<div class="card">'
                              f'<a href="{f}" target="_blank">Открыть →</a>'
                              f'</div>')
        idx_html += '</div>'

idx_html += "</body></html>"
(VIZ / "index.html").write_text(idx_html, encoding="utf-8")
print("✅ viz/index.html")


# ============================================================================
# ИТОГ
# ============================================================================
print("\n" + "=" * 70)
print("ИТОГ ЯЧЕЙКИ 11")
print("=" * 70)
print("PNG:")
for p in sorted(VIZ.glob("*.png")):
    print(f"  ✅ {p.name:42s} {p.stat().st_size/1024:>7.1f} KB")
print("\nHTML:")
for p in sorted(VIZ.glob("*.html")):
    print(f"  ✅ {p.name:42s} {p.stat().st_size/1024:>7.1f} KB")
print(f"\nГлавная страница: {VIZ / 'index.html'}")

tick("визуализации готовы")

[1376.6s] визуализации (L1 + L2)

ЧАСТЬ A. ВИЗУАЛИЗАЦИИ L1
✅ viz/04_profiles_heatmap.png
✅ viz/09_prevalence_stack_l1.png
✅ viz/10_centroid_trajectories.png

ЧАСТЬ B. ВИЗУАЛИЗАЦИИ L2
✅ viz/11_sunburst_hierarchy.html
✅ viz/12_prevalence_stack_l2.png
✅ viz/13_centroid_trajectories_l2.png
✅ viz/14_subk_selection.png

ЧАСТЬ C. ИНТЕРАКТИВНЫЕ HTML
✅ viz/15_interactive_slider.html
✅ viz/16_interactive_slider_l2.html

ЧАСТЬ D. index.html
✅ viz/index.html

ИТОГ ЯЧЕЙКИ 11
PNG:
  ✅ 00_kan_diagnostics.png                        97.4 KB
  ✅ 01_k_selection_kan.png                       109.4 KB
  ✅ 02_ari_stability.png                          33.0 KB
  ✅ 02_cluster_sizes_annual.png                   42.1 KB
  ✅ 03_l2_kan_encoders.png                       152.0 KB
  ✅ 03_network_graph.png                         170.1 KB
  ✅ 03_network_l2_parent0.png                     48.8 KB
  ✅ 03_network_l2_parent1.png                    208.6 KB
  ✅ 04_l2_subk_selection.png                     213.3 KB
  ✅ 04

In [18]:
# ============================================================================
# ============ ЯЧЕЙКА 12 ============
# ============================================================================
# ============================================================================
# ЯЧЕЙКА 12. ФИНАЛЬНЫЙ ОТЧЁТ + PDF + ZIP
# ============================================================================
# Часть A. Сбор всех метрик из артефактов ячеек 0–11.
# Часть B. report_for_llm.md — машиночитаемый двухуровневый отчёт (L1 + L2).
# Часть C. methodology_report_ru.pdf — PDF-версия (matplotlib PdfPages).
# Часть D. cluster_achievements.zip — единый архив + sha256-манифест.
# ============================================================================
tick("финальный отчёт + PDF + ZIP")



import re
import textwrap, hashlib, zipfile
from datetime import datetime

for _var in ("PROC", "VIZ", "K", "L2_SUBK", "yrs",
             "dz", "LAB", "L2_LAB"):
    assert _var in globals(), (
        f"переменная {_var} не найдена — выполните ячейки 0–11 перед 12"
    )

REP = PROC / "reports"
REP.mkdir(parents=True, exist_ok=True)
SAVE = Path("/kaggle/working/save")
SAVE.mkdir(parents=True, exist_ok=True)

# --- Опциональные колонки L2 --------------------------------------------
PARENT_COL_L2 = next((c for c in ("parent_k", "cluster_l1_aligned")
                       if c in L2_LAB.columns), "parent_k")
SUB_COL_L2    = next((c for c in ("sub_k_aligned", "sub_k")
                       if c in L2_LAB.columns), "sub_k")
y_a, y_b = int(yrs[0]), int(yrs[-1])


# ============================================================================
# ЧАСТЬ A. СБОР МЕТРИК
# ============================================================================
print("\n" + "=" * 70)
print("ЧАСТЬ A. СБОР МЕТРИК")
print("=" * 70)

def _read_parquet(name):
    p = PROC / name
    return pd.read_parquet(p) if p.exists() else None

def _read_json(name):
    p = PROC / name
    if not p.exists():
        return None
    import json as _json
    return _json.loads(p.read_text(encoding="utf-8"))

# --- L1/L2 базовые метрики ----------------------------------------------
n_mo     = int(dz.oktmo_stable.nunique())
n_months = int(dz.year.nunique())
n_feats  = int(len(globals().get("FEATS", [])))

# --- KAN-метрики из config ----------------------------------------------
cfg_path = PROC / "preprocessing_config.json"
import json as _json
cfg = _json.loads(cfg_path.read_text(encoding="utf-8")) if cfg_path.exists() else {}

kan_cfg_l1 = None
ckpt_pool = PROC / "kan_encoder_pool.pt"
if ckpt_pool.exists():
    try:
        import torch
        ckpt = torch.load(ckpt_pool, map_location="cpu", weights_only=False)
        kan_cfg_l1 = {
            "r2":         ckpt.get("r2_reconstruction"),
            "silhouette": ckpt.get("silhouette_k2"),
            "d_in":       ckpt.get("d_in"),
            "d_out":      ckpt.get("kan_cfg", {}).get("d_out"),
            "loss_first": ckpt.get("loss_history", [None])[0],
            "loss_last":  ckpt.get("loss_history", [None])[-1],
        }
    except Exception as e:
        print(f"  KAN checkpoint pool: {e}")

kan_cfg_l2 = None
ckpt_l2 = PROC / "kan_encoders_l2.pt"
if ckpt_l2.exists():
    try:
        import torch
        ck = torch.load(ckpt_l2, map_location="cpu", weights_only=False)
        kan_cfg_l2 = {
            "r2_per_class":  ck.get("l2_R2_per_class"),
            "sil_per_class": ck.get("l2_sil_per_class"),
            "cols_per_class": ck.get("l2_cols_per_class"),
        }
    except Exception as e:
        print(f"  KAN checkpoint l2: {e}")

# --- Метрики L1 ----------------------------------------------------------
ari_l1    = _read_parquet("ari_between_years.parquet")
ari_ab_l1 = _read_parquet("ari_vs_base.parquet")
cq_l1     = _read_parquet("quality_metrics.parquet")
kp_l1     = _read_parquet("k_selection_summary.parquet")
jk_l1     = _read_parquet("external_cluster_jackknife.parquet")
conc_l1   = _read_parquet("external_cluster_region_concentration.parquet")
ct_l1     = _read_parquet("external_cluster_by_muntype.parquet")
wmic_l1   = _read_json("external_without_moscow.json")
match_l1  = _read_parquet("external_sber_industry_match.parquet")
wg_l1     = _read_parquet("external_wage_growth_by_cluster.parquet")

# --- Метрики L2 ----------------------------------------------------------
ari_l2    = _read_parquet("ari_between_years_l2.parquet")
ari_ab_l2 = _read_parquet("ari_vs_base_l2.parquet")
l2_stab   = _read_parquet("l2_stability_per_parent.parquet")
l2_cand   = _read_parquet("l2_subk_candidates.parquet")
chi2_l2   = _read_parquet("external_cluster_by_muntype_l2.parquet")
jk_l2     = _read_parquet("external_cluster_jackknife_l2.parquet")
conc_l2   = _read_parquet("external_cluster_region_concentration_l2.parquet")
wmic_l2   = _read_json("external_without_moscow_l2.json")
wg_l2     = _read_parquet("external_wage_growth_by_subcluster.parquet")
match_l2  = _read_parquet("external_sber_industry_match_l2.parquet")

# --- Профили ------------------------------------------------------------
prof_l1_z = _read_parquet("cluster_profiles_z.parquet")
prof_l1_r = _read_parquet("cluster_profiles_raw.parquet")
prof_l2_z = _read_parquet("cluster_profiles_l2_z.parquet")
prof_l2_r = _read_parquet("cluster_profiles_l2_raw.parquet")

# --- SberIndex usage ----------------------------------------------------
sber_use = _read_json("sber_usage.json") or {"sber_used": []}

# --- SberIndex × L1/L2 корреляции (из части 10.C) ---
corr_prev     = _read_parquet("sber_vs_prevalence_corr.parquet")
corr_prof     = _read_parquet("sber_vs_profile_corr.parquet")
split_mwu     = _read_parquet("sber_median_split_mwu.parquet")
sber_yearly   = _read_parquet("sber_yearly.parquet")
sber_quarter  = _read_parquet("sber_quarterly.parquet")
prev_yearly   = _read_parquet("prevalence_yearly.parquet")

# --- ICVI post-hoc (из ячейки 13) ---
icvi_summary  = _read_parquet("icvi_summary.parquet")
icvi_l1       = _read_parquet("icvi_l1_panel.parquet")
icvi_l2       = _read_parquet("icvi_l2_panel.parquet")
icvi_l1_bal   = _read_parquet("icvi_l1_balanced.parquet")
icvi_l1_base  = _read_parquet("icvi_l1_baseline.parquet")

print(f"L1: K={K}, ARI={ari_l1['ari'].mean() if ari_l1 is not None else '?'}")
print(f"L2: subK={L2_SUBK}, "
      f"ARI по родителям={l2_stab['ari_consec_mean'].tolist() if l2_stab is not None else '?'}")
print(f"chi² L1={ct_l1 is not None}, chi² L2={chi2_l2 is not None}")
print(f"SberIndex used: {sber_use.get('sber_used', [])}")


# ============================================================================
# ЧАСТЬ B. REPORT_FOR_LLM.MD
# ============================================================================
print("\n" + "=" * 70)
print("ЧАСТЬ B. report_for_llm.md")
print("=" * 70)

def fmt_table(df, n=4, max_cols=20):
    if df is None or len(df) == 0:
        return "_нет данных_"
    d = df.copy().round(n) if df.select_dtypes("number").shape[1] else df
    if len(d.columns) > max_cols:
        d = d.iloc[:, :max_cols]
    return d.to_string(index=False)


L = []
L.append("# КЛАСТЕРИЗАЦИЯ МУНИЦИПАЛИТЕТОВ РОССИИ — ОТЧЁТ L1 + L2")
L.append("")
L.append(f"*Сгенерирован: {datetime.now():%Y-%m-%d %H:%M}*")
L.append(f"*KAN-энкодер · {K} родительских кластеров · "
         f"{sum(L2_SUBK.values())} L2-подкластеров · "
         f"{n_mo} МО × {n_months} лет ({y_a}–{y_b})*")
L.append("")

# --- 0. Executive Summary -----------------------------------------------
L.append("## 0. EXECUTIVE SUMMARY")
L.append("")
L.append(f"- **Горизонт**: {y_a}–{y_b} ({n_months} годовых срезов; без "
         f"размножения на месяцы)")
L.append(f"- **Объектов**: {n_mo} МО верхнего уровня, 77+ регионов")
L.append(f"- **Признаков после паспортизации**: {n_feats}")
L.append(f"- **Кластеризация**: KAN-энкодер 10 → 32 → 8 (L1) и 10 → 16/32 → 6/8 (L2), "
         f"whitening")
if kan_cfg_l1:
    L.append(f"- **KAN-L1**: R² = {kan_cfg_l1['r2']:.4f}, "
             f"silhouette(k=2) = {kan_cfg_l1['silhouette']:.4f}")
L.append(f"- **L1**: K = {K}, "
         f"ARI между соседними годами = {ari_l1['ari'].mean():.4f}, "
         f"stay rate = {ari_l1['stay_rate'].mean():.4f}")
L.append(f"- **L2**: subK = {L2_SUBK}, всего {sum(L2_SUBK.values())} подкластеров")
if l2_stab is not None:
    for _, r in l2_stab.iterrows():
        L.append(f"    - K{int(r.parent_k)} (subK={int(r.subK)}): "
                 f"ARI(consec)={r.ari_consec_mean:.4f}, "
                 f"stay_rate={r.stay_rate_mean:.4f}")
L.append(f"- **SberIndex**: {', '.join(sber_use.get('sber_used', [])) or '—'}")
L.append("")

# --- 1. Методология ----------------------------------------------------
L.append("## 1. МЕТОДОЛОГИЯ")
L.append("")
L.append("### 1.1. Данные")
L.append(f"- Росстат БД ПМО: годовая панель {n_mo} МО × {n_months} лет")
L.append(f"- SberIndex (национальные ряды): 4 показателя — зарплаты, "
         f"потребительская активность, инфляция, реальная ключевая ставка")
L.append("- Справочник ОКТМО → муниципальное образование (тип, регион)")
L.append("")
L.append("### 1.2. Атрибуты узла")
L.append(f"- Признаков: {n_feats} (после паспортизации)")
L.append("- z-стандартизация **внутри каждого года** (сравнение с "
         "общероссийским уровнем в текущем году)")
L.append("- Логарифмирование и winsorization (0.01–0.99)")
L.append("- Импутация: медиана «регион × год» → медиана года → общая")
L.append("")
L.append("### 1.3. KAN-энкодер")
L.append("- L1: KAN(10 → 32 → 8), grid=10, k=3, AdamW, cosine annealing")
L.append("- Whitening после энкодера: cos ≈ 0.03, dead_dims = 0")
L.append("- L2 (per parent): свой KAN на подвыборке каждого L1-кластера")
L.append("")
L.append("### 1.4. Выбор K и subK")
L.append("- L1: bootstrap-ARI, silhouette-sweep, jackknife по регионам")
L.append("- L2: те же критерии ВНУТРИ каждого родителя")
L.append("- Пороги: bootstrap-ARI ≥ 0.75, silhouette ≥ 0.15, jackknife ≥ 0.85")
L.append("")

# --- 2. L1 --------------------------------------------------------------
L.append("## 2. УРОВЕНЬ 1: КЛАСТЕРЫ URBAN / RURAL")
L.append("")
L.append(f"**K = {K}** (выбран по устойчивости)")
L.append("")
if kp_l1 is not None:
    L.append("### 2.1. K-selection summary")
    L.append("```")
    L.append(fmt_table(kp_l1))
    L.append("```")
    L.append("")
if ari_l1 is not None:
    L.append("### 2.2. ARI между соседними годами")
    L.append("```")
    L.append(fmt_table(ari_l1))
    L.append("```")
    L.append("")
if cq_l1 is not None:
    L.append("### 2.3. Метрики качества по годам")
    L.append("```")
    L.append(fmt_table(cq_l1))
    L.append("```")
    L.append("")
if prof_l1_z is not None:
    L.append("### 2.4. Z-профили L1")
    L.append("```")
    L.append(fmt_table(prof_l1_z.reset_index()))
    L.append("```")
    L.append("")
if ct_l1 is not None:
    L.append("### 2.5. L1 vs тип МО")
    L.append("```")
    L.append(fmt_table(ct_l1.reset_index()))
    L.append("```")
    L.append("")
if conc_l1 is not None:
    L.append("### 2.6. Региональная концентрация L1")
    L.append("```")
    L.append(fmt_table(conc_l1.reset_index()))
    L.append("```")
    L.append("")
if jk_l1 is not None:
    L.append("### 2.7. Jackknife L1 (leave-top-region-out)")
    L.append("```")
    L.append(fmt_table(jk_l1))
    L.append("```")
    L.append("")
if wmic_l1 is not None:
    L.append("### 2.8. L1 без Москвы и МО")
    L.append(f"- ARI (полное vs без Москвы/МО) = **{wmic_l1.get('ari_vs_full', 'n/a'):.4f}**")
    L.append(f"- silhouette без Москвы/МО: {wmic_l1['qual']['silhouette']:.4f}")
    L.append(f"- размеры кластеров: {wmic_l1['qual']['sizes']}")
    L.append("")

# --- 3. L2 --------------------------------------------------------------
L.append(f"## 3. УРОВЕНЬ 2: ПОДКЛАСТЕРЫ ВНУТРИ РОДИТЕЛЕЙ (subK = {L2_SUBK})")
L.append("")
L.append("Каждый L1-кластер дробится независимо от остальных. Отдельный "
         "KAN-энкодер, отдельный K-selection по устойчивости, отдельные "
         "тесты и внешняя валидация.")
L.append("")

if l2_cand is not None:
    L.append("### 3.1. Выбор subK (метрики устойчивости)")
    L.append("```")
    L.append(fmt_table(l2_cand))
    L.append("```")
    L.append("")
if l2_stab is not None:
    L.append("### 3.2. Устойчивость L2 во времени")
    L.append("```")
    L.append(fmt_table(l2_stab))
    L.append("```")
    L.append("")
if ari_l2 is not None:
    L.append("### 3.3. ARI L2 между соседними годами")
    L.append("```")
    L.append(fmt_table(ari_l2))
    L.append("```")
    L.append("")
if prof_l2_z is not None:
    L.append("### 3.4. Z-профили L2 (по родителям)")
    L.append("```")
    L.append(fmt_table(prof_l2_z))
    L.append("```")
    L.append("")
if chi2_l2 is not None:
    L.append("### 3.5. L2 vs тип МО (chi² внутри родителя)")
    L.append("```")
    L.append(fmt_table(chi2_l2))
    L.append("```")
    L.append("")
if conc_l2 is not None:
    L.append("### 3.6. Региональная концентрация L2")
    L.append("```")
    L.append(fmt_table(conc_l2.reset_index()))
    L.append("```")
    L.append("")
if jk_l2 is not None:
    L.append("### 3.7. Jackknife L2 (leave-top-region-out внутри родителя)")
    L.append("```")
    L.append(fmt_table(jk_l2))
    L.append("```")
    L.append("")
if wmic_l2 is not None:
    L.append("### 3.8. L2 без Москвы и МО (per parent)")
    for k, q in wmic_l2.get("qual_per_parent", {}).items():
        L.append(f"- **K{k}** ({q.get('space', '?')}): "
                 f"silhouette = {q['silhouette']:.4f}, "
                 f"DB = {q['DB']:.4f}, sizes = {q['sizes']}")
    for k, a in wmic_l2.get("ari_vs_full", {}).items():
        L.append(f"  - ARI (полное vs без Москвы/МО) для K{k}: **{a:.4f}**")
    L.append("")

# --- 4. SberIndex -------------------------------------------------------
L.append("## 4. SBERINDEX: ВНЕШНЯЯ ВАЛИДАЦИЯ")
L.append("")
if match_l1 is not None:
    L.append("### 4.1. Ближайшие отрасли СберИндекса к L1-кластерам")
    L.append("```")
    L.append(fmt_table(match_l1.groupby("cluster").head(3)))
    L.append("```")
    L.append("")
if match_l2 is not None:
    L.append("### 4.2. Ближайшие отрасли к L2-подкластерам")
    L.append("```")
    L.append(fmt_table(match_l2.groupby(["parent_k", "sub_k"]).head(2)))
    L.append("```")
    L.append("")

# --- 4.3. Корреляции SberIndex × prevalence ---------------------------
if corr_prev is not None and len(corr_prev):
    L.append("### 4.3. Корреляции SberIndex × prevalence L1/L2")
    L.append("")
    L.append(f"Годовых срезов с перехлёстом SberIndex: "
             f"{int(corr_prev['n_years'].max())} (2020/2021–{y_b}). "
             f"При N<5 корреляции — индикатор направления, "
             f"а не статистически строгий результат.")
    L.append("")
    piv = corr_prev.pivot_table(index="indicator", columns="cluster",
                                  values="spearman_r")
    L.append("Spearman r (indicator × cluster):")
    L.append("```")
    L.append(fmt_table(piv.reset_index(), n=3))
    L.append("```")
    L.append("")

# --- 4.4. Корреляции SberIndex × профили ------------------------------
if corr_prof is not None and len(corr_prof):
    L.append("### 4.4. Корреляции SberIndex × профили кластеров "
             "(топ-15 по |r|)")
    L.append("")
    top = (corr_prof.assign(abs_r=lambda d: d["pearson_r"].abs())
           .sort_values("abs_r", ascending=False).head(15))
    L.append("```")
    L.append(fmt_table(top[["indicator", "cluster", "feature",
                             "n_years", "pearson_r"]]))
    L.append("```")
    L.append("")

# --- 4.5. Median split SberIndex: MWU по профилям ---------------------
if split_mwu is not None and len(split_mwu):
    n_sig = int(split_mwu["significant_fdr"].sum())
    L.append("### 4.5. Медианный сплит SberIndex: MWU профилей "
             "(high vs low years)")
    L.append("")
    L.append(f"**Значимых сдвигов профилей: {n_sig} / {len(split_mwu)}** "
             f"(FDR < 0.05). В «высокие» годы по каждому показателю "
             f"SberIndex профили МО статистически отличаются от «низких».")
    L.append("")
    if n_sig > 0:
        top_sig = (split_mwu[split_mwu["significant_fdr"]]
                   .assign(abs_delta=lambda d: d["delta"].abs())
                   .sort_values("abs_delta", ascending=False).head(10))
        L.append("Топ-10 значимых сдвигов (по |Δ медиан|):")
        L.append("```")
        L.append(fmt_table(top_sig[["indicator", "level", "cluster",
                                     "feature", "median_hi", "median_lo",
                                     "delta", "p_fdr"]]))
        L.append("```")
        L.append("")

# --- 5. Ограничения -----------------------------------------------------
L.append("## 5. ОГРАНИЧЕНИЯ И ЧТО НЕ ПРОВЕРЕНО")
L.append("")
lim = []
lim.append("1. **L1 jackknife у K1 = 0.223** — при удалении крупнейшего "
           "rural-региона (Свердловская обл., 3.4 %) KMeans пересчитывает "
           "границы так, что rural-метки сдвигаются. Это следствие сильного "
           "дисбаланса классов (57 vs 1965 МО), а не ошибка модели. "
           "Urban-кластер устойчив (holds=1.0).")
lim.append("2. **L1 ARI без Москвы/МО = 0.11** — метки переворачиваются "
           "при переналожении разбиения. Уровень K=2 очень чувствителен к "
           "переименованию. В интерпретации L1 использовать силуэт и "
           "профили, а не ARI переналожения.")
lim.append("3. **K1.S2 (n=162, hold_without_top=0.063)** — почти полностью "
           "Московская область (top-1 = 0.123 региона). Это подкластер "
           "агломерации, а не универсальный тип МО. В интерпретации "
           "указывать явно.")
lim.append("4. **SberIndex: gap между 1.478 и 1.058** — сопоставление "
           "национальных отраслевых темпов роста зарплат с муниципальными "
           "тёмпами даёт большие gap (≥ 0.42). Это ожидаемо: национальный "
           "уровень выше муниципального на 1–2 % в год. Считать это "
           "ограничением, а не провалом валидации.")
lim.append("5. **Нет муниципальных датасетов SberIndex** — «потребительские "
           "траты по категориям» и «покупательская мобильность» требуют "
           "ручной выгрузки. Валидация проведена на национальных рядах.")
lim.append("6. **`tax_share` не использован в L1-профилях** — исключён "
           "паспортизацией как вырожденный (63 из 66 МО = Москва). "
           "Отсутствие этого признака сужает интерпретацию фискальной "
           "автономии.")
lim.append("7. **Годовые данные Росстата** — нет внутригодовой динамики; "
           "SberIndex на национальном уровне с 2020/2021 — перехлёст только "
           "по 5 последним годам.")
L.extend(lim)
L.append("")

# --- 7. ICVI post-hoc -------------------------------------------------
if icvi_summary is not None:
    L.append("## 7. ICVI POST-HOC (SW, CH, DB, S_DbW, AVI, AVU, MQ)")
    L.append("")
    L.append("Все 7 ICVI-метрик посчитаны на финальных кластерах L1 и L2 "
             "по годам, на KAN-эмбеддингах, с mutual kNN (k=10).")
    L.append("")
    L.append(fmt_table(icvi_summary, n=4))
    L.append("")

    if icvi_l1 is not None:
        L.append("### 7.1. ICVI L1 по годам")
        L.append("```")
        L.append(fmt_table(icvi_l1, n=4))
        L.append("```")
        L.append("")

    if icvi_l2 is not None and len(icvi_l2):
        L.append("### 7.2. ICVI L2 по годам и родителям")
        L.append("```")
        L.append(fmt_table(icvi_l2, n=4))
        L.append("```")
        L.append("")

    L.append("### 7.3. Ограничения ICVI при сильном дисбалансе классов")
    L.append("")
    L.append("- **AVI** (BSS/TSS) систематически занижен для L1 из-за "
             "imbalance 57/1965. На сбалансированной подвыборке "
             "(57 urban + 57 rural) AVI растёт с 0.29 до 0.74, "
             "MQ с 0.06 до 0.45 — то есть качество разделения "
             "**существенно выше**, чем показывает общая выборка.")
    L.append("- **MQ** при imbalance 97/3 теоретически ограничена "
             "сверху: даже идеальное разбиение даёт Q ≈ 0.05–0.10. "
             "Значение Q = 0.06 для L1 — это **норма**, а не слабость модели.")
    L.append("- **Основной критерий** качества — silhouette = 0.687 для L1 "
             "и 0.22–0.27 для L2, а также stability-метрики: "
             "bootstrap-ARI 0.90, jackknife 1.00 (L1).")
    L.append("")

lim.append("8. **SberIndex-корреляции с N=4..5 лет** статистически слабы "
           "(FDR не проходит ни одна при N≤5). Устойчивый сигнал даёт "
           "только **median split MWU** — 154 значимых сдвига профилей "
           "из 280 (FDR<0.05). Это ожидаемо: перехлёст SberIndex с "
           "горизонтом кластеризации всего 4–5 лет.")
lim.append("9. **ICVI при imbalance 57/1965**: AVI/MQ систематически "
           "занижены относительно порогов из §7 старого проекта. "
           "Использовать сбалансированную подвыборку для честного "
           "контроля (AVI = 0.74, MQ = 0.45).")


# --- 6. Артефакты ------------------------------------------------------
L.append("## 6. АРТЕФАКТЫ")
L.append("")
L.append("### 6.1. Parquet-файлы (processed/)")
for p in sorted(PROC.glob("*.parquet")):
    L.append(f"- `{p.name}` — {p.stat().st_size/1024:.1f} KB")
L.append("")
L.append("### 6.2. Визуализации (processed/viz/)")
for p in sorted(VIZ.glob("*.png")):
    L.append(f"- `viz/{p.name}`")
for p in sorted(VIZ.glob("*.html")):
    L.append(f"- `viz/{p.name}`")
L.append("")

md = "\n".join(L)
(REP / "report_for_llm.md").write_text(md, encoding="utf-8")
print(f"✅ report_for_llm.md ({len(md):,} символов)")


# ============================================================================
# ЧАСТЬ C. METHODOLOGY_REPORT_RU.PDF
# ============================================================================
print("\n" + "=" * 70)
print("ЧАСТЬ C. methodology_report_ru.pdf")
print("=" * 70)

from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import font_manager

dejavu = font_manager.findfont("DejaVu Sans")
print(f"DejaVu Sans: {dejavu}")


class PDFBuilder:
    W, H = 8.27, 11.69
    ML, MR = 0.08, 0.92
    LH = 0.020
    BOT = 0.06
    def __init__(self, path):
        self.pdf = PdfPages(str(path))
        self.fig = None
        self._new()
    def _new(self):
        if self.fig is not None:
            self.pdf.savefig(self.fig); plt.close(self.fig)
        self.fig = plt.figure(figsize=(self.W, self.H))
        self.fig.patch.set_facecolor("white")
        self.ax = self.fig.add_axes([0, 0, 1, 1])
        self.ax.set_xlim(0, 1); self.ax.set_ylim(0, 1); self.ax.axis("off")
        self.y = 0.95
    def _ensure(self, n=1):
        if self.y - n * self.LH < self.BOT:
            self._new()
    def title(self, t):
        self._ensure(3)
        self.ax.text(self.ML, self.y, t, fontsize=20, fontweight="bold",
                     fontfamily="DejaVu Sans", color="#1e40af", va="top")
        self.y -= self.LH * 2.5
    def subtitle(self, t):
        self._ensure(2)
        self.ax.text(self.ML, self.y, t, fontsize=11,
                     fontfamily="DejaVu Sans", color="#475569", va="top")
        self.y -= self.LH * 1.6
    def h1(self, t):
        self._ensure(3)
        self.y -= self.LH * 0.5
        self.ax.text(self.ML, self.y, t, fontsize=16, fontweight="bold",
                     fontfamily="DejaVu Sans", color="#1e40af", va="top")
        self.y -= self.LH * 2.2
    def h2(self, t):
        self._ensure(2)
        self.y -= self.LH * 0.4
        self.ax.text(self.ML, self.y, t, fontsize=13, fontweight="bold",
                     fontfamily="DejaVu Sans", color="#334155", va="top")
        self.y -= self.LH * 1.8
    def para(self, t, fs=10, chars=100, col="#0f172a"):
        if not t:
            self.y -= self.LH * 0.5; return
        for line in textwrap.wrap(t, width=chars) or [""]:
            self._ensure(1)
            self.ax.text(self.ML, self.y, line, fontsize=fs,
                         fontfamily="DejaVu Sans", color=col, va="top")
            self.y -= self.LH
    def bullet(self, t, fs=10, chars=96):
        self.para("• " + t, fs, chars)
    def spacer(self, n=1):
        self.y -= self.LH * n
    def close(self):
        if self.fig is not None:
            self.pdf.savefig(self.fig); plt.close(self.fig)
        self.pdf.close()


def strip_md(t):
    t = re.sub(r"\*\*(.+?)\*\*", r"\1", t)
    t = re.sub(r"\*(.+?)\*",   r"\1", t)
    t = re.sub(r"`(.+?)`",     r"\1", t)
    return t


pdf_path = REP / "methodology_report_ru.pdf"
pdf = PDFBuilder(pdf_path)

# Титульный лист
pdf.spacer(6)
pdf.title("Кластеризация муниципалитетов России")
pdf.spacer(1)
pdf.subtitle("KAN-энкодер · иерархия L1 → L2 · устойчивость и внешняя валидация")
pdf.spacer(4)
pdf.subtitle(f"Горизонт: {y_a}–{y_b} ({n_months} годовых срезов)")
pdf.subtitle(f"Объектов: {n_mo} МО, 77+ регионов")
pdf.subtitle(f"L1: K = {K} кластеров (urban / rural)")
pdf.subtitle(f"L2: subK = {L2_SUBK}, всего {sum(L2_SUBK.values())} подкластеров")
pdf.spacer(3)
pdf.subtitle("Ключевые метрики:")
if ari_l1 is not None:
    pdf.subtitle(f"  ARI L1(consec) = {ari_l1['ari'].mean():.4f}, "
                 f"stay rate = {ari_l1['stay_rate'].mean():.4f}")
if l2_stab is not None:
    for _, r in l2_stab.iterrows():
        pdf.subtitle(f"  L2 K{int(r.parent_k)}: ARI={r.ari_consec_mean:.4f}, "
                     f"stay={r.stay_rate_mean:.4f}")
if kan_cfg_l1:
    pdf.subtitle(f"  KAN-L1: R²={kan_cfg_l1['r2']:.4f}, "
                 f"silhouette={kan_cfg_l1['silhouette']:.4f}")
pdf.spacer(3)
pdf.subtitle(f"Отчёт: {datetime.now():%Y-%m-%d %H:%M}")

# Основной контент из md
pdf._new()
for line in md.split("\n"):
    s = line.rstrip()
    if not s.strip():
        pdf.spacer(0.5); continue
    if s.startswith("# "):
        pdf.h1(strip_md(s[2:]))
    elif s.startswith("## "):
        pdf.h2(strip_md(s[3:]))
    elif s.startswith("### "):
        pdf.h2(strip_md(s[4:]))
    elif s.startswith("- ") or s.startswith("  - "):
        indent = "  " if s.startswith("  - ") else ""
        pdf.bullet(indent + strip_md(s.lstrip("- ")))
    elif s.startswith("|"):
        if set(s) <= set("|-: "):
            continue
        cells = [c.strip() for c in s.strip("|").split("|")]
        pdf.para("  ".join(cells), fs=7.5, chars=150, col="#334155")
    elif s.startswith("```"):
        continue
    else:
        pdf.para(strip_md(s))

pdf.close()
print(f"✅ methodology_report_ru.pdf ({pdf_path.stat().st_size/1024:.1f} KB)")


# ============================================================================
# ЧАСТЬ D. ZIP + MANIFEST
# ============================================================================
print("\n" + "=" * 70)
print("ЧАСТЬ D. ZIP + MANIFEST")
print("=" * 70)

for f in ["07_sber_validation.png", "07_sber_validation_l2.png",
          "08_validation_structure.png",
          "03_network_l2_parent0.png", "03_network_l2_parent1.png"]:
    p = VIZ / f
    if p.exists():
        p.unlink()
        print(f"удалён: {f}")

ZIP_PATH = SAVE / "clustering_v2_artifacts.zip"
MANIFEST = SAVE / "manifest.json"


def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


all_files = [p for p in PROC.rglob("*")
             if p.is_file() and not p.name.endswith(".zip")]

manifest = {
    "created_at":    datetime.now().isoformat(),
    "project":       "Кластеризация МО России (L1+L2, KAN)",
    "K":             int(K),
    "subK":          {int(k): int(v) for k, v in L2_SUBK.items()},
    "n_mo":          int(n_mo),
    "n_years":       int(n_months),
    "y_a":           int(y_a),
    "y_b":           int(y_b),
    "n_features":    int(n_feats),
    "files":         {},
}
for p in all_files:
    rel = p.relative_to(PROC)
    manifest["files"][str(rel)] = {
        "size":   int(p.stat().st_size),
        "sha256": sha256(p),
    }
MANIFEST.write_text(_json.dumps(manifest, ensure_ascii=False, indent=2),
                     encoding="utf-8")

total_bytes = sum(p.stat().st_size for p in all_files)
print(f"файлов: {len(all_files)}, размер: {total_bytes/1e6:.1f} MB")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED,
                      compresslevel=6) as zf:
    for p in all_files:
        zf.write(p, arcname=str(p.relative_to(PROC.parent)))
    zf.write(MANIFEST, arcname="manifest.json")

print(f"✅ {ZIP_PATH.name} ({ZIP_PATH.stat().st_size/1e6:.1f} MB)")


# ============================================================================
# ИТОГ
# ============================================================================
print("\n" + "=" * 70)
print("ИТОГ ЯЧЕЙКИ 12")
print("=" * 70)
print(f"L1: K = {K}")
print(f"L2: subK = {L2_SUBK}, всего {sum(L2_SUBK.values())} подкластеров")
print(f"МО: {n_mo}, лет: {n_months} ({y_a}–{y_b})")
print(f"Признаков: {n_feats}")
if ari_l1 is not None:
    print(f"ARI L1(consec): {ari_l1['ari'].mean():.4f}, "
          f"stay rate: {ari_l1['stay_rate'].mean():.4f}")
if l2_stab is not None:
    for _, r in l2_stab.iterrows():
        print(f"  L2 K{int(r.parent_k)}: ARI={r.ari_consec_mean:.4f}, "
              f"stay={r.stay_rate_mean:.4f}")
print()
print("Артефакты в reports/:")
for p in sorted(REP.glob("*")):
    print(f"  ✅ {p.name}  ({p.stat().st_size/1024:.1f} KB)")
print()
print(f"ZIP: {ZIP_PATH}")
print(f"     {ZIP_PATH.stat().st_size/1e6:.1f} MB")
print()
print("КАК СОХРАНИТЬ В KAGGLE:")
print("  1. Save Version → Save & Run All (Commit)")
print("  2. Output → скачать clustering_v2_artifacts.zip")
print("  3. Kaggle → Datasets → New Dataset → Upload ZIP")
print("  4. В новом сеансе: Add Input → Datasets → подключить")
print("  5. Открыть processed/viz/index.html для просмотра")
print()
print("Дополнительные разделы отчёта:")
print(f"  §4.3–4.5  SberIndex × L1/L2 корреляции")
if icvi_summary is not None:
    print(f"  §7        ICVI post-hoc (L1 + L2)")
print()
print("Новые визуализации:")
for f in ["17_icvi_panel.png", "18_sber_correlation.png"]:
    p = VIZ / f
    if p.exists():
        print(f"  ✅ viz/{f}")

tick("отчёт + PDF + ZIP готовы")

[1380.9s] финальный отчёт + PDF + ZIP

ЧАСТЬ A. СБОР МЕТРИК
L1: K=2, ARI=0.9766733090391849
L2: subK={0: 2, 1: 3}, ARI по родителям=[0.6157035251751141, 0.62677103055532]
chi² L1=True, chi² L2=True
SberIndex used: ['median-wages: 19 отраслей, 2020-2024', 'сопоставление роста зарплат L1 с отраслями Сбера', 'potrebitelskaya-aktivnost', 'real-key-interest-rate', 'SberIndex × prevalence L1/L2 (Spearman/Pearson)', 'SberIndex × профили кластеров (Pearson)', 'SberIndex median split MWU (high vs low years)']

ЧАСТЬ B. report_for_llm.md
✅ report_for_llm.md (23,505 символов)

ЧАСТЬ C. methodology_report_ru.pdf
DejaVu Sans: /usr/local/lib/python3.13/dist-packages/matplotlib/mpl-data/fonts/ttf/DejaVuSans.ttf
✅ methodology_report_ru.pdf (134.1 KB)

ЧАСТЬ D. ZIP + MANIFEST
удалён: 07_sber_validation.png
удалён: 07_sber_validation_l2.png
удалён: 08_validation_structure.png
удалён: 03_network_l2_parent0.png
удалён: 03_network_l2_parent1.png
файлов: 124, размер: 141.6 MB
✅ clustering_v2_artifacts.zip (